# Bước 1: Cài đặt các gói hệ thống cần thiết

Chạy ô này để cập nhật danh sách gói và cài đặt các thư viện hệ thống như `sox`, `cairo`, `pango`, `ffmpeg` và `LaTeX`. Đây là những phụ thuộc cần thiết cho Manim và Manim Voiceover.

In [ ]:
!apt-get update -y
!apt-get install -y sox libcairo2-dev libpango1.0-dev ffmpeg texlive texlive-latex-extra texlive-fonts-extra texlive-latex-recommended

# Bước 2: Cài đặt các thư viện Python

Chạy ô này để cài đặt các thư viện Python `manim`, `manim-voiceover`, `edge-tts` và `nest_asyncio`.

In [ ]:
!pip install manim
!pip install manim-voiceover
!pip install edge-tts
!pip install nest_asyncio


# Bước 3: Thiết lập Manim Voiceover

Chạy ô này để áp dụng `nest_asyncio` (cần thiết cho Manim Voiceover trong môi trường Colab) và định nghĩa lớp `EdgeTTSService`. Lớp này cho phép Manim sử dụng Edge TTS để tạo giọng nói cho các cảnh của bạn.

In [ ]:
import nest_asyncio
nest_asyncio.apply()

import asyncio
import hashlib
import numpy as np
from pathlib import Path

from manim import *
from manim_voiceover import VoiceoverScene
from manim_voiceover.services.base import SpeechService

import edge_tts

# EdgeTTSService
VOICE = "en-US-GuyNeural"

class EdgeTTSService(SpeechService):
    def __init__(self, voice="en-US-GuyNeural", **kwargs):
        self.voice = voice
        super().__init__(**kwargs)

    def generate_from_text(self, text, cache_dir=None, path=None, **kwargs):
        if cache_dir is None:
            cache_dir = self.cache_dir
        Path(cache_dir).mkdir(parents=True, exist_ok=True)

        input_data = {"input_text": text, "service": "edge_tts", "voice": self.voice}
        cached = self.get_cached_result(input_data, cache_dir)
        if cached is not None:
            return cached

        audio_basename = hashlib.md5(text.encode()).hexdigest()
        audio_file = audio_basename + ".mp3"
        full_path = str(Path(cache_dir) / audio_file)

        communicate = edge_tts.Communicate(text, self.voice)
        loop = asyncio.get_event_loop()
        loop.run_until_complete(communicate.save(full_path))

        return {
            "input_text": text,
            "input_data": input_data,
            "original_audio": audio_file,
        }


# Bước 4: Chạy cảnh Manim của bạn

Sau khi hoàn tất các bước thiết lập trên, bạn có thể dán mã định nghĩa cảnh Manim của mình (ví dụ: `Scene01_TitleCard`) vào một ô mới. Để chạy cảnh đó, hãy sử dụng lệnh `%%manim` theo sau là tên lớp cảnh.

Ví dụ: để chạy `Scene01_TitleCard`, bạn sẽ thêm một ô mới với nội dung sau:

```python
%%manim Scene01_TitleCard
```

Bạn có thể thay thế `Scene01_TitleCard` bằng tên của bất kỳ cảnh nào khác mà bạn muốn chạy.

# SLIDE 1 TỚI SLIDE 4

In [ ]:
# Scene: Introduction & Invariance -- Slides 01 to 04
# Scenes: Scene01_TitleCard, Scene02_TutorialOverview,
#         Scene03_Invariance, Scene04_MoreInvariance
# Scene 1: Title Card
class Scene01_TitleCard(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        line1 = Text("Recent Developments in", font_size=40, color=WHITE)
        line2 = Text("Geometric Machine Learning", font_size=40, color=YELLOW)
        title_grp = VGroup(line1, line2).arrange(DOWN, buff=0.18).move_to(UP * 1.3)

        subtitle = Text(
            "Foundations, Models, and More",
            font_size=26, color=GREY_B,
        ).next_to(title_grp, DOWN, buff=0.35)

        # --- Beat 1: Title ---
        with self.voiceover(
            text=(
                "Welcome to this tutorial on recent developments in geometric machine learning"
                " -- foundations, models, and more."
                " Throughout this series, we will explore how symmetry structure in data and tasks"
                " shapes the learning process in fundamental ways,"
                " leading to better data efficiency, stronger generalization, and more robust models."
            )
        ) as tracker:
            self.play(Write(line1), run_time=tracker.duration * 0.38)
            self.play(Write(line2), run_time=tracker.duration * 0.38)
            self.play(FadeIn(subtitle, shift=UP * 0.15), run_time=tracker.duration * 0.2)

        self.wait(1.5)
        div = Line(LEFT * 4.8, RIGHT * 4.8, color=GREY_B, stroke_width=1).next_to(
            subtitle, DOWN, buff=0.4
        )
        authors = Text(
            "Stefanie Jegelka (TUM / MIT)    Behrooz Tahmasebi (Harvard)",
            font_size=20, color=GREY_B,
        ).next_to(div, DOWN, buff=0.3)
        venue = Text(
            "NeurIPS 2025  |  San Diego, CA  |  December 2025",
            font_size=18, color=GREY_B,
        ).next_to(authors, DOWN, buff=0.2)

        # --- Beat 2: Authors ---
        with self.voiceover(
            text=(
                "This tutorial was presented at NeurIPS 2025 by Stefanie Jegelka from TUM and MIT,"
                " and Behrooz Tahmasebi from Harvard."
            )
        ) as tracker:
            self.play(Create(div), run_time=tracker.duration * 0.2)
            self.play(FadeIn(authors), run_time=tracker.duration * 0.35)
            self.play(FadeIn(venue), run_time=tracker.duration * 0.3)
            self.wait(tracker.duration * 0.1)

        self.wait(1.5)
        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.6)
        self.wait(0.3)


# Scene 2: What Is This Tutorial About
class Scene02_TutorialOverview(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title = Text(
            "What Is This Tutorial About?", font_size=36, color=WHITE
        ).to_edge(UP, buff=0.45)
        self.play(Write(title), run_time=0.7)

        # --- Beat 1: Input -> f -> Output ---
        in_box = Rectangle(width=2.2, height=1.0, color=BLUE, stroke_width=2)
        in_lbl = Text("Input", font_size=22, color=BLUE).move_to(in_box)
        in_grp = VGroup(in_box, in_lbl).move_to(LEFT * 3.0 + UP * 0.3)

        out_box = Rectangle(width=2.2, height=1.0, color=GREEN, stroke_width=2)
        out_lbl = Text("Output", font_size=22, color=GREEN).move_to(out_box)
        out_grp = VGroup(out_box, out_lbl).move_to(RIGHT * 3.0 + UP * 0.3)

        f_arrow = Arrow(
            in_grp.get_right() + RIGHT * 0.08,
            out_grp.get_left() + LEFT * 0.08,
            buff=0.0, color=WHITE, stroke_width=2,
        )
        f_lbl = Text("f", font_size=28, color=WHITE).next_to(f_arrow, UP, buff=0.1)

        with self.voiceover(
            text=(
                "Machine learning models encounter data that carries hidden structure"
                " -- data that remains essentially the same under certain transformations."
                " Think of an image: shift it by a few pixels, and the content has not changed."
                " Or a molecule: rotate it in space, and its chemical properties remain identical."
                " Or a set of points: list them in a different order, and the underlying object is the same."
                " We call these transformations symmetries,"
                " and understanding them is a core challenge in modern machine learning."
            )
        ) as tracker:
            self.play(FadeIn(in_grp), FadeIn(out_grp), run_time=tracker.duration * 0.4)
            self.play(GrowArrow(f_arrow), FadeIn(f_lbl), run_time=tracker.duration * 0.35)
            self.wait(tracker.duration * 0.2)

        # --- Beat 2: Three symmetry examples (mini-visuals) ---
        cell_sz = 0.22

        def make_image_mini():
            def grid(pattern):
                g = VGroup()
                for ri, row in enumerate(pattern):
                    for ci, filled in enumerate(row):
                        cell = Square(
                            side_length=cell_sz,
                            fill_color=ORANGE if filled else GREY_E,
                            fill_opacity=0.9,
                            stroke_color=WHITE, stroke_width=0.6,
                        )
                        cell.move_to([ci * cell_sz, -ri * cell_sz, 0])
                        g.add(cell)
                return g
            g_a = grid([[1, 0], [0, 0]])
            g_b = grid([[0, 1], [0, 0]])
            arr = MathTex(r"\rightarrow", font_size=20, color=GREY_B)
            return VGroup(g_a, arr, g_b).arrange(RIGHT, buff=0.18)

        def make_molecule_mini():
            atom_r = 0.13
            def cluster(angles):
                clrs = [BLUE, TEAL, GOLD]
                dots = VGroup()
                for a, c in zip(angles, clrs):
                    d = Dot(radius=atom_r, color=c, fill_opacity=1.0)
                    d.move_to([0.28 * np.cos(a), 0.28 * np.sin(a), 0])
                    dots.add(d)
                pts = [d.get_center() for d in dots]
                bonds = VGroup(*[
                    Line(pts[i], pts[(i + 1) % 3], stroke_width=1.0, color=GREY_B)
                    for i in range(3)
                ])
                return VGroup(bonds, dots)
            ang_a = [PI / 2, PI / 2 + 2 * PI / 3, PI / 2 + 4 * PI / 3]
            ang_b = [PI / 2 + PI / 3, PI / 2 + PI, PI / 2 + 5 * PI / 3]
            arr = MathTex(r"\rightarrow", font_size=20, color=GREY_B)
            return VGroup(cluster(ang_a), arr, cluster(ang_b)).arrange(RIGHT, buff=0.18)

        def make_pointset_mini():
            clr_map = {"A": BLUE, "B": TEAL, "C": GOLD}
            def col(lbls):
                g = VGroup()
                for l in lbls:
                    d = Dot(radius=0.13, color=clr_map[l], fill_opacity=1.0)
                    t = Text(l, font_size=12, color=WHITE).move_to(d)
                    g.add(VGroup(d, t))
                return g.arrange(DOWN, buff=0.14)
            arr = MathTex(r"\rightarrow", font_size=20, color=GREY_B)
            return VGroup(col(["A", "B", "C"]), arr, col(["C", "A", "B"])).arrange(RIGHT, buff=0.18)

        def wrap_mini(title_str, mini_row, result_str, x_offset):
            head = Text(title_str, font_size=17, color=ORANGE, weight=BOLD)
            res = Text(result_str, font_size=15, color=GREEN)
            grp = VGroup(head, mini_row, res).arrange(DOWN, buff=0.18)
            grp.move_to([x_offset, -0.8, 0])
            return grp

        ex1 = wrap_mini("Image", make_image_mini(), "f = same label", -3.5)
        ex2 = wrap_mini("Molecule", make_molecule_mini(), "f = same property", 0.0)
        ex3 = wrap_mini("Point set", make_pointset_mini(), "f = same output", 3.5)

        self.play(
            FadeOut(in_grp), FadeOut(out_grp),
            FadeOut(f_arrow), FadeOut(f_lbl),
            run_time=0.35,
        )
        with self.voiceover(
            text=(
                "The first example is image classification:"
                " shifting a picture by a few pixels does not change what it depicts,"
                " so the predicted label should remain the same."
                " The second is molecular property prediction:"
                " rotating a molecule in 3D preserves all of its physical and chemical properties."
                " The third is set processing:"
                " a function on a collection of objects must not depend on the order they are listed."
                " In each case, the transformation leaves the relevant information intact."
            )
        ) as tracker:
            self.play(FadeIn(ex1), run_time=tracker.duration * 0.3)
            self.play(FadeIn(ex2), run_time=tracker.duration * 0.3)
            self.play(FadeIn(ex3), run_time=tracker.duration * 0.3)
            self.wait(tracker.duration * 0.1)

        # --- Beat 3: Impact list ---
        hdr = Text(
            "Symmetry affects our models:",
            font_size=24, color=WHITE,
        ).next_to(title, DOWN, buff=0.55)
        items = VGroup(
            Text("- data efficiency and sample complexity", font_size=21, color=YELLOW),
            Text("- generalization and robustness", font_size=21, color=YELLOW),
            Text("- optimization and sampling", font_size=21, color=YELLOW),
            Text("- interpretability and model analysis", font_size=21, color=YELLOW),
        ).arrange(DOWN, aligned_edge=LEFT, buff=0.22).next_to(hdr, DOWN, buff=0.35)


        self.play(FadeOut(ex1), FadeOut(ex2), FadeOut(ex3), run_time=0.35)

        with self.voiceover(
            text=(
                "These symmetries, when explicitly built into our models, have a profound impact."
                " Models that respect symmetry require far less training data,"
                " because they generalize correctly across all transformed versions of an input."
                " They generalize better to new examples, resist noisy perturbations more effectively,"
                " and are often easier to optimize and interpret."
                " This motivates a systematic study of symmetry in machine learning."
            )
        ) as tracker:
            self.play(Write(hdr), run_time=tracker.duration * 0.25)
            for item in items:
                self.play(FadeIn(item, shift=RIGHT * 0.15), run_time=tracker.duration * 0.15)
            self.wait(tracker.duration * 0.1)


        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.6)
        self.wait(0.3)


# Scene 3: Invariance -- Definition and Two Examples
class Scene03_Invariance(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title = Text(
            "What Is a Symmetry?  Invariance",
            font_size=34, color=WHITE,
        ).to_edge(UP, buff=0.45)
        self.play(Write(title), run_time=0.7)

        # --- Beat 1: Main formula ---
        formula = MathTex(r"f(T(x)) = f(x)", font_size=56, color=YELLOW).move_to(UP * 0.9)
        defn = Text(
            "Invariance: applying T to the input does not change the output.",
            font_size=22, color=GREY_B,
        ).next_to(formula, DOWN, buff=0.4)

        with self.voiceover(
            text=(
                "Let us be precise."
                " Given a function f, a transformation T is a symmetry of f"
                " if applying T to any input does not change the output."
                " Mathematically: f of T of x equals f of x, for all valid inputs x."
                " We call this property invariance."
                " The output is invariant -- it does not change --"
                " no matter how we apply the transformation T to the input."
                " This is the central definition we will build everything on."
            )
        ) as tracker:
            self.play(Write(formula), run_time=tracker.duration * 0.40)
            self.play(FadeIn(defn), run_time=tracker.duration * 0.30)
            self.wait(tracker.duration * 0.25)

        self.wait(0.8)
        # --- Beat 2: Shift invariance ---
        self.play(FadeOut(defn), run_time=0.3)

        formula_shift = MathTex(
            r"f(\mathrm{shift}(x)) = f(x)",
            font_size=38, color=YELLOW,
        ).next_to(title, DOWN, buff=0.55)

        # Pixel strip: 5 squares showing original and shifted image
        sq = 0.36
        orig_colors = [BLUE_D, ORANGE, ORANGE, BLUE_D, BLUE_D]
        shft_colors = [BLUE_D, BLUE_D, ORANGE, ORANGE, BLUE_D]

        def make_strip(clrs, cx):
            grp = VGroup()
            for i, c in enumerate(clrs):
                cell = Square(
                    side_length=sq,
                    fill_color=c, fill_opacity=0.85,
                    stroke_color=WHITE, stroke_width=0.8,
                )
                cell.move_to([cx + (i - 2) * sq, -0.65, 0])
                grp.add(cell)
            return grp

        strip_a = make_strip(orig_colors, -2.5)
        strip_b = make_strip(shft_colors, 2.5)

        sh_arrow = Arrow(
            np.array([-1.1, -0.65, 0]),
            np.array([1.1, -0.65, 0]),
            buff=0.0, stroke_width=2, color=GREY_B,
        )
        sh_lbl = Text("shift", font_size=18, color=ORANGE).next_to(sh_arrow, UP, buff=0.1)
        eq_shift = MathTex(
            r"f(I) = f(\mathrm{shift}(I))",
            font_size=26, color=GREEN,
        ).move_to(DOWN * 1.4)

        with self.voiceover(
            text=(
                "The simplest and most familiar example is image classification."
                " If you shift a photograph of a cat a few pixels to the left,"
                " it is still a photograph of a cat."
                " A good classifier must be shift-invariant:"
                " applying the shift transformation T cannot change the predicted label."
                " This is why convolutional neural networks use weight sharing across spatial locations"
                " -- it encodes shift invariance directly into the architecture."
                " The same filter detects a feature wherever it appears in the image."
            )
        ) as tracker:
            # ReplacementTransform morphs the formula in-place so the old MathTex is
            # consumed; avoids leaving a ghost object on screen that FadeOut might miss.
            self.play(
                ReplacementTransform(formula, formula_shift),
                run_time=tracker.duration * 0.28,
            )
            self.play(FadeIn(strip_a), run_time=tracker.duration * 0.2)
            self.play(
                GrowArrow(sh_arrow), FadeIn(sh_lbl), FadeIn(strip_b),
                run_time=tracker.duration * 0.25,
            )
            self.play(FadeIn(eq_shift), run_time=tracker.duration * 0.2)
            self.wait(tracker.duration * 0.05)

        self.wait(0.8)
        # --- Beat 3: Permutation invariance ---
        self.play(
            FadeOut(strip_a), FadeOut(strip_b),
            FadeOut(sh_arrow), FadeOut(sh_lbl), FadeOut(eq_shift),
            run_time=0.4,
        )

        formula_perm = MathTex(
            r"f(\mathrm{permute}(x)) = f(x)",
            font_size=38, color=YELLOW,
        ).next_to(title, DOWN, buff=0.55)

        pt_clrs = [BLUE, TEAL, GOLD, ORANGE]
        pt_lbls = [r"x_1", r"x_2", r"x_3", r"x_4"]
        perm = [2, 0, 3, 1]

        def make_dot_list(clrs, lbls, x_pos):
            grp = VGroup()
            for c, l in zip(clrs, lbls):
                dot = Dot(radius=0.18, color=c, fill_opacity=1.0)
                lbl = MathTex(l, font_size=16, color=WHITE).move_to(dot)
                grp.add(VGroup(dot, lbl))
            grp.arrange(DOWN, buff=0.22).move_to([x_pos, -0.55, 0])
            return grp

        orig_dots = make_dot_list(pt_clrs, pt_lbls, -2.8)
        perm_dots = make_dot_list(
            [pt_clrs[i] for i in perm],
            [pt_lbls[i] for i in perm],
            2.8,
        )
        p_arrow = Arrow(
            np.array([-1.3, -0.55, 0]),
            np.array([1.3, -0.55, 0]),
            buff=0.0, stroke_width=2, color=GREY_B,
        )
        p_lbl = Text("permute", font_size=18, color=ORANGE).next_to(p_arrow, UP, buff=0.1)
        eq_perm = MathTex(
            r"f(x_1,x_2,x_3,x_4) = f(x_3,x_1,x_4,x_2)",
            font_size=22, color=GREEN,
        ).move_to(DOWN * 1.6)

        with self.voiceover(
            text=(
                "The second example comes from functions on sets or point clouds."
                " A 3D point cloud representing an object is just a collection of points"
                " -- there is no natural canonical ordering among them."
                " Any function that correctly describes the shape"
                " must produce the same output regardless of how those points are ordered."
                " This is permutation invariance,"
                " and it is the defining property of architectures like PointNet"
                " that process unordered 3D data."
            )
        ) as tracker:
            self.play(
                ReplacementTransform(formula_shift, formula_perm),
                run_time=tracker.duration * 0.25,
            )
            self.play(FadeIn(orig_dots), run_time=tracker.duration * 0.2)
            self.play(
                GrowArrow(p_arrow), FadeIn(p_lbl), FadeIn(perm_dots),
                run_time=tracker.duration * 0.25,
            )
            self.play(FadeIn(eq_perm), run_time=tracker.duration * 0.2)
            self.wait(tracker.duration * 0.05)

        self.wait(0.8)
        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.6)
        self.wait(0.3)


# Scene 4: More Invariances -- Graphs and Neural Nets
class Scene04_MoreInvariance(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title = Text(
            "More Examples of Invariance",
            font_size=34, color=WHITE,
        ).to_edge(UP, buff=0.45)
        self.play(Write(title), run_time=0.7)

        # --- Beat 1: Setup ---
        with self.voiceover(
            text=(
                "Invariance shows up across machine learning in surprising places."
                " Let us look at two concrete examples:"
                " one from graph neural networks, and one from the internal structure of neural networks themselves."
            )
        ) as tracker:
            intro = Text(
                "Invariance appears in many settings:",
                font_size=24, color=GREY_B,
            ).next_to(title, DOWN, buff=0.5)
            self.play(FadeIn(intro), run_time=tracker.duration * 0.6)
            self.wait(tracker.duration * 0.35)

        self.play(FadeOut(intro), run_time=0.3)

        # --- Beat 2: Graph invariance ---
        # Build 4-node graph manually using Dots and Lines
        n_nodes = 4
        r = 0.9
        base_angles = [PI / 2 + 2 * PI * i / n_nodes for i in range(n_nodes)]
        node_pos = {
            i + 1: np.array([r * np.cos(a), r * np.sin(a), 0])
            for i, a in enumerate(base_angles)
        }
        node_clrs_orig = {1: BLUE, 2: GOLD, 3: TEAL, 4: ORANGE}
        node_clrs_perm = {1: GOLD, 2: BLUE, 3: TEAL, 4: ORANGE}  # swap 1 <-> 2
        edge_pairs = [(1, 2), (2, 3), (3, 4), (1, 4), (1, 3)]
        left_off = np.array([-3.5, -0.5, 0])
        right_off = np.array([3.5, -0.5, 0])

        def build_graph(pos, clrs, edges, off):
            nodes_grp = VGroup()
            for v, p in pos.items():
                dot = Dot(p + off, radius=0.2, color=clrs[v], fill_opacity=1.0)
                lbl = MathTex(str(v), font_size=16, color=WHITE).move_to(p + off)
                nodes_grp.add(VGroup(dot, lbl))
            edges_grp = VGroup(*[
                Line(pos[u] + off, pos[v] + off, stroke_width=1.5, color=GREY_B)
                for u, v in edges
            ])
            return VGroup(edges_grp, nodes_grp)

        g_orig = build_graph(node_pos, node_clrs_orig, edge_pairs, left_off)
        g_perm = build_graph(node_pos, node_clrs_perm, edge_pairs, right_off)

        g_arrow = Arrow(
            left_off + np.array([1.2, 0, 0]),
            right_off + np.array([-1.2, 0, 0]),
            buff=0.0, stroke_width=2, color=GREY_B,
        )
        g_arr_lbl = MathTex(r"P", font_size=24, color=ORANGE).next_to(g_arrow, UP, buff=0.1)

        formula_graph = MathTex(
            r"f(PAP^{\top}) = f(A)",
            font_size=32, color=BLUE,
        ).next_to(title, DOWN, buff=0.55)

        eq_graph = Text(
            "same graph -- only node labels changed",
            font_size=15, color=GREEN,
        ).move_to(np.array([0.0, -1.85, 0]))

        with self.voiceover(
            text=(
                "In graph learning, nodes have no canonical ordering."
                " Permuting the rows and columns of the adjacency matrix A"
                " by a permutation matrix P simply relabels the nodes"
                " -- the underlying graph structure is unchanged."
                " Our model must therefore satisfy: f of P A P-transpose equals f of A,"
                " for any permutation P."
                " Two adjacency matrices related by such a permutation"
                " represent the same graph and must produce the same output."
            )
        ) as tracker:
            self.play(Write(formula_graph), run_time=tracker.duration * 0.25)
            self.play(Create(g_orig), run_time=tracker.duration * 0.25)
            self.play(
                GrowArrow(g_arrow), FadeIn(g_arr_lbl),
                run_time=tracker.duration * 0.15,
            )
            self.play(Create(g_perm), run_time=tracker.duration * 0.2)
            self.play(FadeIn(eq_graph), run_time=tracker.duration * 0.1)
            self.wait(tracker.duration * 0.02)

        self.wait(0.8)
        # --- Beat 3: NN weight symmetry ---
        self.play(
            FadeOut(g_orig), FadeOut(g_perm),
            FadeOut(g_arrow), FadeOut(g_arr_lbl),
            FadeOut(eq_graph),
            run_time=0.4,
        )

        formula_nn = MathTex(
            r"f(\cdot,\, PW_1,\, W_2 P^{\top}) = f(\cdot,\, W_1,\, W_2)",
            font_size=28, color=BLUE,
        ).next_to(title, DOWN, buff=0.55)

        # W1: 3 colored rows (one row per neuron)
        bar_h = 0.38
        bar_w = 1.5
        neuron_clrs = [BLUE, GOLD, TEAL]

        w1_rows = VGroup(*[
            Rectangle(
                width=bar_w, height=bar_h,
                fill_color=c, fill_opacity=0.8,
                stroke_color=WHITE, stroke_width=1.0,
            )
            for c in neuron_clrs
        ]).arrange(DOWN, buff=0.06)
        w1_lbl = Text("W1", font_size=20, color=WHITE).next_to(w1_rows, UP, buff=0.18)
        w1_grp = VGroup(w1_lbl, w1_rows).move_to(LEFT * 3.2 + DOWN * 0.4)

        # W2: 3 colored columns (same neuron colors)
        w2_cols = VGroup(*[
            Rectangle(
                width=bar_h, height=bar_w,
                fill_color=c, fill_opacity=0.8,
                stroke_color=WHITE, stroke_width=1.0,
            )
            for c in neuron_clrs
        ]).arrange(RIGHT, buff=0.06)
        w2_lbl = Text("W2", font_size=20, color=WHITE).next_to(w2_cols, UP, buff=0.18)
        w2_grp = VGroup(w2_lbl, w2_cols).move_to(RIGHT * 3.2 + DOWN * 0.4)

        nn_conn = MathTex(r"\rightarrow", font_size=32, color=GREY_B).move_to(
            DOWN * 0.5
        )

        # Neuron color legend
        legend = VGroup(
            Text("Each color = one neuron", font_size=15, color=GREY_B),
        ).move_to(DOWN * 1.85)

        with self.voiceover(
            text=(
                "Even neural networks themselves have an intrinsic symmetry."
                " In a two-layer network, permuting the neurons of the hidden layer"
                " -- rearranging the rows of W1 and simultaneously"
                " rearranging the corresponding columns of W2"
                " -- produces a network that computes exactly the same function."
                " Here, each color represents one neuron:"
                " the same color in W1 and W2 belongs to the same neuron."
                " This means neural networks do not have a unique weight representation"
                " -- there are always infinitely many equivalent parameterizations."
            )
        ) as tracker:
            self.play(
                ReplacementTransform(formula_graph, formula_nn),
                run_time=tracker.duration * 0.20,
            )
            self.play(FadeIn(w1_grp), run_time=tracker.duration * 0.20)
            self.play(
                FadeIn(nn_conn), FadeIn(w2_grp),
                run_time=tracker.duration * 0.20,
            )
            self.play(FadeIn(legend), run_time=tracker.duration * 0.12)
            # Animate row/column swap: rows 0 <-> 1 in W1, cols 0 <-> 1 in W2
            r0, r1 = w1_rows[0], w1_rows[1]
            c0, c1 = w2_cols[0], w2_cols[1]
            pos_r0 = r0.get_center().copy()
            pos_r1 = r1.get_center().copy()
            pos_c0 = c0.get_center().copy()
            pos_c1 = c1.get_center().copy()
            self.play(
                r0.animate.move_to(pos_r1),
                r1.animate.move_to(pos_r0),
                c0.animate.move_to(pos_c1),
                c1.animate.move_to(pos_c0),
                run_time=1.2,
            )
            self.wait(tracker.duration * 0.20)

        # --- Beat 4a: Closing statement ---
        with self.voiceover(
            text=(
                "Both cases illustrate invariance:"
                " a permutation of the input leaves the function's output unchanged."
                " These symmetries are hiding in plain sight throughout the models we use every day."
            )
        ) as tracker:
            box = SurroundingRectangle(formula_nn, color=YELLOW, stroke_width=2, buff=0.22)
            self.play(Create(box), run_time=tracker.duration * 0.45)
            self.play(
                Indicate(formula_nn, color=YELLOW, scale_factor=1.04),
                run_time=tracker.duration * 0.4,
            )
            self.wait(tracker.duration * 0.15)

        # --- Beat 4b: Bridge to Equivariance ---
        with self.voiceover(
            text=(
                "In the next section, we will extend this idea to equivariance,"
                " where the output transforms together with the input."
            )
        ) as tracker:
            bridge_title = Text("Next: Equivariance", font_size=22, color=YELLOW)
            bridge_eq = MathTex(r"f(T(x)) = T(f(x))", font_size=30, color=YELLOW)
            bridge = VGroup(bridge_title, bridge_eq).arrange(DOWN, buff=0.15).next_to(
                box, DOWN, buff=0.35
            )
            self.play(FadeIn(bridge, shift=UP * 0.15), run_time=tracker.duration * 0.6)
            self.wait(tracker.duration * 0.35)

        self.wait(0.8)
        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.6)
        self.wait(0.3)


In [ ]:
%%manim Scene01_TitleCard


In [ ]:
%%manim Scene02_TutorialOverview

In [ ]:
%%manim Scene03_Invariance

In [ ]:
%%manim Scene04_MoreInvariance

# SLIDE 5 TỚI SLIDE 6


In [ ]:
# Scene: Equivariance -- Slides 05 to 06
# Scenes: Scene05_Equivariance, Scene06_EquivExamples
# Utility: colored feature bar representing node embeddings
def make_feature_bar(colors, bar_w=0.21, bar_h=0.45):
    return VGroup(*[
        Rectangle(
            width=bar_w, height=bar_h,
            fill_color=c, fill_opacity=1, stroke_width=0,
        )
        for c in colors
    ]).arrange(RIGHT, buff=0.03)


# Utility: abstract protein structure (colored dots + bonds)
def make_protein(center):
    # 5 nodes matching Scene05 graph colors: BLUE, TEAL, grey, GREEN, ORANGE
    rel = [
        UP * 0.65,
        RIGHT * 0.70 + UP * 0.18,
        RIGHT * 0.52 + DOWN * 0.58,
        LEFT * 0.52 + DOWN * 0.58,
        LEFT * 0.70 + UP * 0.18,
    ]
    colors = [BLUE, TEAL, "#808080", GREEN, ORANGE]
    bond_idx = [(0, 1), (1, 2), (2, 3), (3, 4), (4, 0), (1, 3)]

    dots = [Dot(point=center + p, color=c, radius=0.13) for p, c in zip(rel, colors)]
    bonds = VGroup(*[
        Line(dots[i].get_center(), dots[j].get_center(),
             color=GREY_B, stroke_width=1.5)
        for i, j in bond_idx
    ])
    return VGroup(bonds, VGroup(*dots))


# SCENE 05 -- Equivariance: formula + commutative diagram
class Scene05_Equivariance(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title = Text("Equivariance", font_size=44, color=WHITE).to_edge(UP, buff=0.5)
        self.play(Write(title), run_time=0.8)

        # --- Beat 1: introduce equivariance formula (~69 words, ~23s) ---
        formula = MathTex(
            r"f(", r"T", r"(x)) = ", r"T", r"(f(x))",
            font_size=42,
        )
        formula.set_color(WHITE)
        formula[1].set_color(YELLOW)
        formula[3].set_color(YELLOW)
        formula.next_to(title, DOWN, buff=0.7)

        with self.voiceover(
            text=(
                "With invariance fresh in mind, let's introduce a closely related and more general property: equivariance. "
                "Where invariance requires the output to be completely unaffected by a transformation, "
                "equivariance is more flexible. "
                "It says that a transformation applied to the input will also appear — "
                "in the same structured way — in the output. "
                "Formally, f is equivariant with respect to T if applying T before f "
                "gives the same result as applying f before T."
            )
        ) as tracker:
            self.play(Write(formula), run_time=tracker.duration * 0.50)
            self.wait(tracker.duration * 0.40)

        # shrink formula to upper-left corner before building diagram
        self.play(
            formula.animate.scale(0.65).to_corner(UL, buff=0.65).shift(DOWN * 1.1),
            run_time=0.45,
        )

        # --- Build commutative diagram elements (created before Beat 2) ---
        c_orig = [BLUE, GREEN, RED, YELLOW, TEAL]
        c_perm = [RED, BLUE, TEAL, YELLOW, GREEN]

        bar_X   = make_feature_bar(c_orig).move_to(RIGHT * 1.4 + UP * 1.55)
        bar_TX  = make_feature_bar(c_perm).move_to(RIGHT * 4.5 + UP * 1.55)
        bar_fX  = make_feature_bar(c_orig).move_to(RIGHT * 1.4 + DOWN * 0.90)
        bar_fTX = make_feature_bar(c_perm).move_to(RIGHT * 4.5 + DOWN * 0.90)

        lbl_X  = Text("X",    font_size=20, color=WHITE).next_to(bar_X,   LEFT,  buff=0.16)
        lbl_TX = Text("T(X)", font_size=20, color=WHITE).next_to(bar_TX,  RIGHT, buff=0.16)
        lbl_fX = Text("f(X)", font_size=20, color=WHITE).next_to(bar_fX,  LEFT,  buff=0.16)
        lbl_eq = VGroup(
            Text("f(T(X))",   font_size=19, color=WHITE),
            Text("= T(f(X))", font_size=19, color=GREEN),
        ).arrange(DOWN, buff=0.10).next_to(bar_fTX, RIGHT, buff=0.16)

        arr_top = DashedVMobject(
            Arrow(
                bar_X.get_right(), bar_TX.get_left(),
                buff=0.08, color=WHITE, stroke_width=2,
                max_tip_length_to_length_ratio=0.12,
            ),
            num_dashes=10,
        )
        arr_bot = DashedVMobject(
            Arrow(
                bar_fX.get_right(), bar_fTX.get_left(),
                buff=0.08, color=WHITE, stroke_width=2,
                max_tip_length_to_length_ratio=0.12,
            ),
            num_dashes=10,
        )
        arr_fl = Arrow(
            bar_X.get_bottom(), bar_fX.get_top(),
            buff=0.08, color=WHITE, stroke_width=2,
            max_tip_length_to_length_ratio=0.12,
        )
        arr_fr = Arrow(
            bar_TX.get_bottom(), bar_fTX.get_top(),
            buff=0.08, color=WHITE, stroke_width=2,
            max_tip_length_to_length_ratio=0.12,
        )

        lbl_p1 = Text("permute", font_size=15, color=GREY_B).next_to(arr_top, UP,   buff=0.07)
        lbl_p2 = Text("permute", font_size=15, color=GREY_B).next_to(arr_bot, UP,   buff=0.07)
        lbl_f1 = Text("f", font_size=18, color=WHITE).next_to(arr_fl, LEFT,  buff=0.09)
        lbl_f2 = Text("f", font_size=18, color=WHITE).next_to(arr_fr, RIGHT, buff=0.09)

        graph = Graph(
            [1, 2, 3, 4, 5],
            [(1, 2), (2, 3), (3, 4), (4, 5), (1, 5), (2, 4)],
            layout="circular",
            layout_scale=1.15,
            vertex_config={
                1: {"fill_color": BLUE,      "radius": 0.13},
                2: {"fill_color": TEAL,      "radius": 0.13},
                3: {"fill_color": "#808080", "radius": 0.13},
                4: {"fill_color": GREEN,     "radius": 0.13},
                5: {"fill_color": ORANGE,    "radius": 0.13},
            },
            edge_config={"stroke_width": 1.5, "stroke_color": WHITE},
        ).move_to(LEFT * 4.0 + DOWN * 0.20)

        # --- Beat 2: commutative diagram walkthrough (~133 words, ~45s) ---
        with self.voiceover(
            text=(
                "The cleanest way to build intuition here is through a commutative diagram. "
                "Imagine a graph with colored nodes — each node carrying a feature vector. "
                "We have two paths. "
                "Path one: first permute the node order with transformation T, "
                "producing T of X — "
                "then apply our function f to arrive at f of T of X, shown at the bottom right. "
                "Path two: apply f first to get f of X — "
                "then apply the same permutation T to reach T of f of X. "
                "Equivariance says both paths produce the same result. "
                "F of T of X equals T of f of X. "
                "The function f and the transformation T are interchangeable in this diagram — "
                "apply-then-permute is identical to permute-then-apply. "
                "Whichever road you take through the diagram, "
                "you always arrive at exactly the same place."
            )
        ) as tracker:
            self.play(
                FadeIn(graph), FadeIn(bar_X), FadeIn(lbl_X),
                run_time=tracker.duration * 0.12,
            )
            self.play(
                Create(arr_top), FadeIn(lbl_p1),
                FadeIn(bar_TX), FadeIn(lbl_TX),
                run_time=tracker.duration * 0.14,
            )
            self.play(
                GrowArrow(arr_fl), FadeIn(lbl_f1),
                FadeIn(bar_fX), FadeIn(lbl_fX),
                run_time=tracker.duration * 0.12,
            )
            self.play(
                GrowArrow(arr_fr), FadeIn(lbl_f2),
                FadeIn(bar_fTX),
                run_time=tracker.duration * 0.11,
            )
            self.play(
                Create(arr_bot), FadeIn(lbl_p2), FadeIn(lbl_eq),
                run_time=tracker.duration * 0.11,
            )
            self.play(
                Indicate(bar_fTX, color=GREEN, scale_factor=1.08),
                Indicate(lbl_eq,  color=GREEN, scale_factor=1.04),
                run_time=tracker.duration * 0.10,
            )
            self.wait(tracker.duration * 0.30)

        # --- Beat 3: closing highlight (~35 words, ~12s) ---
        diagram_rect_group = VGroup(
            bar_X, bar_TX, bar_fX, bar_fTX,
            lbl_X, lbl_TX, lbl_fX, lbl_eq,
            arr_top, arr_bot, arr_fl, arr_fr,
            lbl_p1, lbl_p2, lbl_f1, lbl_f2,
        )
        box_graph   = SurroundingRectangle(graph,               color=YELLOW, buff=0.22, stroke_width=2)
        box_diagram = SurroundingRectangle(diagram_rect_group,  color=YELLOW, buff=0.22, stroke_width=2)

        with self.voiceover(
            text=(
                "This is why equivariant functions are so powerful for graphs and point clouds: "
                "whatever permutation you apply to the input will show up, unchanged, in the output — "
                "the network structure and the symmetry are perfectly synchronized."
            )
        ) as tracker:
            self.play(Create(box_graph), Create(box_diagram), run_time=tracker.duration * 0.30)
            self.wait(tracker.duration * 0.60)

        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.6)
        self.wait(0.3)


# SCENE 06 -- Equivariance: real-world examples + invariance insight
class Scene06_EquivExamples(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title = Text("Equivariance: Examples", font_size=44, color=WHITE).to_edge(UP, buff=0.5)
        formula_small = MathTex(
            r"f(T(x)) = T(f(x))", font_size=26, color=BLUE,
        ).next_to(title, DOWN, buff=0.28)

        self.play(Write(title), run_time=0.7)
        self.play(FadeIn(formula_small), run_time=0.4)

        # --- Beat 1: protein rotation (~80 words, ~27s) ---
        protein_a = make_protein(center=LEFT * 3.8 + DOWN * 0.50)
        protein_b = make_protein(center=LEFT * 3.8 + DOWN * 0.50)
        protein_b.rotate(PI / 4)
        protein_b.move_to(LEFT * 0.30 + DOWN * 0.50)

        arr_rotate = Arrow(
            protein_a.get_right() + RIGHT * 0.05,
            protein_b.get_left()  + LEFT  * 0.05,
            buff=0.12, color=WHITE, stroke_width=2,
            max_tip_length_to_length_ratio=0.14,
        )
        lbl_rot = Text("rotate 45 deg", font_size=16, color=GREY_B).next_to(
            arr_rotate, UP, buff=0.08,
        )
        lbl_a = Text("X", font_size=20, color=WHITE).next_to(protein_a, DOWN, buff=0.20)
        lbl_b = Text("T(X)", font_size=20, color=WHITE).next_to(protein_b, DOWN, buff=0.20)

        with self.voiceover(
            text=(
                "Let's make equivariance concrete with real-world examples. "
                "First, consider a protein structure prediction network. "
                "A protein is a three-dimensional arrangement of atoms. "
                "If you rotate the entire protein in 3D space, "
                "you're applying a rotation transformation T to the input. "
                "An equivariant model will produce an output "
                "that has been rotated by exactly the same rotation. "
                "Predicted atom coordinates, atomic forces, binding sites — "
                "all rotate in lock-step with the input structure. "
                "This is the property molecular dynamics simulations "
                "need to be physically meaningful."
            )
        ) as tracker:
            self.play(FadeIn(protein_a), FadeIn(lbl_a),
                      run_time=tracker.duration * 0.15)
            self.play(GrowArrow(arr_rotate), FadeIn(lbl_rot),
                      run_time=tracker.duration * 0.12)
            self.play(FadeIn(protein_b), FadeIn(lbl_b),
                      run_time=tracker.duration * 0.18)
            self.wait(tracker.duration * 0.55)

        # --- Beat 2: robotic arm (~45 words, ~15s) ---
        self.play(
            FadeOut(protein_a), FadeOut(protein_b),
            FadeOut(arr_rotate), FadeOut(lbl_rot),
            FadeOut(lbl_a), FadeOut(lbl_b),
            run_time=0.4,
        )

        # L-shape arm: base (vertical) + upper (horizontal) + forearm (vertical)
        p_base  = LEFT * 3.8 + DOWN * 1.20
        p_elbow = LEFT * 3.8 + UP   * 0.20
        p_wrist = LEFT * 2.3 + UP   * 0.20
        p_tip   = LEFT * 2.3 + UP   * 1.20

        arm = VGroup(
            Line(p_base,  p_elbow, color=BLUE_D, stroke_width=6),
            Line(p_elbow, p_wrist, color=BLUE_D, stroke_width=6),
            Line(p_wrist, p_tip,   color=BLUE_D, stroke_width=6),
            Dot(p_base,  color=GREY_B, radius=0.14),
            Dot(p_elbow, color=GREY_B, radius=0.14),
            Dot(p_wrist, color=GREY_B, radius=0.14),
            Dot(p_tip,   color=GREY_B, radius=0.14),
        )

        arm_rot = arm.copy()
        arm_rot.rotate(PI / 5)
        arm_rot.move_to(RIGHT * 1.80 + DOWN * 0.10)
        arm_rot.set_color(GREEN)

        arr_between = Arrow(
            arm.get_right()     + RIGHT * 0.10,
            arm_rot.get_left()  + LEFT  * 0.10,
            buff=0.12, color=WHITE, stroke_width=2,
            max_tip_length_to_length_ratio=0.14,
        )
        lbl_arm_a = Text("input", font_size=17, color=WHITE ).next_to(arm,     DOWN, buff=0.20)
        lbl_arm_b = Text("output rotates with input",
                          font_size=17, color=GREEN).next_to(arm_rot, DOWN, buff=0.20)
        arm_eq = MathTex(
            r"f(T(X)) = T(f(X))", font_size=22, color=BLUE,
        ).next_to(VGroup(arm, arm_rot), DOWN, buff=0.48)

        with self.voiceover(
            text=(
                "Second example: a robotic arm. "
                "The arm operates in 3D space — "
                "if you rotate your entire frame of reference, "
                "an equivariant model's output rotates with it in exactly the same way. "
                "Whatever transformation you apply to the input, "
                "the output mirrors it faithfully."
            )
        ) as tracker:
            self.play(FadeIn(arm), FadeIn(lbl_arm_a),
                      run_time=tracker.duration * 0.25)
            self.play(GrowArrow(arr_between),
                      FadeIn(arm_rot), FadeIn(lbl_arm_b),
                      run_time=tracker.duration * 0.30)
            self.play(FadeIn(arm_eq),
                      run_time=tracker.duration * 0.22)
            self.wait(tracker.duration * 0.18)

        # --- Beat 3: invariance is special case (~64 words, ~22s) ---
        self.play(
            FadeOut(arm), FadeOut(arm_rot),
            FadeOut(arr_between),
            FadeOut(lbl_arm_a), FadeOut(lbl_arm_b),
            FadeOut(arm_eq),
            run_time=0.4,
        )

        insight_title = Text("Key Insight", font_size=26, color=YELLOW, weight=BOLD)
        insight_body  = Text(
            "Invariance  =  equivariance\nwith scalar output",
            font_size=29, color=WHITE, line_spacing=1.35,
        )
        insight_group = VGroup(insight_title, insight_body).arrange(DOWN, buff=0.35)
        insight_group.move_to(UP * 0.30)
        insight_box = SurroundingRectangle(
            insight_group,
            color=YELLOW, buff=0.22, corner_radius=0.12, stroke_width=2.5,
        )
        scalar_demo = VGroup(
            MathTex(r"f(x) \in \mathbb{R}:", font_size=23, color=GREY_B),
            MathTex(r"T(f(x)) = f(x)",       font_size=25, color=GREEN),
        ).arrange(RIGHT, buff=0.30)
        scalar_demo.next_to(insight_group, DOWN, buff=0.38)

        with self.voiceover(
            text=(
                "This leads to a key insight worth pausing on. "
                "What happens when the output of our function is just a single scalar — "
                "a real-valued prediction like energy or a classification score? "
                "A permutation or rotation of a scalar leaves it unchanged. "
                "So for scalar outputs, equivariance collapses directly to invariance. "
                "Invariance is simply a special case of equivariance "
                "where the output space is one-dimensional."
            )
        ) as tracker:
            self.play(Write(insight_title), Create(insight_box),
                      run_time=tracker.duration * 0.20)
            self.play(Write(insight_body),
                      run_time=tracker.duration * 0.22)
            self.play(FadeIn(scalar_demo),
                      run_time=tracker.duration * 0.25)
            self.play(Indicate(insight_box, color=YELLOW, scale_factor=1.04),
                      run_time=tracker.duration * 0.18)
            self.wait(tracker.duration * 0.10)

        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.6)
        self.wait(0.3)


In [ ]:
%%manim Scene05_Equivariance

In [ ]:
%%manim Scene06_EquivExamples

# SLIDE 7 TỚI SLIDE 8

In [ ]:
# Scene: Why Equivariant Models + Active Area -- Slides 07 to 08
# Scenes: Scene07_WhyEquivariant, Scene08_ActiveArea
# SCENE 07 -- Why equivariant/invariant models?
class Scene07_WhyEquivariant(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title = Text(
            "Equivariant/Invariant Models: Why?",
            font_size=40, color=WHITE,
        ).to_edge(UP, buff=0.45)
        self.play(Write(title), run_time=0.8)

        # --- Beat 1: fewer degrees of freedom (~50 words, ~17s) ---
        bullet1_left  = Text("Fewer degrees of freedom:  ", font_size=28, color=WHITE)
        bullet1_right = Text("better data efficiency",     font_size=28, color=ORANGE)
        bullet1 = VGroup(bullet1_left, bullet1_right).arrange(RIGHT, buff=0.0)
        bullet1.next_to(title, DOWN, buff=0.65).to_edge(LEFT, buff=0.7)

        with self.voiceover(
            text=(
                "Now that we understand what equivariance and invariance are, "
                "the natural question is: why should we care? "
                "The answer comes down to two fundamental benefits. "
                "First, symmetry-constrained models have fewer free parameters to learn — "
                "because they cannot encode representations that violate the known symmetry."
            )
        ) as tracker:
            self.play(FadeIn(bullet1, shift=RIGHT * 0.15), run_time=tracker.duration * 0.45)
            self.wait(tracker.duration * 0.55)

        # --- Build plot objects (right side) before Beat 2a ---
        ax = Axes(
            x_range=[0, 4, 1],
            y_range=[0, 3, 1],
            x_length=5.2,
            y_length=3.2,
            axis_config={"color": GREY_B, "stroke_width": 1.5, "include_ticks": False},
            tips=False,
        ).move_to(RIGHT * 2.6 + DOWN * 0.70)

        x_labels = VGroup(*[
            Text(lbl, font_size=13, color=GREY_B).next_to(
                ax.c2p(xi, 0), DOWN, buff=0.12
            )
            for xi, lbl in zip([0.5, 1.5, 2.5, 3.5],
                               ["10^16", "10^17", "10^18", "10^19"])
        ])
        y_labels = VGroup(*[
            Text(lbl, font_size=13, color=GREY_B).next_to(
                ax.c2p(0, yi), LEFT, buff=0.12
            )
            for yi, lbl in zip([2.4, 0.8], ["10^-4", "10^-5"])
        ])
        x_title = Text("Training compute", font_size=14, color=GREY_B).next_to(
            ax, DOWN, buff=0.55
        )
        y_title = Text("Loss", font_size=14, color=GREY_B).next_to(
            ax, LEFT, buff=0.32
        ).rotate(PI / 2)
        axis_labels = VGroup(x_labels, y_labels, x_title, y_title)
        baseline_pts = [ax.c2p(0.2, 2.55), ax.c2p(3.8, 1.10)]
        baseline_line = DashedLine(
            baseline_pts[0], baseline_pts[1],
            color=BLUE, stroke_width=2.5, dash_length=0.15,
        )
        equiv_pts = [ax.c2p(0.2, 1.85), ax.c2p(3.8, 0.30)]
        equiv_line = Line(equiv_pts[0], equiv_pts[1], color=GREEN, stroke_width=2.5)

        np.random.seed(42)
        baseline_dots = VGroup(*[
            Dot(
                ax.c2p(
                    0.2 + t * 3.6 + np.random.uniform(-0.15, 0.15),
                    2.55 - t * 1.45 + np.random.uniform(-0.22, 0.22),
                ),
                color=BLUE, radius=0.055,
            )
            for t in np.linspace(0, 1, 8)
        ])
        equiv_dots = VGroup(*[
            Dot(
                ax.c2p(
                    0.2 + t * 3.6 + np.random.uniform(-0.15, 0.15),
                    1.85 - t * 1.55 + np.random.uniform(-0.20, 0.20),
                ),
                color=GREEN, radius=0.055,
            )
            for t in np.linspace(0, 1, 8)
        ])

        leg_baseline = VGroup(
            DashedLine(ORIGIN, RIGHT * 0.38, color=BLUE, stroke_width=2, dash_length=0.10),
            Text("Baseline",    font_size=13, color=BLUE),
        ).arrange(RIGHT, buff=0.08)
        leg_equiv = VGroup(
            Line(ORIGIN, RIGHT * 0.38, color=GREEN, stroke_width=2),
            Text("Equivariant", font_size=13, color=GREEN),
        ).arrange(RIGHT, buff=0.08)
        legend = VGroup(leg_baseline, leg_equiv).arrange(DOWN, aligned_edge=LEFT, buff=0.12)
        legend.next_to(ax, UP, buff=0.10).align_to(ax, RIGHT)

        # citations placed below x_title with enough spacing to avoid overlap
        batzner_cite = Text(
            "Batzner et al, Nature Communications 2022",
            font_size=11, color=GREY_B,
        ).next_to(x_title, DOWN, buff=0.18)
        brehmer_cite = Text(
            "Brehmer et al 2025",
            font_size=11, color=GREY_B,
        ).next_to(batzner_cite, DOWN, buff=0.14)

        quote_line1 = Text(
            '"NequIP outperforms existing models with up to',
            font_size=16, color=WHITE,
        )
        quote_line2 = Text(
            "three orders of magnitude fewer training data",
            font_size=16, color=YELLOW, weight=BOLD,
        )
        quote_line3 = Text(
            '[...] enables high-fidelity molecular dynamics"',
            font_size=16, color=WHITE,
        )
        quote_block = VGroup(quote_line1, quote_line2, quote_line3).arrange(
            DOWN, aligned_edge=LEFT, buff=0.10
        )
        quote_block.next_to(bullet1, DOWN, buff=0.38).align_to(bullet1, LEFT)

        # --- Beat 2a: NequIP paper + quote  ---
        with self.voiceover(
            text=(
                "This leads directly to better data efficiency. "
                "The NequIP paper — by Batzner and colleagues, Nature Communications 2022 — "
                "demonstrated this dramatically. "
                "Their equivariant model outperforms existing baselines with up to "
                "three orders of magnitude fewer training data points."
            )
        ) as tracker:
            self.play(FadeIn(quote_block), run_time=tracker.duration * 0.50)
            self.wait(tracker.duration * 0.50)

        # --- Beat 2b: describe plot + Brehmer citation  ---
        with self.voiceover(
            text=(
                "This log-log plot from Brehmer and colleagues, 2025, shows it clearly. "
                "The x-axis is training compute, the y-axis is prediction loss. "
                "The green equivariant line sits consistently below the blue dashed baseline — "
                "lower error with far less data."
            )
        ) as tracker:
            self.play(Create(ax),             run_time=tracker.duration * 0.18)
            self.play(FadeIn(axis_labels),    run_time=tracker.duration * 0.12)
            self.play(
                Create(baseline_line), FadeIn(baseline_dots), FadeIn(leg_baseline),
                run_time=tracker.duration * 0.22,
            )
            self.play(
                Create(equiv_line), FadeIn(equiv_dots), FadeIn(leg_equiv),
                run_time=tracker.duration * 0.22,
            )
            self.play(
                FadeIn(batzner_cite), FadeIn(brehmer_cite),
                run_time=tracker.duration * 0.15,
            )
            self.wait(tracker.duration * 0.11)

        # Ẩn biểu đồ sau khi đã hiển thị xong
        chart_group = VGroup(
            ax, axis_labels,
            baseline_line, baseline_dots, leg_baseline,
            equiv_line, equiv_dots, leg_equiv,
            batzner_cite, brehmer_cite,
        )
        self.play(FadeOut(chart_group), run_time=0.5)

        # --- Beat 3: robustness + OOD generalization ---
        bullet2_left  = Text("Robustness", font_size=28, color=ORANGE)
        bullet2_right = Text(",  OOD generalization", font_size=28, color=WHITE)
        bullet2 = VGroup(bullet2_left, bullet2_right).arrange(RIGHT, buff=0.0)
        bullet2.next_to(quote_block, DOWN, buff=0.40).align_to(bullet1, LEFT)

        with self.voiceover(
            text=(
                "The second benefit is robustness and out-of-distribution generalization. "
                "Because the symmetry is encoded into the architecture by construction — "
                "not learned from data — the model generalizes reliably to inputs never seen "
                "during training, as long as those inputs obey the same underlying symmetry."
            )
        ) as tracker:
            self.play(FadeIn(bullet2, shift=RIGHT * 0.15), run_time=tracker.duration * 0.35)
            self.wait(tracker.duration * 0.65)

        # --- Beat 4: closing highlight ---
        summary_rect = SurroundingRectangle(
            VGroup(bullet1, bullet2),
            color=YELLOW, buff=0.20, corner_radius=0.10, stroke_width=2,
        )

        with self.voiceover(
            text=(
                "In short: if your data has a known symmetry, "
                "encoding it directly into the architecture is almost always worth it. "
                "You get more from less data, and far stronger generalization."
            )
        ) as tracker:
            self.play(Create(summary_rect), run_time=tracker.duration * 0.35)
            self.play(
                ShowPassingFlash(summary_rect.copy().set_color(GOLD), time_width=0.5),
                run_time=tracker.duration * 0.30,
            )
            self.wait(tracker.duration * 0.35)

        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.6)
        self.wait(0.3)


# SCENE 08 -- This has been an active area
class Scene08_ActiveArea(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title = Text(
            "This Has Been an Active Area...",
            font_size=40, color=WHITE,
        ).to_edge(UP, buff=0.45)
        self.play(Write(title), run_time=0.7)

        # --- Paper cards ---
        gdl_title_t = Text(
            "Geometric Deep Learning",
            font_size=17, color=WHITE, weight=BOLD,
        )
        gdl_subtitle_t = Text(
            "Grids, Groups, Graphs,\nGeodesics, and Gauges",
            font_size=16, color=WHITE, weight=BOLD,
            line_spacing=1.2,
        )
        gdl_authors_t = Text(
            "Bronstein  Bruna  Cohen  Velickovic",
            font_size=12, color=GREY_B,
        )
        gdl_inner = VGroup(gdl_title_t, gdl_subtitle_t, gdl_authors_t).arrange(DOWN, buff=0.18)
        gdl_box   = SurroundingRectangle(gdl_inner, color=WHITE, buff=0.28, stroke_width=1.8)
        gdl_card  = VGroup(gdl_box, gdl_inner)
        gdl_card.move_to(LEFT * 3.00 + DOWN * 1.00)

        tut_title_t = Text(
            "Tutorial: Equivariant Networks",
            font_size=16, color="#9B59B6", weight=BOLD,
        )
        tut_presenters_t = Text(
            "Risi Kondor,  Taco Cohen",
            font_size=13, color=WHITE,
        )
        tut_date_t = Text(
            "Dec 7, 2020  @  11:30-14:00 CET",
            font_size=12, color=GREY_B,
        )
        tut_inner = VGroup(tut_title_t, tut_presenters_t, tut_date_t).arrange(DOWN, buff=0.16)
        tut_box   = SurroundingRectangle(tut_inner, color="#9B59B6", buff=0.26, stroke_width=1.8)
        tut_card  = VGroup(tut_box, tut_inner)
        tut_card.move_to(RIGHT * 2.80 + DOWN * 1.00)

        # --- Beat 1a: GDL paper identity ---
        with self.voiceover(
            text=(
                "The foundations of this field were laid in a landmark paper "
                "by Bronstein, Bruna, Cohen, and Velickovic — "
                "Geometric Deep Learning: Grids, Groups, Graphs, Geodesics, and Gauges."
            )
        ) as tracker:
            self.play(FadeIn(gdl_card), run_time=tracker.duration * 0.60)
            self.wait(tracker.duration * 0.40)

        # --- Beat 1b: GDL paper significance  ---
        with self.voiceover(
            text=(
                "The title is not accidental: those five G-words name the core domains "
                "the framework unifies. "
                "It gave CNNs, GNNs, and Transformers a single symmetry-based blueprint "
                "and a common mathematical language."
            )
        ) as tracker:
            self.play(
                Indicate(gdl_box, color=WHITE, scale_factor=1.04),
                run_time=tracker.duration * 0.35,
            )
            self.wait(tracker.duration * 0.65)

        # --- Beat 2: NeurIPS 2020 Tutorial  ---
        with self.voiceover(
            text=(
                "Around the same time, at NeurIPS 2020, "
                "Risi Kondor and Taco Cohen ran a full tutorial on equivariant networks — "
                "a clear signal that the community recognized this as a major research direction "
                "worth a dedicated tutorial track."
            )
        ) as tracker:
            self.play(FadeIn(tut_card),                                     run_time=tracker.duration * 0.40)
            self.play(Indicate(tut_box, color="#9B59B6", scale_factor=1.04), run_time=tracker.duration * 0.28)
            self.wait(tracker.duration * 0.32)

        # FadeOut cards before bullet section
        self.play(FadeOut(gdl_card), FadeOut(tut_card), run_time=0.45)

        # Bullet section
        devs_label = Text(
            "Many developments since!",
            font_size=26, color=WHITE,
        ).next_to(title, DOWN, buff=0.55)

        bullets_data = [
            ("New modeling strategies",                        WHITE),
            ("New areas (e.g. neural parameter symmetries,",  WHITE),
            ("   weight space learning)",                      GREY_B),
            ("New theory",                                     WHITE),
            ("New observations, insights, discussions, ...",  WHITE),
        ]
        bullet_items = VGroup(*[
            Text(txt, font_size=22, color=col)
            for txt, col in bullets_data
        ]).arrange(DOWN, aligned_edge=LEFT, buff=0.22)
        bullet_items.next_to(devs_label, DOWN, buff=0.35).to_edge(LEFT, buff=0.90)

        b0 = bullet_items[0]
        b1 = VGroup(bullet_items[1], bullet_items[2])
        b2 = bullet_items[3]
        b3 = bullet_items[4]

        # --- Beat 3a: first 2 bullets ---
        with self.voiceover(
            text=(
                "And since those foundational works, the field has moved fast. "
                "New modeling strategies have emerged for building equivariant architectures "
                "more efficiently. "
                "Entirely new problem areas have opened up — most notably "
                "neural parameter symmetries and weight space learning, "
                "where the symmetries live inside the model weights themselves."
            )
        ) as tracker:
            self.play(Write(devs_label),                         run_time=tracker.duration * 0.20)
            self.play(FadeIn(b0, shift=RIGHT * 0.15),            run_time=tracker.duration * 0.22)
            self.play(FadeIn(b1, shift=RIGHT * 0.15),            run_time=tracker.duration * 0.28)
            self.wait(tracker.duration * 0.30)

        # --- Beat 3b: last 2 bullets (~35 words, ~12s) ---
        with self.voiceover(
            text=(
                "New theory has sharpened our understanding of what equivariant models "
                "can and cannot express. "
                "And ongoing empirical observations, insights, and open discussions "
                "continue to push the frontier forward."
            )
        ) as tracker:
            self.play(FadeIn(b2, shift=RIGHT * 0.15), run_time=tracker.duration * 0.28)
            self.play(FadeIn(b3, shift=RIGHT * 0.15), run_time=tracker.duration * 0.28)
            self.wait(tracker.duration * 0.44)

        # --- Beat 4: closing bridge  ---
        all_bullets = VGroup(b0, b1, b2, b3)
        closing_rect = SurroundingRectangle(
            all_bullets, color=YELLOW, buff=0.22, corner_radius=0.10, stroke_width=2,
        )

        with self.voiceover(
            text=(
                "This tutorial sits right at the frontier of all four of these threads. "
                "By the end, you will have the tools to understand, implement, "
                "and reason about equivariant models — "
                "including the newest strategies that go well beyond "
                "the original geometric deep learning blueprint."
            )
        ) as tracker:
            self.play(Create(closing_rect), run_time=tracker.duration * 0.28)
            self.play(
                ShowPassingFlash(closing_rect.copy().set_color(GOLD), time_width=0.5),
                run_time=tracker.duration * 0.28,
            )
            self.wait(tracker.duration * 0.44)

        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.6)
        self.wait(0.3)


In [ ]:
%manim Scene07_WhyEquivariant

In [ ]:
%manim Scene08_ActiveArea

# SLIDE 9 TỚI SLIDE 10

In [ ]:
# Scene: Roadmap + Groups -- Slide 09 to 10
# Scenes: Scene09_Roadmap, Scene10_Groups
# Scene09_Roadmap
class Scene09_Roadmap(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        # --- Title (shown at start, persists) ---
        title = Text("Roadmap", font_size=44, color=WHITE).to_edge(UP, buff=0.5)

        with self.voiceover(
            text=(
                "Before we dive into the math, let us step back and orient ourselves. "
                "This tutorial is structured in two parts."
            )
        ) as tracker:
            # 1a. Write title
            self.play(Write(title), run_time=tracker.duration * 0.28)
            # 1b. Underline + both part headers appear together
            underline = Underline(title, color=GREY_B, stroke_width=1.5)
            part1_hdr = Text("Part 1", font_size=28, color=BLUE, weight=BOLD)
            part2_hdr = Text("Part 2", font_size=28, color=BLUE, weight=BOLD)
            part1_hdr.move_to(LEFT * 3.2 + UP * 0.45)
            part2_hdr.move_to(RIGHT * 3.2 + UP * 0.45)
            self.play(
                GrowFromCenter(underline),
                FadeIn(part1_hdr, shift=RIGHT * 0.4),
                FadeIn(part2_hdr, shift=LEFT * 0.4),
                run_time=tracker.duration * 0.42,
            )
            # 1c. Vertical divider grows
            divider = DashedLine(
                start=UP * 0.95, end=DOWN * 2.6,
                color=GREY_B, stroke_width=1.2, dash_length=0.12,
            )
            self.play(Create(divider), run_time=tracker.duration * 0.18)
            self.wait(tracker.duration * 0.12)

        # Beat 2 -- Part 1 bullet points (fade in one by one)
        p1_b1 = Text("- Introduction and basics",      font_size=22, color=WHITE)
        p1_b2 = Text("- Techniques for equivariance",  font_size=22, color=WHITE)
        p1_b3 = Text("  (with examples)",               font_size=18, color=GREY_B)
        p1_bullets = VGroup(p1_b1, p1_b2, p1_b3).arrange(DOWN, aligned_edge=LEFT, buff=0.18)
        p1_bullets.next_to(part1_hdr, DOWN, buff=0.38)

        with self.voiceover(
            text=(
                "Part one covered the introduction and basics of symmetry -- "
                "invariance and equivariance -- and techniques for building "
                "equivariant models, with concrete examples."
            )
        ) as tracker:
            self.play(FadeIn(p1_b1, shift=UP * 0.12), run_time=tracker.duration * 0.30)
            self.play(FadeIn(p1_b2, shift=UP * 0.12), run_time=tracker.duration * 0.30)
            self.play(FadeIn(p1_b3, shift=UP * 0.12), run_time=tracker.duration * 0.20)
            self.wait(tracker.duration * 0.20)

        # Beat 3 -- Part 2 bullet points (fade in one by one)
        p2_b1 = Text("- Neural parameter symmetries", font_size=22, color=WHITE)
        p2_b2 = Text("  and recent directions",        font_size=18, color=GREY_B)
        p2_b3 = Text("- Theory results and directions",font_size=22, color=WHITE)
        p2_b4 = Text("- A bit of discussion",          font_size=22, color=WHITE)
        p2_bullets = VGroup(p2_b1, p2_b2, p2_b3, p2_b4).arrange(DOWN, aligned_edge=LEFT, buff=0.18)
        p2_bullets.next_to(part2_hdr, DOWN, buff=0.38)

        with self.voiceover(
            text=(
                "Part two goes deeper: neural parameter symmetries and other recent directions, "
                "theoretical results, and a broader discussion of the field."
            )
        ) as tracker:
            self.play(FadeIn(p2_b1, shift=UP * 0.12), run_time=tracker.duration * 0.22)
            self.play(FadeIn(p2_b2, shift=UP * 0.12), run_time=tracker.duration * 0.18)
            self.play(FadeIn(p2_b3, shift=UP * 0.12), run_time=tracker.duration * 0.22)
            self.play(FadeIn(p2_b4, shift=UP * 0.12), run_time=tracker.duration * 0.22)
            self.wait(tracker.duration * 0.16)

        # Beat 4 -- Invariance recap formula
        recap_inv_lbl   = Text("Invariance",   font_size=18, color=WHITE)
        formula_inv     = MathTex(r"f(g \cdot x) = f(x)", font_size=28, color=BLUE)
        recap_equiv_lbl = Text("Equivariance", font_size=18, color=WHITE)
        formula_equiv   = MathTex(r"f(g \cdot x) = g \cdot f(x)", font_size=28, color=GREEN)
        inv_row   = VGroup(recap_inv_lbl,   formula_inv  ).arrange(RIGHT, buff=0.28, aligned_edge=DOWN)
        equiv_row = VGroup(recap_equiv_lbl, formula_equiv).arrange(RIGHT, buff=0.28, aligned_edge=DOWN)
        all_rows  = VGroup(inv_row, equiv_row).arrange(DOWN, buff=0.25, aligned_edge=LEFT)
        all_rows.next_to(title, DOWN, buff=0.55)
        recap_box = SurroundingRectangle(
            all_rows, color=YELLOW, stroke_width=1.8,
            corner_radius=0.14, buff=0.22,
        )

        with self.voiceover(
            text=(
                "Throughout, we keep coming back to two key definitions. "
                "The first is invariance: f of g-dot-x equals f of x. "
                "Applying a transformation to the input does not change the output."
            )
        ) as tracker:
            # Write inv_row only -- no box yet (box appears after equiv_row)
            self.play(
                Write(recap_inv_lbl), Write(formula_inv),
                run_time=tracker.duration * 0.50,
            )
            self.play(
                Indicate(formula_inv, color=YELLOW, scale_factor=1.06),
                run_time=tracker.duration * 0.30,
            )
            self.wait(tracker.duration * 0.20)

        # Beat 5 -- Equivariance recap formula
        with self.voiceover(
            text=(
                "The second is equivariance: f of g-dot-x equals g-dot-f of x. "
                "The transformation propagates through the function. "
                "Keep both in mind -- they are the core of everything we will see."
            )
        ) as tracker:
            self.play(
                Write(recap_equiv_lbl), Write(formula_equiv),
                run_time=tracker.duration * 0.38,
            )
            # Now both rows exist -- Create full box
            self.play(Create(recap_box),                          run_time=tracker.duration * 0.18)
            self.play(
                Indicate(formula_equiv, color=YELLOW, scale_factor=1.06),
                run_time=tracker.duration * 0.22,
            )
            self.wait(tracker.duration * 0.22)

        # Beat 6 -- Disclaimer
        disclaimer = Text(
            "Disclaimer: cannot cover all works",
            font_size=17, color=GREY_B, slant=ITALIC,
        ).to_corner(DR, buff=0.35)

        with self.voiceover(
            text=(
                "A quick disclaimer: this tutorial covers wide ground, "
                "but the field is vast -- many excellent works lie beyond our scope."
            )
        ) as tracker:
            self.play(FadeIn(disclaimer), run_time=tracker.duration * 0.40)
            self.wait(tracker.duration * 0.60)

        # --- End Scene ---
        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.6)
        self.wait(0.3)


# Scene10_Groups
class Scene10_Groups(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        # --- Title -- kept on screen throughout the scene ---
        title = Text("Groups", font_size=44, color=WHITE).to_edge(UP, buff=0.5)
        self.play(Write(title), run_time=0.70)

        # Beat 1 -- Motivation sentence; word "group" highlighted yellow
        motivation = Text(
            "Invariance/equivariance for transformations that form a group.",
            font_size=23, color=WHITE,
            t2c={"group": YELLOW},
        )
        motivation.next_to(title, DOWN, buff=0.60)

        with self.voiceover(
            text=(
                "In the last part, we talked about symmetry transformations. "
                "Now, a key insight: the transformations we care about almost always "
                "form a mathematical structure called a group."
            )
        ) as tracker:
            self.play(FadeIn(motivation), run_time=tracker.duration * 0.45)
            self.play(
                Indicate(motivation, color=YELLOW, scale_factor=1.02),
                run_time=tracker.duration * 0.30,
            )
            self.wait(tracker.duration * 0.25)

        # Beat 2 -- Formal definition: (G, o) with surrounding rect
        self.play(FadeOut(motivation), run_time=0.35)

        group_formula   = MathTex(r"(G,\; \circ)", font_size=46, color=BLUE)
        group_sublabel  = Text("A set  +  a composition rule", font_size=22, color=WHITE)
        group_content   = VGroup(group_formula, group_sublabel).arrange(DOWN, buff=0.28)
        group_content.next_to(title, DOWN, buff=0.90)
        group_rect = SurroundingRectangle(
            group_content, color=BLUE, stroke_width=1.8,
            corner_radius=0.12, buff=0.32,
        )

        with self.voiceover(
            text=(
                "Formally, a group is a set G, together with a binary operation -- "
                "a way to compose two elements -- satisfying four properties."
            )
        ) as tracker:
            self.play(Write(group_formula),          run_time=tracker.duration * 0.32)
            self.play(FadeIn(group_sublabel),        run_time=tracker.duration * 0.28)
            self.play(Create(group_rect),            run_time=tracker.duration * 0.22)
            self.wait(tracker.duration * 0.18)

        self.play(FadeOut(group_content), FadeOut(group_rect), run_time=0.35)

        # Helper -- build axiom card; accepts pre-built MathTex object
        def make_card(ax_title, ax_formula_mob):
            t = Text(ax_title, font_size=20, color=YELLOW, weight=BOLD)
            content = VGroup(t, ax_formula_mob).arrange(DOWN, buff=0.18)
            rect = SurroundingRectangle(
                content, color=BLUE, stroke_width=1.5,
                corner_radius=0.10, buff=0.22,
            )
            return VGroup(rect, content)

        form1 = MathTex(r"(gh)\ell", r"=", r"g(h\ell)", font_size=24, color=WHITE)
        card1 = make_card("Associativity", form1)
        form2 = MathTex(
            r"\exists\,", r"e", r"\in G :\; eg = ge = g",
            font_size=24, color=WHITE,
        )
        card2 = make_card("Identity", form2)
        form3 = MathTex(
            r"\forall g\in G\;\exists\,", r"g^{-1}",
            r":\;gg^{-1}=g^{-1}g=", r"e",
            font_size=19, color=WHITE,
        )
        card3 = make_card("Inverse", form3)

        form4 = MathTex(r"g,\, h \in G \implies gh \in G", font_size=24, color=WHITE)
        card4 = make_card("Closure", form4)

        # 2x2 grid layout
        card1.move_to(LEFT  * 3.10 + UP   * 1.05)
        card2.move_to(RIGHT * 3.10 + UP   * 1.05)
        card3.move_to(LEFT  * 3.10 + DOWN * 1.30)
        card4.move_to(RIGHT * 3.10 + DOWN * 1.30)

        # Beat 3 -- Associativity
        with self.voiceover(
            text=(
                "First, associativity. The order of grouping does not matter: "
                "g-compose-h, then compose-ell, equals g compose h-compose-ell."
            )
        ) as tracker:
            self.play(FadeIn(card1),                              run_time=tracker.duration * 0.35)
            # Left bracket group (gh)l turns BLUE
            self.play(
                form1[0].animate.set_color(BLUE),
                run_time=tracker.duration * 0.25,
            )
            # Right bracket group g(hl) turns ORANGE
            self.play(
                form1[2].animate.set_color(ORANGE),
                run_time=tracker.duration * 0.25,
            )
            self.wait(tracker.duration * 0.15)

        # Beat 4 -- Identity
        with self.voiceover(
            text=(
                "Second, identity. There is a unique element e -- "
                "the do-nothing transformation -- such that composing anything "
                "with e leaves it unchanged."
            )
        ) as tracker:
            self.play(FadeIn(card2),                              run_time=tracker.duration * 0.40)
            # Indicate only the "e" symbol, not the whole formula
            self.play(
                Indicate(form2[1], color=GREEN, scale_factor=1.40),
                run_time=tracker.duration * 0.35,
            )
            self.wait(tracker.duration * 0.25)

        # Beat 5 -- Inverse
        with self.voiceover(
            text=(
                "Third, inverses. Every element g has a unique reverse: g-inverse. "
                "Applying g then g-inverse -- or the reverse order -- "
                "brings you back to the identity element e."
            )
        ) as tracker:
            self.play(FadeIn(card3),                              run_time=tracker.duration * 0.35)
            # Step 1: highlight g^{-1} in RED
            self.play(
                Indicate(form3[1], color=RED, scale_factor=1.40),
                run_time=tracker.duration * 0.25,
            )
            # Step 2: highlight e in GREEN (the cancellation result)
            self.play(
                Indicate(form3[3], color=GREEN, scale_factor=1.40),
                run_time=tracker.duration * 0.25,
            )
            self.wait(tracker.duration * 0.15)

        # Beat 6 -- Closure + explicit circle diagram
        c_center = card4.get_bottom() + DOWN * 0.85

        cl_circle = Circle(radius=0.55, color=BLUE, stroke_width=2.0)
        cl_circle.move_to(c_center)

        dot_g  = Dot(radius=0.065, color=ORANGE).move_to(c_center + LEFT  * 0.25 + UP * 0.17)
        dot_h  = Dot(radius=0.065, color=ORANGE).move_to(c_center + RIGHT * 0.25 + UP * 0.17)
        dot_gh = Dot(radius=0.075, color=GREEN ).move_to(c_center + DOWN  * 0.22)

        lbl_g  = Text("g",  font_size=13, color=ORANGE).next_to(dot_g,  UP,   buff=0.05)
        lbl_h  = Text("h",  font_size=13, color=ORANGE).next_to(dot_h,  UP,   buff=0.05)
        lbl_gh = Text("gh", font_size=13, color=GREEN ).next_to(dot_gh, DOWN, buff=0.04)
        lbl_G  = Text("G",  font_size=14, color=BLUE  ).next_to(cl_circle, UR, buff=0.05)

        mid_gh = (dot_g.get_center() + dot_h.get_center()) / 2
        arr_to_gh = Arrow(
            start=mid_gh, end=dot_gh.get_center(),
            color=GREEN, stroke_width=1.8, buff=0.07,
            max_tip_length_to_length_ratio=0.20,
        )

        closure_diagram = VGroup(
            cl_circle, dot_g, dot_h, dot_gh,
            lbl_g, lbl_h, lbl_gh, lbl_G, arr_to_gh,
        )

        with self.voiceover(
            text=(
                "Fourth and finally, closure. Composing any two group elements "
                "gives you an element that is still inside the group. "
                "You cannot escape."
            )
        ) as tracker:
            self.play(FadeIn(card4),                              run_time=tracker.duration * 0.28)
            self.play(Create(cl_circle), FadeIn(lbl_G),          run_time=tracker.duration * 0.18)
            self.play(
                FadeIn(dot_g), FadeIn(dot_h), FadeIn(lbl_g), FadeIn(lbl_h),
                run_time=tracker.duration * 0.18,
            )
            self.play(
                Create(arr_to_gh), FadeIn(dot_gh), FadeIn(lbl_gh),
                run_time=tracker.duration * 0.22,
            )
            self.wait(tracker.duration * 0.14)

        self.play(
            FadeOut(card1), FadeOut(card2), FadeOut(card3),
            FadeOut(card4), FadeOut(closure_diagram),
            run_time=0.50,
        )
        self.wait(0.20)

        # Section sub-header + column dividers
        ex_header = Text("Examples", font_size=30, color=WHITE)
        ex_header.next_to(title, DOWN, buff=0.35)

        div_l = DashedLine(
            start=ex_header.get_bottom() + DOWN * 0.15 + LEFT  * 1.70,
            end=DOWN * 3.50 + LEFT  * 1.70,
            color=GREY_B, stroke_width=1.0, dash_length=0.10,
        )
        div_r = DashedLine(
            start=ex_header.get_bottom() + DOWN * 0.15 + RIGHT * 1.70,
            end=DOWN * 3.50 + RIGHT * 1.70,
            color=GREY_B, stroke_width=1.0, dash_length=0.10,
        )
        self.play(FadeIn(ex_header), Create(div_l), Create(div_r), run_time=0.45)

        # Column x-centers
        CX_L = -3.60
        CX_M =  0.00
        CX_R =  3.60
        # y of content top (just below ex_header bottom)
        content_top_y = ex_header.get_bottom()[1] - 0.45

        # Beat 7 -- Permutations  [FIX M2: explicit diagram spec]
        col1_hdr = VGroup(
            Text("Permutations", font_size=19, color=ORANGE),
            MathTex(r"S_n", font_size=18, color=ORANGE),
        ).arrange(DOWN, buff=0.06)
        col1_hdr.move_to(np.array([CX_L, content_top_y, 0]))

        row_top_y = col1_hdr.get_bottom()[1] - 0.40
        row_bot_y = row_top_y - 1.10  # Tăng khoảng cách giữa 2 hàng để mũi tên không đè
        xs = [CX_L - 0.55, CX_L, CX_L + 0.55]  # Tăng khoảng cách giữa các điểm

        top_dots = VGroup(*[
            Dot(radius=0.068, color=WHITE).move_to(np.array([x, row_top_y, 0]))
            for x in xs
        ])
        top_lbls = VGroup(*[
            Text(str(i + 1), font_size=12, color=WHITE).next_to(top_dots[i], UP, buff=0.08)
            for i in range(3)
        ])

        # Permutation sigma: 1->2, 2->3, 3->1
        bot_dots = VGroup(*[
            Dot(radius=0.068, color=YELLOW).move_to(np.array([x, row_bot_y, 0]))
            for x in xs
        ])
        perm_out = ["2", "3", "1"]
        bot_lbls = VGroup(*[
            Text(perm_out[i], font_size=12, color=YELLOW).next_to(bot_dots[i], DOWN, buff=0.08)
            for i in range(3)
        ])

        # Curved arrows: 0->1 and 1->2 curve right; 2->0 bows out left
        # Thay đổi góc cong để không che chữ số
        perm_pairs  = [(0, 1), (1, 2), (2, 0)]
        perm_angles = [PI / 4, PI / 4, -PI / 2]
        perm_arrows = VGroup(*[
            CurvedArrow(
                top_dots[s].get_bottom() + DOWN * 0.05,
                bot_dots[d].get_top() + UP * 0.05,
                color=ORANGE, stroke_width=1.3,
                angle=perm_angles[i],
            )
            for i, (s, d) in enumerate(perm_pairs)
        ])

        sigma_note = Text(
            "sigma: 1->2, 2->3, 3->1",
            font_size=10, color=GREY_B,
        ).next_to(bot_lbls, DOWN, buff=0.30)

        with self.voiceover(
            text=(
                "Permutations of n elements form a group. "
                "Composing two permutations gives another permutation, "
                "and every permutation is reversible."
            )
        ) as tracker:
            self.play(FadeIn(col1_hdr),                           run_time=tracker.duration * 0.18)
            self.play(FadeIn(top_dots), FadeIn(top_lbls),         run_time=tracker.duration * 0.18)
            self.play(*[Create(a) for a in perm_arrows],          run_time=tracker.duration * 0.30)
            self.play(FadeIn(bot_dots), FadeIn(bot_lbls),         run_time=tracker.duration * 0.18)
            self.play(FadeIn(sigma_note),                         run_time=tracker.duration * 0.06)
            self.wait(tracker.duration * 0.10)

        col2_hdr = VGroup(
            Text("Rotations", font_size=19, color=ORANGE),
            MathTex(r"SO(3)", font_size=18, color=ORANGE),
        ).arrange(DOWN, buff=0.06)
        col2_hdr.move_to(np.array([CX_M, content_top_y, 0]))

        rot_circ = Circle(radius=0.50, color=WHITE, stroke_width=1.4)
        rot_circ.next_to(col2_hdr, DOWN, buff=0.28)
        circ_c = rot_circ.get_center()

        ref_dot = Dot(radius=0.058, color=WHITE).move_to(circ_c + RIGHT * 0.50)
        ref_lbl = Text("P", font_size=11, color=WHITE).next_to(ref_dot, RIGHT, buff=0.05)

        # Arc R1: 0 -> PI/2 (counterclockwise, BLUE)
        arc1 = Arc(
            radius=0.50, start_angle=0, angle=PI / 2,
            color=BLUE, stroke_width=2.2, arc_center=circ_c,
        )
        lbl_r1 = Text("R1", font_size=11, color=BLUE).move_to(
            circ_c + 0.66 * np.array([np.cos(PI / 4), np.sin(PI / 4), 0])
        )

        # Arc R2: PI/2 -> PI (counterclockwise, GREEN)
        arc2 = Arc(
            radius=0.50, start_angle=PI / 2, angle=PI / 2,
            color=GREEN, stroke_width=2.2, arc_center=circ_c,
        )
        lbl_r2 = Text("R2", font_size=11, color=GREEN).move_to(
            circ_c + 0.66 * np.array([np.cos(3 * PI / 4), np.sin(3 * PI / 4), 0])
        )

        rot_note = Text(
            "R1 o R2 = R3  (still a rotation)",
            font_size=10, color=GREY_B,
        ).next_to(rot_circ, DOWN, buff=0.12)

        with self.voiceover(
            text=(
                "Rotations in three-dimensional space -- the group SO of three -- "
                "also form a group. Rotating twice is still a rotation, "
                "and every rotation has an inverse rotation."
            )
        ) as tracker:
            self.play(FadeIn(col2_hdr),                            run_time=tracker.duration * 0.18)
            self.play(
                Create(rot_circ), FadeIn(ref_dot), FadeIn(ref_lbl),
                run_time=tracker.duration * 0.20,
            )
            self.play(Create(arc1), FadeIn(lbl_r1),               run_time=tracker.duration * 0.22)
            self.play(Create(arc2), FadeIn(lbl_r2),               run_time=tracker.duration * 0.22)
            self.play(FadeIn(rot_note),                            run_time=tracker.duration * 0.08)
            self.wait(tracker.duration * 0.10)

        # Beat 9 -- Translations
        col3_hdr = Text("Translations", font_size=19, color=ORANGE)
        col3_hdr.move_to(np.array([CX_R, content_top_y, 0]))

        # Anchor diagram below col3_hdr; arrows form a V-shape pointing right
        t_base_y = col3_hdr.get_bottom()[1] - 0.48
        t_start  = np.array([CX_R - 0.68, t_base_y,        0])
        t_v_end  = np.array([CX_R + 0.04, t_base_y + 0.30, 0])   # after shift v
        t_w_end  = np.array([CX_R + 0.62, t_base_y + 0.15, 0])   # after shift w

        t_start_dot = Dot(radius=0.058, color=WHITE).move_to(t_start)
        t_start_lbl = Text("O", font_size=10, color=GREY_B).next_to(t_start_dot, DOWN, buff=0.05)

        arr_v = Arrow(
            start=t_start, end=t_v_end,
            color=BLUE, stroke_width=1.8, buff=0,
            max_tip_length_to_length_ratio=0.18,
        )
        lbl_v = Text("v", font_size=12, color=BLUE).move_to(
            (t_start + t_v_end) / 2 + UP * 0.16
        )

        t_mid_dot = Dot(radius=0.048, color=BLUE).move_to(t_v_end)

        arr_w = Arrow(
            start=t_v_end, end=t_w_end,
            color=GREEN, stroke_width=1.8, buff=0,
            max_tip_length_to_length_ratio=0.18,
        )
        lbl_w = Text("w", font_size=12, color=GREEN).move_to(
            (t_v_end + t_w_end) / 2 + UP * 0.16
        )

        t_end_dot = Dot(radius=0.068, color=TEAL).move_to(t_w_end)

        dashed_vw = DashedLine(
            start=t_start, end=t_w_end,
            color=TEAL, stroke_width=1.6, dash_length=0.09,
        )
        lbl_vw = Text("v+w", font_size=12, color=TEAL).move_to(
            (t_start + t_w_end) / 2 + DOWN * 0.22
        )

        with self.voiceover(
            text=(
                "And translations in space also form a group. "
                "Shift by v, then by w, and you arrive at v plus w. "
                "The inverse is simply the negative shift."
            )
        ) as tracker:
            self.play(FadeIn(col3_hdr),                           run_time=tracker.duration * 0.12)
            self.play(FadeIn(t_start_dot), FadeIn(t_start_lbl),   run_time=tracker.duration * 0.10)
            self.play(GrowArrow(arr_v), FadeIn(lbl_v),            run_time=tracker.duration * 0.20)
            self.play(
                FadeIn(t_mid_dot), GrowArrow(arr_w), FadeIn(lbl_w),
                run_time=tracker.duration * 0.22,
            )
            self.play(
                FadeIn(t_end_dot), Create(dashed_vw), FadeIn(lbl_vw),
                run_time=tracker.duration * 0.24,
            )
            self.wait(tracker.duration * 0.12)

        # Beat 10 -- Closing synthesis
        with self.voiceover(
            text=(
                "So whenever we talk about symmetries -- in images, graphs, molecules, "
                "or point clouds -- the underlying structure is always a group. "
                "This is why group theory is the language of geometric machine learning."
            )
        ) as tracker:
            self.play(
                Indicate(col1_hdr, color=YELLOW, scale_factor=1.10),
                Indicate(col2_hdr, color=YELLOW, scale_factor=1.10),
                Indicate(col3_hdr, color=YELLOW, scale_factor=1.10),
                run_time=tracker.duration * 0.28,
            )
            closing = Text(
                "Groups = the language of symmetry in ML",
                font_size=23, color=TEAL, weight=BOLD,
            ).to_edge(DOWN, buff=0.38)
            self.play(Write(closing),                             run_time=tracker.duration * 0.40)
            self.wait(tracker.duration * 0.32)

        # --- End Scene ---
        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.6)
        self.wait(0.3)


In [ ]:
%manim Scene09_Roadmap

In [ ]:
%manim Scene10_Groups

# SLIDE 11 TỚI SLIDE 13


In [ ]:
# Scene11_GroupActions
class Scene11_GroupActions(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        # --- Title (persists throughout) ---
        title = Text(
            "Group Actions and Representations",
            font_size=36, color=WHITE,
        ).to_edge(UP, buff=0.45)
        self.play(Write(title), run_time=0.70)

        # Subtitle "(informal)"
        subtitle = Text(
            "(informal)",
            font_size=20, color=GREY_B, slant=ITALIC,
        ).next_to(title, DOWN, buff=0.10)

        # Beat 1 -- Intro + definition of group action
        action_def = MathTex(
            r"g \cdot x \quad \text{for } g \in G,\; x \in X",
            font_size=34, color=BLUE,
        )
        action_def.next_to(subtitle, DOWN, buff=0.55)

        with self.voiceover(
            text=(
                "Group actions and representations -- informally. "
                "We build intuition first; the formal treatment comes later. "
                "A group action is a way for G to act on a set X: "
                "for each g and x, we get g-dot-x back in X."
            )
        ) as tracker:
            self.play(FadeIn(subtitle), run_time=tracker.duration * 0.18)
            self.play(Write(action_def), run_time=tracker.duration * 0.50)
            self.wait(tracker.duration * 0.32)

        # Beat 2 -- Permutation example: Sn acts on {1,...,n}
        # Section header
        action_hdr = Text("Group Action", font_size=21, color=BLUE, weight=BOLD)
        action_hdr.next_to(action_def, DOWN, buff=0.50)

        # Example row 1: Sn
        sn_label = Text("Permutations S_n on {1,...,n}:", font_size=19, color=GREY_B)
        sn_formula = MathTex(r"\sigma \cdot i = \sigma(i)", font_size=28, color=ORANGE)
        sn_row = VGroup(sn_label, sn_formula).arrange(RIGHT, buff=0.30)
        sn_row.next_to(action_hdr, DOWN, buff=0.28)

        # Mini permutation diagram: 3 top dots -> 3 bottom dots
        # Compact, LEFT-shifted to avoid overlap with GL(n) that follows
        diag_top_y  =  sn_row.get_bottom()[1] - 0.45
        diag_bot_y  =  diag_top_y - 0.60
        diag_xs     = [-2.60, -1.95, -1.30]

        sn_top = VGroup(*[
            Dot(radius=0.062, color=WHITE).move_to(np.array([x, diag_top_y, 0]))
            for x in diag_xs
        ])
        sn_top_lbl = VGroup(*[
            Text(str(i + 1), font_size=12, color=WHITE).next_to(sn_top[i], UP, buff=0.04)
            for i in range(3)
        ])

        # sigma: 1->2, 2->3, 3->1
        sn_bot = VGroup(*[
            Dot(radius=0.062, color=ORANGE).move_to(np.array([x, diag_bot_y, 0]))
            for x in diag_xs
        ])
        sn_perm_out = ["2", "3", "1"]
        sn_bot_lbl = VGroup(*[
            Text(sn_perm_out[i], font_size=12, color=ORANGE).next_to(sn_bot[i], DOWN, buff=0.04)
            for i in range(3)
        ])
        sn_arrows = VGroup(*[
            CurvedArrow(
                sn_top[s].get_bottom(),
                sn_bot[d].get_top(),
                color=ORANGE, stroke_width=1.2,
                angle=[PI / 5, PI / 5, -PI / 1.7][i],
            )
            for i, (s, d) in enumerate([(0, 1), (1, 2), (2, 0)])
        ])
        sn_diagram = VGroup(sn_top, sn_top_lbl, sn_bot, sn_bot_lbl, sn_arrows)

        with self.voiceover(
            text=(
                "For example, the permutation group Sn acts on indices one through n. "
                "A permutation sigma relabels each index: "
                "it maps i to sigma of i."
            )
        ) as tracker:
            self.play(FadeIn(action_hdr), run_time=tracker.duration * 0.15)
            self.play(FadeIn(sn_row),     run_time=tracker.duration * 0.25)
            self.play(
                FadeIn(sn_top), FadeIn(sn_top_lbl),
                run_time=tracker.duration * 0.20,
            )
            self.play(*[Create(a) for a in sn_arrows], run_time=tracker.duration * 0.25)
            self.play(FadeIn(sn_bot), FadeIn(sn_bot_lbl), run_time=tracker.duration * 0.10)
            self.wait(tracker.duration * 0.05)

        # Beat 3 -- GL(n) example with skeleton matrix
        gln_label   = Text("GL(n) on R^n:", font_size=19, color=GREY_B)
        gln_formula = MathTex(r"A \cdot x = Ax", font_size=28, color=ORANGE)
        gln_row = VGroup(gln_label, gln_formula).arrange(RIGHT, buff=0.30)
        # Di chuyển gln_row sang phải để ngang hàng với top của sn_diagram và không đè lên hình
        gln_row.move_to(np.array([2.2, diag_top_y, 0]))

        # Skeleton 3x3 matrix (RIGHT side, compact)
        CELL = 0.45  # Tăng kích thước ô ma trận
        MAT_X = 1.40  # Căn chỉnh khối ma trận nằm ngay dưới gln_row
        MAT_Y = gln_row.get_bottom()[1] - 0.80  # center y

        # Build matrix cells as small squares with entry text
        matrix_cells = VGroup()
        entry_labels = VGroup()
        for r in range(3):
            for c in range(3):
                cx = MAT_X + (c - 1) * CELL
                cy = MAT_Y + (1 - r) * CELL
                sq = Square(
                    side_length=CELL * 0.90,
                    color=GREY_B, stroke_width=0.8,
                ).move_to(np.array([cx, cy, 0]))
                lbl = Text(
                    f"a{r + 1}{c + 1}",
                    font_size=10, color=GREY_B,
                ).move_to(np.array([cx, cy, 0]))
                matrix_cells.add(sq)
                entry_labels.add(lbl)

        # Column vector x (right of matrix)
        VEC_X = MAT_X + CELL * 2.0
        vec_entries = VGroup()
        vec_labels  = VGroup()
        for r in range(3):
            cy = MAT_Y + (1 - r) * CELL
            sq = Square(
                side_length=CELL * 0.90,
                color=GREY_B, stroke_width=0.8,
            ).move_to(np.array([VEC_X, cy, 0]))
            lbl = Text(
                f"x{r + 1}",
                font_size=10, color=WHITE,
            ).move_to(np.array([VEC_X, cy, 0]))
            vec_entries.add(sq)
            vec_labels.add(lbl)

        # Row-highlight rectangles (one per row)
        def row_rect(row_idx, clr):
            x_left  = MAT_X - CELL * 1.55
            x_right = MAT_X + CELL * 1.55
            cy = MAT_Y + (1 - row_idx) * CELL
            return Rectangle(
                width=x_right - x_left, height=CELL * 0.88,
                color=clr, stroke_width=2.0,
                fill_color=clr, fill_opacity=0.10,
            ).move_to(np.array([MAT_X, cy, 0]))

        highlight_row0 = row_rect(0, BLUE)
        highlight_row1 = row_rect(1, ORANGE)

        OUT_X = VEC_X + CELL * 1.30
        out_lbl_0 = MathTex(r"y_1", font_size=14, color=BLUE).move_to(
            np.array([OUT_X, MAT_Y + CELL, 0])
        )
        out_lbl_1 = MathTex(r"y_2", font_size=14, color=ORANGE).move_to(
            np.array([OUT_X, MAT_Y, 0])
        )
        # Two separate equals signs -- each aligned to its own output row
        eq_sign_0 = MathTex(r"=", font_size=14, color=GREY_B).move_to(
            np.array([VEC_X + CELL * 0.72, MAT_Y + CELL, 0])
        )
        eq_sign_1 = MathTex(r"=", font_size=14, color=GREY_B).move_to(
            np.array([VEC_X + CELL * 0.72, MAT_Y, 0])
        )

        matrix_skeleton = VGroup(matrix_cells, entry_labels, vec_entries, vec_labels)

        with self.voiceover(
            text=(
                "Another example: invertible matrices -- the group GL of n -- "
                "act on vectors in n-dimensional space by matrix multiplication: "
                "A-dot-x equals Ax. "
                "Each row of A dot-products with x to produce one output coordinate."
            )
        ) as tracker:
            self.play(FadeIn(gln_row),           run_time=tracker.duration * 0.20)
            self.play(FadeIn(matrix_skeleton),   run_time=tracker.duration * 0.20)
            # Highlight row 0 -> show y_1 = ...
            self.play(Create(highlight_row0),    run_time=tracker.duration * 0.18)
            self.play(
                FadeIn(out_lbl_0), FadeIn(eq_sign_0),
                run_time=tracker.duration * 0.10,
            )
            # [FIX #2] FadeOut row0 highlight instead of set_opacity (unreliable API)
            self.play(
                FadeOut(highlight_row0),
                Create(highlight_row1),
                run_time=tracker.duration * 0.18,
            )
            self.play(
                FadeIn(out_lbl_1), FadeIn(eq_sign_1),
                run_time=tracker.duration * 0.08,
            )
            self.wait(tracker.duration * 0.06)

        # Beat 4 -- Bridge from actions to representations
        # FadeOut action example content; keep title + subtitle + action_def
        self.play(
            FadeOut(action_hdr),
            FadeOut(sn_row), FadeOut(sn_diagram),
            FadeOut(gln_row), FadeOut(matrix_skeleton),
            FadeOut(highlight_row1),
            FadeOut(out_lbl_0), FadeOut(out_lbl_1),
            FadeOut(eq_sign_0), FadeOut(eq_sign_1),
            run_time=0.45,
        )

        repr_hdr = Text("Group Representation", font_size=21, color=GREEN, weight=BOLD)
        repr_hdr.next_to(action_def, DOWN, buff=0.50)

        bridge_text = Text(
            "But to compute with actions -- train networks -- we need explicit matrices.",
            font_size=18, color=GREY_B,
        )
        bridge_text.next_to(repr_hdr, DOWN, buff=0.22)

        with self.voiceover(
            text=(
                "So far, actions tell us how G transforms elements of X. "
                "But to compute with these actions -- to write algorithms, "
                "train neural networks -- we need them as explicit matrix operations. "
                "That is what group representations provide."
            )
        ) as tracker:
            self.play(FadeIn(repr_hdr),       run_time=tracker.duration * 0.20)
            self.play(FadeIn(bridge_text),    run_time=tracker.duration * 0.30)
            self.wait(tracker.duration * 0.32)

        # Beat 5a -- Permutation representation (sigma . x = P_sigma x)
        self.play(FadeOut(bridge_text), run_time=0.30)

        repr_row1_lbl = Text("Permutations of vectors:", font_size=18, color=GREY_B)
        repr_row1_fx  = MathTex(r"\sigma \cdot x = P_\sigma x", font_size=30, color=GREEN)
        repr_row1 = VGroup(repr_row1_lbl, repr_row1_fx).arrange(RIGHT, buff=0.28)
        repr_row1.next_to(repr_hdr, DOWN, buff=0.38)

        # Skeleton P_sigma: 3x3 permutation matrix (0s and 1 per row)
        # sigma: 1->2, 2->3, 3->1  => P[0][1]=1, P[1][2]=1, P[2][0]=1
        PMAT_X = 3.20
        PMAT_Y = repr_row1.get_bottom()[1] - 0.58
        PCELL  = 0.32
        one_positions = [(0, 1), (1, 2), (2, 0)]  # (row, col) of the 1-entries

        pmat_cells = VGroup()
        pmat_labels = VGroup()
        for r in range(3):
            for c in range(3):
                cx = PMAT_X + (c - 1) * PCELL
                cy = PMAT_Y + (1 - r) * PCELL
                val = "1" if (r, c) in one_positions else "0"
                clr = YELLOW if (r, c) in one_positions else GREY_B
                sq = Square(
                    side_length=PCELL * 0.88,
                    color=GREY_B, stroke_width=0.7,
                ).move_to(np.array([cx, cy, 0]))
                lbl = Text(val, font_size=11, color=clr).move_to(
                    np.array([cx, cy, 0])
                )
                pmat_cells.add(sq)
                pmat_labels.add(lbl)

        pmat_title = MathTex(r"P_\sigma", font_size=16, color=YELLOW).next_to(
            VGroup(pmat_cells), UP, buff=0.08
        )
        pmat_group = VGroup(pmat_cells, pmat_labels, pmat_title)

        with self.voiceover(
            text=(
                "A group representation maps each group element to a matrix -- "
                "a linear stand-in for the action. "
                "For permutations of vectors, sigma-dot-x equals P-sigma times x, "
                "where P-sigma is a permutation matrix encoding sigma."
            )
        ) as tracker:
            self.play(FadeIn(repr_row1),       run_time=tracker.duration * 0.32)
            self.play(FadeIn(pmat_group),      run_time=tracker.duration * 0.30)
            # Indicate the 1-entries (YELLOW) to show structure
            one_lbl_indices = [r * 3 + c for (r, c) in one_positions]
            self.play(*[
                Indicate(pmat_labels[i], color=YELLOW, scale_factor=1.35)
                for i in one_lbl_indices
            ], run_time=tracker.duration * 0.25)
            self.wait(tracker.duration * 0.13)

        # Beat 5b -- Rotation representation + homomorphism requirement
        repr_row2_lbl = Text("Rotations:", font_size=18, color=GREY_B)
        repr_row2_fx  = MathTex(
            r"g \cdot x = Rx = \rho(g)x",
            font_size=30, color=GREEN,
        )
        repr_row2 = VGroup(repr_row2_lbl, repr_row2_fx).arrange(RIGHT, buff=0.28)
        repr_row2.next_to(repr_row1, DOWN, buff=0.30)

        # Requirement box
        req_formula = MathTex(
            r"\rho(gh) = \rho(g)\rho(h)",
            font_size=30, color=YELLOW,
        )
        req_box = SurroundingRectangle(
            req_formula, color=YELLOW, stroke_width=1.8,
            corner_radius=0.12, buff=0.28,
        )
        req_group = VGroup(req_box, req_formula)
        req_group.next_to(repr_row2, DOWN, buff=0.35)

        with self.voiceover(
            text=(
                "For rotations, g-dot-x equals rho of g times x. "
                "The map rho assigns a matrix to each group element. "
                "The crucial requirement: rho must respect composition -- "
                "rho of g-h must equal rho of g times rho of h. "
                "This is the homomorphism property."
            )
        ) as tracker:
            self.play(FadeIn(repr_row2),          run_time=tracker.duration * 0.25)
            self.play(
                Create(req_box), Write(req_formula),
                run_time=tracker.duration * 0.30,
            )
            self.play(
                Indicate(req_group, color=YELLOW, scale_factor=1.06),
                run_time=tracker.duration * 0.25,
            )
            self.wait(tracker.duration * 0.20)

        # Beat 6 -- Orbit: circular diagram with curved arrows
        # FadeOut representation content; keep title + subtitle + action_def + div + repr_hdr
        self.play(
            FadeOut(repr_row1), FadeOut(pmat_group),
            FadeOut(repr_row2), FadeOut(req_group),
            run_time=0.40,
        )

        orbit_lbl = Text("Orbit", font_size=21, color=TEAL, weight=BOLD)
        orbit_lbl.next_to(repr_hdr, DOWN, buff=0.35)

        orbit_formula = MathTex(
            r"\text{Orb}_G(x) = \{g \cdot x \mid g \in G\}",
            font_size=26, color=TEAL,
        )
        orbit_formula.next_to(orbit_lbl, DOWN, buff=0.22)

        # Circular orbit diagram
        # Central dot x + 6 orbit dots on a ring
        ORBIT_CENTER = np.array([1.60, -1.55, 0])
        ORBIT_R = 0.88

        center_dot = Dot(radius=0.10, color=WHITE).move_to(ORBIT_CENTER)
        center_lbl = Text("x", font_size=15, color=WHITE).next_to(
            center_dot, UP, buff=0.08
        )

        N_ORBIT = 6
        orbit_dots = VGroup()
        orbit_dot_lbls = VGroup()
        orbit_arrows = VGroup()
        for k in range(N_ORBIT):
            angle = 2 * PI * k / N_ORBIT + PI / N_ORBIT  # offset so first dot not at right
            pos = ORBIT_CENTER + ORBIT_R * np.array([np.cos(angle), np.sin(angle), 0])
            d = Dot(radius=0.072, color=TEAL).move_to(pos)
            # label outside the ring
            label_pos = ORBIT_CENTER + (ORBIT_R + 0.28) * np.array(
                [np.cos(angle), np.sin(angle), 0]
            )
            lbl = Text(f"g{k + 1}x", font_size=11, color=TEAL).move_to(label_pos)
            # CurvedArrow from center_dot to orbit_dot (fanning outward)
            arr = Arrow(
                start=center_dot.get_center(),
                end=d.get_center(),
                color=TEAL, stroke_width=1.2, buff=0.12,
                max_tip_length_to_length_ratio=0.15,
            )
            orbit_dots.add(d)
            orbit_dot_lbls.add(lbl)
            orbit_arrows.add(arr)

        # Thin circle outline showing the orbit "ring"
        orbit_ring = Circle(
            radius=ORBIT_R, color=TEAL,
            stroke_width=0.8, stroke_opacity=0.40,
        ).move_to(ORBIT_CENTER)

        orbit_diagram = VGroup(
            orbit_ring, center_dot, center_lbl,
            orbit_dots, orbit_dot_lbls, orbit_arrows,
        )

        with self.voiceover(
            text=(
                "We now add one more concept: the orbit. "
                "The orbit of x under G is every point reachable from x "
                "by some group element. "
                "Formally, Orb G of x is the set of all g-dot-x. "
                "Think of rotating a vector -- every rotation traces a circle."
            )
        ) as tracker:
            self.play(FadeIn(orbit_lbl),          run_time=tracker.duration * 0.12)
            self.play(Write(orbit_formula),       run_time=tracker.duration * 0.22)
            self.play(
                Create(orbit_ring),
                FadeIn(center_dot), FadeIn(center_lbl),
                run_time=tracker.duration * 0.16,
            )
            # Arrows fan out one by one in pairs for visual effect
            self.play(
                *[GrowArrow(arr) for arr in orbit_arrows],
                run_time=tracker.duration * 0.28,
            )
            self.play(
                FadeIn(orbit_dots), FadeIn(orbit_dot_lbls),
                run_time=tracker.duration * 0.14,
            )
            self.play(
                Indicate(orbit_dots, color=YELLOW, scale_factor=1.18),
                run_time=tracker.duration * 0.08,
            )

        # --- End Scene ---
        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.6)
        self.wait(0.3)


# Scene13_WrapUp
class Scene13_WrapUp(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        # --- Title ---
        title = Text(
            "How can we get equivariant models?",
            font_size=32, color=WHITE,
        ).to_edge(UP, buff=0.45)
        self.play(Write(title), run_time=0.70)

        # Beat 1 -- Micro-bridge + central question + recap formulas
        recap_inv_lbl  = Text("Inv.:",   font_size=14, color=BLUE)
        recap_inv_fx   = MathTex(r"f(g\cdot x)=f(x)", font_size=16, color=BLUE)
        recap_inv_row  = VGroup(recap_inv_lbl,  recap_inv_fx ).arrange(RIGHT, buff=0.14)

        recap_eq_lbl   = Text("Equiv.:", font_size=14, color=GREEN)
        recap_eq_fx    = MathTex(r"f(g\cdot x)=g\cdot f(x)", font_size=16, color=GREEN)
        recap_eq_row   = VGroup(recap_eq_lbl, recap_eq_fx).arrange(RIGHT, buff=0.14)

        recap_block = VGroup(recap_inv_row, recap_eq_row).arrange(DOWN, buff=0.14, aligned_edge=LEFT)
        recap_block.next_to(title, DOWN, buff=0.35)
        recap_box = SurroundingRectangle(
            recap_block, color=GREY_B, stroke_width=1.0,
            corner_radius=0.10, buff=0.14,
        )

        with self.voiceover(
            text=(
                "Having established groups, group actions, and representations "
                "as our building blocks, we return to the central question: "
                "how can we get equivariant and invariant models?"
            )
        ) as tracker:
            self.play(
                FadeIn(recap_block), Create(recap_box),
                run_time=tracker.duration * 0.30,
            )
            self.play(
                Indicate(title, color=YELLOW, scale_factor=1.03),
                run_time=tracker.duration * 0.40,
            )
            self.wait(tracker.duration * 0.30)

        # Beat 2 -- Structure FIRST (divider + both headers), then left bullets
        div_x = 0.0
        col_div = DashedLine(
            start=np.array([div_x, 1.20, 0]),
            end=np.array([div_x, -3.30, 0]),
            color=GREY_B, stroke_width=1.0, dash_length=0.10,
        )

        # Column headers
        left_hdr  = Text(
            "Off-the-shelf models",
            font_size=20, color=BLUE, weight=BOLD,
        ).move_to(np.array([-2.85, 0.95, 0]))
        right_hdr = Text(
            "Constructing equivariant models",
            font_size=18, color=GREEN, weight=BOLD,
        ).move_to(np.array([2.85, 0.95, 0]))

        # Left column bullets
        left_b1 = Text("- Data augmentation",      font_size=19, color=WHITE)
        left_b2 = Text("- Canoni(cali)zation",      font_size=19, color=WHITE)
        left_b3 = Text("- Group / frame averaging", font_size=19, color=WHITE)
        left_bullets = VGroup(left_b1, left_b2, left_b3).arrange(
            DOWN, aligned_edge=LEFT, buff=0.24
        )
        left_bullets.next_to(left_hdr, DOWN, buff=0.35)
        left_bullets.align_to(np.array([-4.80, 0, 0]), LEFT)

        with self.voiceover(
            text=(
                "The first family uses off-the-shelf models, "
                "made equivariant through processing. "
                "Three techniques: data augmentation, canonicalization, "
                "and group or frame averaging."
            )
        ) as tracker:
            self.play(
                Create(col_div),
                FadeIn(left_hdr),
                FadeIn(right_hdr),
                run_time=tracker.duration * 0.28,
            )
            self.play(FadeIn(left_b1, shift=RIGHT * 0.15), run_time=tracker.duration * 0.22)
            self.play(FadeIn(left_b2, shift=RIGHT * 0.15), run_time=tracker.duration * 0.22)
            self.play(FadeIn(left_b3, shift=RIGHT * 0.15), run_time=tracker.duration * 0.20)
            self.wait(tracker.duration * 0.08)

        # Beat 3 -- Right column bullets (left stays visible)
        right_b1 = Text("- Linear equivariant layers",         font_size=19, color=WHITE)
        right_b2 = Text("  (param. sharing, group conv.)",     font_size=16, color=GREY_B)
        right_b3 = Text("- Invariant theory",                  font_size=19, color=WHITE)
        right_b4 = Text("- Representation theory",             font_size=19, color=WHITE)
        right_bullets = VGroup(right_b1, right_b2, right_b3, right_b4).arrange(
            DOWN, aligned_edge=LEFT, buff=0.20
        )
        right_bullets.next_to(right_hdr, DOWN, buff=0.35)
        right_bullets.align_to(np.array([0.22, 0, 0]), LEFT)

        with self.voiceover(
            text=(
                "The second family constructs equivariant models by design. "
                "Approaches include linear equivariant layers "
                "with parameter sharing or group convolution, "
                "invariant theory, and representation theory."
            )
        ) as tracker:
            self.play(FadeIn(right_b1, shift=LEFT * 0.15), run_time=tracker.duration * 0.20)
            self.play(FadeIn(right_b2, shift=LEFT * 0.15), run_time=tracker.duration * 0.15)
            self.play(FadeIn(right_b3, shift=LEFT * 0.15), run_time=tracker.duration * 0.20)
            self.play(FadeIn(right_b4, shift=LEFT * 0.15), run_time=tracker.duration * 0.20)
            self.wait(tracker.duration * 0.25)

        # Beat 4 -- Visual payoff closing
        closing = Text(
            "You now have the language.",
            font_size=26, color=TEAL, weight=BOLD,
        ).to_edge(DOWN, buff=0.40)

        with self.voiceover(
            text=(
                "Both families build on the mathematical language we just developed. "
                "Our series continues by exploring each approach in depth."
            )
        ) as tracker:
            # Flash recap formulas (callback to Scene09) + write closing simultaneously
            self.play(
                Indicate(recap_block, color=YELLOW, scale_factor=1.08),
                Write(closing),
                run_time=tracker.duration * 0.55,
            )
            self.wait(tracker.duration * 0.45)

        # --- End Scene ---
        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.6)
        self.wait(0.3)


In [ ]:
%manim Scene11_GroupActions

In [ ]:
%manim Scene13_WrapUp

# SLIDE 14 TỚI SLIDE 17

In [ ]:
# Scenes: Scene14_DataAug, Scene15_InvariantRepr,
#         Scene16a_EquivReprNaive, Scene16b_EquivReprRotation
class Scene14_DataAug(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title = Text(
            "Data Augmentation",
            font_size=40, color=WHITE,
        ).to_edge(UP, buff=0.45)
        self.play(Write(title), run_time=0.60)

        # Beat 1 -- Augmentation concept + NN diagram
        # Input sample x (center-left)
        INPUT_X = np.array([-3.90, 0.20, 0])  # [FIX #7] shifted right to prevent label crop
        dot_x     = Dot(radius=0.12, color=WHITE).move_to(INPUT_X)
        lbl_x     = Text("x", font_size=18, color=WHITE).next_to(dot_x, LEFT, buff=0.12)

        # Transformed samples g_k . x  (arranged vertically above/below x)
        offsets   = [UP * 1.10, UP * 0.40, DOWN * 0.35, DOWN * 1.10]
        aug_colors = [ORANGE, ORANGE, ORANGE, ORANGE]
        aug_labels = ["g1.x", "g2.x", "g3.x", "g4.x"]

        aug_dots = VGroup(*[
            Dot(radius=0.088, color=aug_colors[i]).move_to(INPUT_X + offsets[i])
            for i in range(4)
        ])
        aug_lbls = VGroup(*[
            Text(aug_labels[i], font_size=13, color=ORANGE).next_to(
                aug_dots[i], LEFT, buff=0.10
            )
            for i in range(4)
        ])

        # Neural Network box (center-right)
        nn_box = Rectangle(
            width=2.20, height=1.60,
            color=WHITE, stroke_width=1.8,
            fill_color=GREY_D, fill_opacity=0.30,
        ).move_to(np.array([1.60, 0.20, 0]))
        nn_lbl = Text(
            "Neural\nNetwork",
            font_size=18, color=WHITE,
        ).move_to(nn_box.get_center())

        # Arrows from x + aug dots into NN
        NN_LEFT = np.array([nn_box.get_left()[0], 0.20, 0])

        arr_x = Arrow(
            start=dot_x.get_right(),
            end=NN_LEFT,
            color=WHITE, stroke_width=1.5, buff=0.06,
            max_tip_length_to_length_ratio=0.12,
        )
        aug_arrows = VGroup(*[
            Arrow(
                start=aug_dots[i].get_right(),
                end=np.array([nn_box.get_left()[0],
                              aug_dots[i].get_center()[1] * 0.45 + 0.20 * 0.55, 0]),
                color=ORANGE, stroke_width=1.2, buff=0.06,
                max_tip_length_to_length_ratio=0.12,
            )
            for i in range(4)
        ])

        with self.voiceover(
            text=(
                "Data augmentation is the simplest approach to symmetry. "
                "Given a data point x, we add transformed copies -- "
                "g-dot-x for various group elements g -- "
                "directly to the training data, then train as usual."
            )
        ) as tracker:
            self.play(FadeIn(dot_x), FadeIn(lbl_x),     run_time=tracker.duration * 0.15)
            self.play(
                FadeIn(aug_dots), FadeIn(aug_lbls),
                run_time=tracker.duration * 0.20,
            )
            self.play(
                FadeIn(nn_box), FadeIn(nn_lbl),
                run_time=tracker.duration * 0.20,
            )
            self.play(
                GrowArrow(arr_x),
                *[GrowArrow(a) for a in aug_arrows],
                run_time=tracker.duration * 0.35,
            )
            self.wait(tracker.duration * 0.10)

        # Beat 2 -- Pros / Cons
        pros_hdr = Text("Pros:", font_size=19, color=GREEN, weight=BOLD)
        pros_b1  = Text("- simple", font_size=18, color=GREEN)
        pros_col = VGroup(pros_hdr, pros_b1).arrange(DOWN, aligned_edge=LEFT, buff=0.14)
        pros_col.move_to(np.array([-1.20, -1.20, 0]))

        cons_hdr = Text("Cons:", font_size=19, color=RED, weight=BOLD)
        cons_b1  = Text(
            "- does not guarantee invariance everywhere",
            font_size=17, color=RED,
        )
        cons_col = VGroup(cons_hdr, cons_b1).arrange(DOWN, aligned_edge=LEFT, buff=0.14)
        cons_col.next_to(pros_col, RIGHT, buff=1.20, aligned_edge=UP)

        with self.voiceover(
            text=(
                "The advantage is simplicity -- no architectural changes, "
                "any model works. "
                "The limitation: augmentation does not guarantee invariance "
                "everywhere, only on the augmented training distribution."
            )
        ) as tracker:
            self.play(FadeIn(pros_col), run_time=tracker.duration * 0.30)
            self.play(FadeIn(cons_col), run_time=tracker.duration * 0.40)
            self.play(
                Indicate(cons_b1, color=RED, scale_factor=1.05),
                run_time=tracker.duration * 0.20,
            )
            self.wait(tracker.duration * 0.10)

        # Beat 3 -- Real-world context
        practice_note = Text(
            "Used in practice: images, graphs, point clouds, molecules",
            font_size=15, color=GREY_B, slant=ITALIC,
        ).to_edge(DOWN, buff=0.38)

        with self.voiceover(
            text=(
                "Despite this limitation, data augmentation is ubiquitous -- "
                "from image classification to molecular property prediction."
            )
        ) as tracker:
            self.play(FadeIn(practice_note),     run_time=tracker.duration * 0.35)
            self.play(
                Indicate(aug_dots, color=YELLOW, scale_factor=1.06),
                Indicate(nn_box,   color=YELLOW, scale_factor=1.04),
                run_time=tracker.duration * 0.40,
            )
            self.wait(tracker.duration * 0.25)

        # --- End Scene ---
        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.6)
        self.wait(0.3)


# Scene15_InvariantRepr
class Scene15_InvariantRepr(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title = Text(
            "Learning Invariant Representations",
            font_size=34, color=WHITE,
        ).to_edge(UP, buff=0.45)
        self.play(Write(title), run_time=0.65)

        # Beat 1 -- 3-class orbit diagram + formula reveal
        credit = Text(
            "Slide inspired by M. Bronstein",
            font_size=12, color=GREY_B, slant=ITALIC,
        ).to_corner(DR, buff=0.30)

        subtitle = Text(
            "(via contrastive learning)",
            font_size=18, color=GREY_B, slant=ITALIC,
        ).next_to(title, DOWN, buff=0.12)

        # 3 orbit classes -- LEFT side, stacked vertically
        ORBIT_X   = -4.00
        CLASS_CLR  = [BLUE, ORANGE, GREEN]
        CLASS_LBL  = ["orbit A", "orbit B", "orbit C"]
        # Each orbit: 2 dots on a small dashed ellipse
        orbit_ys  = [1.20, 0.00, -1.20]
        ORBIT_R_X = 0.55
        ORBIT_R_Y = 0.28

        orbit_groups = VGroup()
        for ci, (oy, clr, lbl) in enumerate(zip(orbit_ys, CLASS_CLR, CLASS_LBL)):
            center = np.array([ORBIT_X, oy, 0])
            # 2 dots per orbit
            d0 = Dot(radius=0.08, color=clr).move_to(center + LEFT  * 0.35)
            d1 = Dot(radius=0.08, color=clr).move_to(center + RIGHT * 0.35)
            # Dashed ellipse outline
            ellipse = DashedVMobject(
                Ellipse(width=ORBIT_R_X * 2 + 0.20, height=ORBIT_R_Y * 2 + 0.20,
                        color=clr, stroke_width=0.9).move_to(center),
                num_dashes=12,
            )
            label = Text(lbl, font_size=11, color=clr).next_to(
                ellipse, UP, buff=0.06
            )
            orbit_groups.add(VGroup(ellipse, d0, d1, label))

        # Invariance formula (below orbit groups)
        inv_formula = MathTex(
            r"f(g \cdot x) = f(x)",
            font_size=28, color=BLUE,
        )
        inv_formula.move_to(np.array([ORBIT_X, -2.30, 0]))  # [FIX #5] y=-2.30 prevents overlap

        # Encoder arrow
        enc_arrow = Arrow(
            start=np.array([ORBIT_X + 0.80, 0.00, 0]),
            end=np.array([ORBIT_X + 2.20, 0.00, 0]),
            color=GREY_B, stroke_width=2.0, buff=0.05,
            max_tip_length_to_length_ratio=0.14,
        )
        enc_lbl = Text("f", font_size=20, color=GREY_B).next_to(enc_arrow, UP, buff=0.08)

        with self.voiceover(
            text=(
                "Another approach: learn an encoder f that maps all elements "
                "of an orbit to the same latent point. "
                "Inputs related by a group action -- x and g-dot-x -- "
                "should get the same representation. "
                "Formally, f of g-dot-x equals f of x."
            )
        ) as tracker:
            self.play(FadeIn(credit), FadeIn(subtitle), run_time=tracker.duration * 0.12)
            self.play(
                FadeIn(orbit_groups[0]),
                FadeIn(orbit_groups[1]),
                FadeIn(orbit_groups[2]),
                run_time=tracker.duration * 0.30,
            )
            self.play(Write(inv_formula), run_time=tracker.duration * 0.30)
            self.play(
                FadeIn(enc_arrow), FadeIn(enc_lbl),
                run_time=tracker.duration * 0.15,
            )
            self.wait(tracker.duration * 0.13)

        # Beat 2 -- 3-color latent space
        # Latent space circle (RIGHT side)
        lat_center = np.array([2.60, 0.00, 0])
        lat_circle = Circle(radius=1.35, color=GREY_B, stroke_width=1.2).move_to(lat_center)
        lat_lbl    = Text("Latent Space", font_size=15, color=GREY_B).next_to(
            lat_circle, UP, buff=0.10
        )

        # 3 latent dots -- different positions inside the circle
        lat_dot_offsets = [UP * 0.75, RIGHT * 0.70 + DOWN * 0.30, LEFT * 0.50 + DOWN * 0.55]
        lat_dots = VGroup(*[
            Dot(radius=0.12, color=CLASS_CLR[i]).move_to(lat_center + lat_dot_offsets[i])
            for i in range(3)
        ])
        lat_dot_lbls = VGroup(*[
            Text(["A", "B", "C"][i], font_size=12, color=CLASS_CLR[i]).next_to(
                lat_dots[i], UP, buff=0.06
            )
            for i in range(3)
        ])

        # Arrows: each orbit group -> its latent dot
        # orbit_groups[0] (BLUE) -> lat_dots[0], etc.
        conv_arrows = VGroup(*[
            Arrow(
                start=np.array([ORBIT_X + 0.65, orbit_ys[i], 0]),
                end=lat_dots[i].get_left() + LEFT * 0.05,
                color=CLASS_CLR[i], stroke_width=1.2, buff=0.06,
                max_tip_length_to_length_ratio=0.12,
            )
            for i in range(3)
        ])

        with self.voiceover(
            text=(
                "In contrastive learning, we train f to pull same-orbit elements "
                "together and push different-orbit elements apart. "
                "Each orbit collapses to one point -- "
                "but different orbits stay separated, preserving class structure."
            )
        ) as tracker:
            self.play(
                Create(lat_circle), FadeIn(lat_lbl),
                run_time=tracker.duration * 0.20,
            )
            self.play(
                FadeIn(lat_dots), FadeIn(lat_dot_lbls),
                run_time=tracker.duration * 0.20,
            )
            self.play(
                *[GrowArrow(a) for a in conv_arrows],
                run_time=tracker.duration * 0.35,
            )
            self.play(
                Indicate(lat_dots, color=YELLOW, scale_factor=1.20),
                run_time=tracker.duration * 0.15,
            )
            self.wait(tracker.duration * 0.10)

        # Beat 3 -- Closing note
        closing_note = Text(
            "Invariance gained -- equivariance not preserved",
            font_size=16, color=GREY_B, slant=ITALIC,
        ).to_edge(DOWN, buff=0.38)

        with self.voiceover(
            text=(
                "This is powerful: the network learns symmetry from data, "
                "without any hard-coded equivariance in the architecture. "
                "The price: we get invariance, not transformation-tracking."
            )
        ) as tracker:
            self.play(FadeIn(closing_note), run_time=tracker.duration * 0.30)
            self.play(
                Indicate(inv_formula, color=YELLOW, scale_factor=1.06),
                run_time=tracker.duration * 0.40,
            )
            self.wait(tracker.duration * 0.30)

        # --- End Scene ---
        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.6)
        self.wait(0.3)


# Scene16a_EquivReprNaive
class Scene16a_EquivReprNaive(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title = Text(
            "Equivariant Representation Learning",
            font_size=34, color=WHITE,
        ).to_edge(UP, buff=0.45)
        self.play(Write(title), run_time=0.65)

        # Beat 1 -- Goal formula + parameterization question
        goal_want = Text("Want:", font_size=18, color=GREY_B)
        goal_fx   = MathTex(
            r"f(a(x)) = T_a f(x)",
            font_size=34, color=BLUE,
        )
        goal_row = VGroup(goal_want, goal_fx).arrange(RIGHT, buff=0.30, aligned_edge=DOWN)
        goal_row.next_to(title, DOWN, buff=0.70)
        goal_box = SurroundingRectangle(
            goal_row, color=BLUE, stroke_width=1.8,
            corner_radius=0.12, buff=0.25,
        )

        param_q = Text(
            "How to parameterize T_a?",
            font_size=18, color=GREY_B, slant=ITALIC,
        ).next_to(goal_box, DOWN, buff=0.30)

        with self.voiceover(
            text=(
                "Now a more ambitious goal: instead of invariance, "
                "we want equivariance with a learnable linear map. "
                "We want f of a of x to equal T-sub-a applied to f of x. "
                "The question is: how do we parameterize T-sub-a?"
            )
        ) as tracker:
            self.play(Write(goal_want), Write(goal_fx), run_time=tracker.duration * 0.38)
            self.play(Create(goal_box),                 run_time=tracker.duration * 0.18)
            self.play(
                Indicate(goal_fx, color=YELLOW, scale_factor=1.05),
                run_time=tracker.duration * 0.18,
            )
            self.play(FadeIn(param_q),                  run_time=tracker.duration * 0.16)
            self.wait(tracker.duration * 0.10)

        # Beat 2 -- Commutative diagram + MLP loss
        self.play(FadeOut(goal_box), FadeOut(goal_want),
                  FadeOut(goal_fx),  FadeOut(param_q),
                  run_time=0.35)

        # 4-corner commutative diagram
        # corners: top-left x, top-right f(x), bot-left a(x), bot-right f(a(x))
        DX, DY = 2.40, 1.20   # half-width, half-height of rectangle
        CX, CY = 0.00, 0.20   # center of diagram

        pos_x   = np.array([CX - DX, CY + DY, 0])
        pos_fx  = np.array([CX + DX, CY + DY, 0])
        pos_ax  = np.array([CX - DX, CY - DY, 0])
        pos_fax = np.array([CX + DX, CY - DY, 0])

        node_r = 0.10
        node_x   = Dot(radius=node_r, color=WHITE).move_to(pos_x)
        node_fx  = Dot(radius=node_r, color=BLUE ).move_to(pos_fx)
        node_ax  = Dot(radius=node_r, color=WHITE).move_to(pos_ax)
        node_fax = Dot(radius=node_r, color=GREEN).move_to(pos_fax)

        lbl_x   = Text("x",      font_size=18, color=WHITE).next_to(node_x,   UL, buff=0.08)
        lbl_fx  = Text("f(x)",   font_size=18, color=BLUE ).next_to(node_fx,  UR, buff=0.08)
        lbl_ax  = Text("a(x)",   font_size=18, color=WHITE).next_to(node_ax,  DL, buff=0.08)
        lbl_fax = Text("f(a(x))",font_size=16, color=GREEN).next_to(node_fax, DR, buff=0.08)

        arr_top    = Arrow(pos_x   + RIGHT * 0.12, pos_fx  + LEFT  * 0.12,
                           color=BLUE,   stroke_width=1.8, buff=0,
                           max_tip_length_to_length_ratio=0.10)
        arr_left   = Arrow(pos_x   + DOWN  * 0.12, pos_ax  + UP    * 0.12,
                           color=ORANGE, stroke_width=1.8, buff=0,
                           max_tip_length_to_length_ratio=0.10)
        arr_right  = Arrow(pos_fx  + DOWN  * 0.12, pos_fax + UP    * 0.12,
                           color=GREEN,  stroke_width=1.8, buff=0,
                           max_tip_length_to_length_ratio=0.10)
        arr_bottom = Arrow(pos_ax  + RIGHT * 0.12, pos_fax + LEFT  * 0.12,
                           color=BLUE,   stroke_width=1.8, buff=0,
                           max_tip_length_to_length_ratio=0.10)

        lbl_top    = Text("f",          font_size=16, color=BLUE  ).move_to(
            (pos_x + pos_fx) / 2 + UP * 0.22)
        lbl_left   = Text("a",          font_size=16, color=ORANGE).move_to(
            (pos_x + pos_ax) / 2 + LEFT * 0.28)
        lbl_right  = Text("MLP(a, .)",  font_size=14, color=GREEN ).move_to(
            (pos_fx + pos_fax) / 2 + RIGHT * 0.52)
        lbl_bottom = Text("f",          font_size=16, color=BLUE  ).move_to(
            (pos_ax + pos_fax) / 2 + DOWN * 0.22)

        diagram = VGroup(
            node_x,  node_fx,  node_ax,  node_fax,
            lbl_x,   lbl_fx,   lbl_ax,   lbl_fax,
            arr_top, arr_left, arr_right, arr_bottom,
            lbl_top, lbl_left, lbl_right, lbl_bottom,
        )

        loss_fx = MathTex(
            r"\|f(a(x)) - \mathrm{MLP}(a,\, f(x))\|",
            font_size=24, color=WHITE,
        )
        loss_row = VGroup(
            Text("Loss:", font_size=18, color=GREY_B),
            loss_fx,
        ).arrange(RIGHT, buff=0.25, aligned_edge=DOWN)
        loss_row.next_to(diagram, DOWN, buff=0.45)

        with self.voiceover(
            text=(
                "One approach: use a neural network. "
                "An MLP takes the transformation a and the representation f of x, "
                "and predicts what T-sub-a applied to f of x should be. "
                "The training loss minimizes the distance between "
                "f of a of x and MLP of a comma f of x."
            )
        ) as tracker:
            # Build diagram progressively
            self.play(
                FadeIn(node_x), FadeIn(node_fx), FadeIn(node_ax), FadeIn(node_fax),
                FadeIn(lbl_x),  FadeIn(lbl_fx),  FadeIn(lbl_ax),  FadeIn(lbl_fax),
                run_time=tracker.duration * 0.22,
            )
            self.play(
                GrowArrow(arr_top), GrowArrow(arr_left),
                GrowArrow(arr_right), GrowArrow(arr_bottom),
                FadeIn(lbl_top), FadeIn(lbl_left),
                FadeIn(lbl_right), FadeIn(lbl_bottom),
                run_time=tracker.duration * 0.32,
            )
            self.play(FadeIn(loss_row),     run_time=tracker.duration * 0.28)
            self.wait(tracker.duration * 0.18)

        # Beat 3 -- Composition inconsistency
        self.play(FadeOut(diagram), FadeOut(loss_row), run_time=0.40)

        comp_lhs  = MathTex(r"T_{a_2 \circ a_1}", font_size=34, color=RED)
        comp_neq  = MathTex(r"\neq",               font_size=34, color=RED)
        comp_rhs  = MathTex(r"T_{a_2} T_{a_1}",   font_size=34, color=RED)
        comp_row  = VGroup(comp_lhs, comp_neq, comp_rhs).arrange(RIGHT, buff=0.30)
        comp_row.move_to(np.array([0.00, 0.20, 0]))
        comp_box_dashed = DashedVMobject(
            SurroundingRectangle(comp_row, color=RED, stroke_width=1.8,
                                 corner_radius=0.10, buff=0.28),
            num_dashes=30,
        )

        comp_note = Text(
            "MLP does not guarantee consistency with group composition",
            font_size=17, color=GREY_B,
        ).next_to(comp_row, DOWN, buff=0.40)

        with self.voiceover(
            text=(
                "However, this approach has a fundamental problem: "
                "the MLP does not guarantee consistency with how transformations compose. "
                "Applying a-sub-1 then a-sub-2 should give the same result "
                "as applying their composition directly -- "
                "but the MLP may not satisfy this."
            )
        ) as tracker:
            self.play(
                Write(comp_lhs), Write(comp_neq), Write(comp_rhs),
                run_time=tracker.duration * 0.35,
            )
            self.play(Create(comp_box_dashed),       run_time=tracker.duration * 0.18)
            self.play(
                Indicate(comp_row, color=RED, scale_factor=1.06),
                run_time=tracker.duration * 0.22,
            )
            self.play(FadeIn(comp_note),              run_time=tracker.duration * 0.15)
            self.wait(tracker.duration * 0.10)

        self.wait(1.5)

        # Beat 4 -- Citations
        with self.voiceover(
            text=(
                "This inconsistency was identified by Dangovski et al. in 2022, "
                "Devillers and Lefort in 2023, and Garrido et al. in 2023."
            )
        ) as tracker:
            cit_text = Text(
                "Dangovski et al. 2022  |  Devillers & Lefort 2023  |  Garrido et al. 2023",
                font_size=13, color=GREY_B, slant=ITALIC,
            ).to_edge(DOWN, buff=0.35)
            self.play(FadeIn(cit_text), run_time=tracker.duration * 0.40)
            self.wait(tracker.duration * 0.60)

        # --- End Scene ---
        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.6)
        self.wait(0.3)


# Scene16b_EquivReprRotation
class Scene16b_EquivReprRotation(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title = Text(
            "Equivariant Repr.: Rotation Fix",
            font_size=34, color=WHITE,
        ).to_edge(UP, buff=0.45)
        self.play(Write(title), run_time=0.65)

        # Beat 1 -- Before/after: MLP -> rotation
        # TransformMatchingTex candidate (morph formula)
        mlp_fx = MathTex(
            r"T_a = \mathrm{MLP}(a, \cdot)",
            font_size=28, color=RED,
        )
        rot_fx = MathTex(
            r"T_a = R_a \;(\text{rotation})",
            font_size=28, color=GREEN,
        )
        arrow_fix = Text("->", font_size=22, color=GREY_B)
        comparison = VGroup(mlp_fx, arrow_fix, rot_fx).arrange(RIGHT, buff=0.40)
        comparison.next_to(title, DOWN, buff=0.70)

        # so mlp_fx has a valid bounding box before Line() reads get_left/get_right
        self.add(mlp_fx)
        strike = Line(
            mlp_fx.get_left() + LEFT * 0.05,
            mlp_fx.get_right() + RIGHT * 0.05,
            color=RED, stroke_width=2.0,
        )

        with self.voiceover(
            text=(
                "There is a cleaner parameterization: "
                "instead of an unconstrained MLP, we restrict T-sub-a "
                "to be a rotation matrix R-sub-a. "
                "Rotations automatically preserve composition, "
                "and they preserve inner products."
            )
        ) as tracker:
            # mlp_fx already added via self.add() above -- animate it in-place
            self.play(
                mlp_fx.animate.set_opacity(1.0),  # visible (was added silently)
                run_time=tracker.duration * 0.10,
            )
            self.play(Create(strike),            run_time=tracker.duration * 0.14)
            self.play(
                FadeIn(arrow_fix), FadeIn(rot_fx),
                run_time=tracker.duration * 0.25,
            )
            self.play(
                Indicate(rot_fx, color=GREEN, scale_factor=1.08),
                run_time=tracker.duration * 0.30,
            )
            self.wait(tracker.duration * 0.21)

        # Beat 2 -- Inner-product loss
        self.play(FadeOut(comparison), FadeOut(strike), run_time=0.35)

        loss_label = Text("Loss:", font_size=18, color=GREY_B)
        loss_note  = Text(
            "(rotation preserves inner products)",
            font_size=15, color=GREY_B, slant=ITALIC,
        )
        loss_formula = MathTex(
            r"\mathbb{E}_{a,x,x'}\!\Bigl["
            r"f(a(x'))^\top f(a(x)) - f(x)^\top f(x')"
            r"\Bigr]^{\!2}",
            font_size=26, color=WHITE,
        )
        loss_block = VGroup(loss_label, loss_formula).arrange(
            DOWN, aligned_edge=LEFT, buff=0.22
        )
        loss_block.move_to(np.array([0.00, 0.25, 0]))
        loss_note.next_to(loss_formula, DOWN, buff=0.18)

        loss_box = SurroundingRectangle(
            VGroup(loss_block, loss_note),
            color=BLUE, stroke_width=1.6,
            corner_radius=0.12, buff=0.28,
        )

        with self.voiceover(
            text=(
                "The training loss changes too. "
                "Instead of Euclidean distance, we use an inner-product loss -- "
                "minimizing the expected discrepancy between inner products "
                "before and after the transformation. "
                "This loss is zero precisely when T-sub-a is a rotation."
            )
        ) as tracker:
            self.play(FadeIn(loss_label),          run_time=tracker.duration * 0.12)
            self.play(Write(loss_formula),         run_time=tracker.duration * 0.42)  # [FIX #4]
            self.play(FadeIn(loss_note),           run_time=tracker.duration * 0.16)
            self.play(Create(loss_box),            run_time=tracker.duration * 0.18)
            self.play(
                Indicate(loss_box, color=YELLOW, scale_factor=1.04),
                run_time=tracker.duration * 0.08,
            )
            self.wait(tracker.duration * 0.04)

        # Beat 3 -- Result: T_a = R_a = rho(a) is a group representation
        self.play(FadeOut(loss_block), FadeOut(loss_note), FadeOut(loss_box),
                  run_time=0.40)

        result_hdr = Text(
            "When loss = 0:",
            font_size=18, color=GREY_B,
        )
        result_fx = MathTex(
            r"T_a = R_a = \rho(a)",
            font_size=38, color=GREEN,
        )
        result_sub = Text(
            "is a group representation",
            font_size=19, color=GREEN,
        )
        result_block = VGroup(result_hdr, result_fx, result_sub).arrange(
            DOWN, buff=0.22, aligned_edge=LEFT
        )
        result_block.move_to(np.array([0.00, 0.30, 0]))
        result_box = SurroundingRectangle(
            result_block, color=GREEN, stroke_width=1.8,
            corner_radius=0.12, buff=0.28,
        )

        input_note = Text(
            "Input transformation need not be a rotation!",
            font_size=16, color=GREY_B, slant=ITALIC,
        ).next_to(result_box, DOWN, buff=0.32)

        with self.voiceover(
            text=(
                "When this loss reaches zero, something remarkable happens. "
                "Each T-sub-a is encoded as a rotation R-sub-a that preserves composition. "
                "In other words, T-sub-a equals R-sub-a equals rho of a -- "
                "it is a group representation. "
                "And importantly: the input transformation a does not need "
                "to be a rotation itself."
            )
        ) as tracker:
            self.play(
                FadeIn(result_hdr), Write(result_fx),
                run_time=tracker.duration * 0.30,
            )
            self.play(
                FadeIn(result_sub), Create(result_box),
                run_time=tracker.duration * 0.28,
            )
            self.play(
                Indicate(result_box, color=YELLOW, scale_factor=1.05),
                run_time=tracker.duration * 0.20,
            )
            self.play(FadeIn(input_note),          run_time=tracker.duration * 0.16)
            self.wait(tracker.duration * 0.06)

        # Beat 4 -- Citations
        with self.voiceover(
            text=(
                "This result connects to Peter and Weyl from 1927, "
                "and was applied in the learning setting by "
                "Shakerinava et al. in 2022 and Gupta et al. in 2023."
            )
        ) as tracker:
            cit_text = Text(
                "Peter & Weyl 1927  |  Shakerinava et al. 2022  |  Gupta et al. 2023",
                font_size=13, color=GREY_B, slant=ITALIC,
            ).to_edge(DOWN, buff=0.35)
            self.play(FadeIn(cit_text), run_time=tracker.duration * 0.35)
            self.wait(tracker.duration * 0.65)

        self.wait(1.0)

        # --- End Scene ---
        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.6)
        self.wait(0.3)


In [ ]:
%manim Scene14_DataAug

In [ ]:
%manim Scene15_InvariantRepr


In [ ]:
%manim Scene16a_EquivReprNaive

In [ ]:
%manim Scene16b_EquivReprRotation

# SLIDE 18 TỚI SLIDE 22

In [ ]:
# Scenes: Scene18_Recap, Scene19_TwoIdeas, Scene20_Canonicalization,
#         Scene21_CanonChallenges1, Scene22_CanonChallenges2
# Returns (VGroup of cells, VGroup of labels)
def make_cell_row(values, cell_w=0.60, cell_h=0.52,
                  cell_color=GREY_B, txt_color=WHITE,
                  font_size=20, stroke_w=1.2):
    cells  = VGroup()
    labels = VGroup()
    for v in values:
        rect = Rectangle(
            width=cell_w, height=cell_h,
            color=cell_color, stroke_width=stroke_w,
            fill_color=BLACK, fill_opacity=0.0,
        )
        lbl = Text(str(v), font_size=font_size, color=txt_color).move_to(rect)
        cells.add(rect)
        labels.add(lbl)
    cells.arrange(RIGHT, buff=0.0)
    for i, lbl in enumerate(labels):
        lbl.move_to(cells[i])
    return cells, labels


# Scene18_Recap
class Scene18_Recap(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title = Text(
            "How can we get equivariant models?",
            font_size=32, color=WHITE,
        ).to_edge(UP, buff=0.40)
        self.play(Write(title), run_time=0.65)

        # Recap formulas UR -- visual continuity with Scene13/Scene16
        recap_inv_lbl  = Text("Inv.:",   font_size=14, color=BLUE)
        recap_inv_fx   = MathTex(r"f(g\cdot x)=f(x)", font_size=16, color=BLUE)
        recap_inv_row  = VGroup(recap_inv_lbl,  recap_inv_fx ).arrange(RIGHT, buff=0.14)

        recap_eq_lbl   = Text("Equiv.:", font_size=14, color=GREEN)
        recap_eq_fx    = MathTex(r"f(g\cdot x)=g\cdot f(x)", font_size=16, color=GREEN)
        recap_eq_row   = VGroup(recap_eq_lbl, recap_eq_fx).arrange(RIGHT, buff=0.14)

        recap_block = VGroup(recap_inv_row, recap_eq_row).arrange(DOWN, buff=0.14, aligned_edge=LEFT)
        recap_block.next_to(title, DOWN, buff=0.35)
        recap_box = SurroundingRectangle(
            recap_block, color=GREY_B, stroke_width=1.0,
            corner_radius=0.10, buff=0.14,
        )
        self.play(FadeIn(recap_block), Create(recap_box), run_time=0.50)

        # Beat 1 -- Introduce 2-column structure
        divider = DashedLine(
            start=np.array([0.00,  1.75, 0]),
            end  =np.array([0.00, -3.00, 0]),
            color=GREY_B, stroke_width=0.8, dash_length=0.14,
        )
        hdr_left = Text(
            "Off-the-shelf models",
            font_size=18, color=BLUE, weight=BOLD,
        ).move_to(np.array([-3.00, 1.40, 0]))
        hdr_right = Text(
            "Constructing equivariant models",
            font_size=17, color=GREEN, weight=BOLD,
        ).move_to(np.array([3.20, 1.40, 0]))

        with self.voiceover(
            text=(
                "With the mathematical foundations in place, "
                "let us quickly recap the full toolkit: two broad families of approaches "
                "for getting equivariant or invariant models."
            )
        ) as tracker:
            self.play(
                Create(divider),
                FadeIn(hdr_left), FadeIn(hdr_right),
                run_time=tracker.duration * 0.55,
            )
            self.wait(tracker.duration * 0.45)

        # Beat 2 -- All bullets, fast recall
        left_b1 = Text("- Data augmentation",      font_size=16, color=WHITE)
        left_b2 = Text("- Canonicalization",        font_size=16, color=WHITE)
        left_b3 = Text("- Group / frame averaging", font_size=16, color=WHITE)
        left_col = VGroup(left_b1, left_b2, left_b3).arrange(
            DOWN, aligned_edge=LEFT, buff=0.22
        ).move_to(np.array([-3.00, -0.10, 0]))

        # RIGHT bullets
        right_b1  = Text("- Linear equivariant layers",      font_size=15, color=WHITE)
        right_b1b = Text("  (param sharing, group conv)",     font_size=13, color=GREY_B)
        right_b2  = Text("- Invariant theory",                font_size=15, color=WHITE)
        right_b3  = Text("- Representation theory",           font_size=15, color=WHITE)
        right_col = VGroup(right_b1, right_b1b, right_b2, right_b3).arrange(
            DOWN, aligned_edge=LEFT, buff=0.18
        ).move_to(np.array([3.20, -0.22, 0]))

        with self.voiceover(
            text=(
                "The off-the-shelf family includes data augmentation, "
                "canonicalization, and group or frame averaging. "
                "The constructing family includes equivariant layers, "
                "invariant theory, and representation theory."
            )
        ) as tracker:
            self.play(
                FadeIn(left_col), FadeIn(right_col),
                run_time=tracker.duration * 0.55,
            )
            self.wait(tracker.duration * 0.45)

        # Beat 3 -- Highlight canonicalization
        canon_box = SurroundingRectangle(
            left_b2, color=YELLOW, stroke_width=1.8,
            corner_radius=0.06, buff=0.10,
        )
        with self.voiceover(
            text=(
                "In this section, we zoom in on canonicalization -- "
                "the idea of mapping every input to a standardized orbit representative "
                "before applying the model."
            )
        ) as tracker:
            self.play(
                Indicate(left_b2, color=YELLOW, scale_factor=1.12),
                run_time=tracker.duration * 0.40,
            )
            self.play(Create(canon_box), run_time=tracker.duration * 0.30)
            self.wait(tracker.duration * 0.30)

        # --- End Scene ---
        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.6)
        self.wait(0.3)


# Scene19_TwoIdeas
class Scene19_TwoIdeas(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title = Text(
            "Two Ideas for Making f Invariant",
            font_size=34, color=WHITE,
        ).to_edge(UP, buff=0.45)
        self.play(Write(title), run_time=0.60)

        # Vertical separator
        separator = DashedLine(
            start=np.array([0.00,  2.00, 0]),
            end  =np.array([0.00, -2.80, 0]),
            color=GREY_B, stroke_width=1.0, dash_length=0.14,
        )

        # LEFT: canonicalization -- single f applied once
        lbl_canon = Text("canonicalization", font_size=20, color=BLUE, weight=BOLD)
        lbl_canon.move_to(np.array([-2.90, 1.55, 0]))

        arr_down_L = Arrow(
            start=np.array([-2.90, 0.90, 0]),
            end  =np.array([-2.90, 0.10, 0]),
            color=GREY_B, stroke_width=1.5,
            max_tip_length_to_length_ratio=0.20,
        )
        lbl_x_arr = Text("x →", font_size=14, color=GREY_B).next_to(arr_down_L, UP, buff=0.05)

        box_f_L = Rectangle(
            width=0.70, height=0.50,
            color=BLUE, stroke_width=1.6,
            fill_color=BLUE, fill_opacity=0.15,
        ).move_to(np.array([-2.90, -0.25, 0]))
        lbl_f_L = Text("f", font_size=22, color=BLUE).move_to(box_f_L)
        note_L  = Text(
            "apply f once",
            font_size=14, color=GREY_B, slant=ITALIC,
        ).next_to(box_f_L, DOWN, buff=0.20)

        # RIGHT: symmetrization -- 4 stacked f boxes
        lbl_sym = Text(
            "symmetrization / group averaging",
            font_size=17, color=GREEN, weight=BOLD,
        ).move_to(np.array([2.80, 1.55, 0]))

        f_boxes_R = VGroup()
        for i in range(4):
            arr = Arrow(
                start=np.array([2.80, 0.88 - i * 0.52, 0]),
                end  =np.array([2.80, 0.55 - i * 0.52, 0]),
                color=GREY_B, stroke_width=1.0,
                max_tip_length_to_length_ratio=0.25,
            )
            box = Rectangle(
                width=0.60, height=0.38,
                color=GREEN, stroke_width=1.2,
                fill_color=GREEN, fill_opacity=0.12,
            ).move_to(np.array([2.80, 0.35 - i * 0.52, 0]))
            lbl = Text("f", font_size=16, color=GREEN).move_to(box)
            f_boxes_R.add(VGroup(arr, box, lbl))

        note_R = Text(
            "apply f to all, then average",
            font_size=14, color=GREY_B, slant=ITALIC,
        ).move_to(np.array([2.80, -1.50, 0]))

        left_group  = VGroup(lbl_canon, lbl_x_arr, arr_down_L, box_f_L, lbl_f_L, note_L)
        right_group = VGroup(lbl_sym, f_boxes_R, note_R)

        with self.voiceover(
            text=(
                "There are two natural strategies for making a function f invariant. "
                "Canonicalization applies f exactly once -- to a standardized "
                "representative of the orbit. "
                "Symmetrization, or group averaging, applies f to every orbit element "
                "and averages the results."
            )
        ) as tracker:
            self.play(Create(separator),           run_time=tracker.duration * 0.10)
            self.play(
                FadeIn(left_group), FadeIn(right_group),
                run_time=tracker.duration * 0.45,
            )
            self.play(
                Indicate(lbl_canon, color=YELLOW, scale_factor=1.12),
                run_time=tracker.duration * 0.30,
            )
            self.wait(tracker.duration * 0.15)

        # --- End Scene ---
        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.6)
        self.wait(0.3)


# Scene20_Canonicalization
class Scene20_Canonicalization(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title = Text(
            "Canonicalization",
            font_size=40, color=WHITE,
        ).to_edge(UP, buff=0.45)
        self.play(Write(title), run_time=0.60)

        # Beat 1 -- Concept definition
        # Simple concept row: x -> canon(x) -> f -> output
        lbl_x_c  = Text("x",        font_size=22, color=WHITE)
        arr_c1   = Arrow(ORIGIN, RIGHT * 0.8, color=GREY_B,
                         stroke_width=1.4, max_tip_length_to_length_ratio=0.18)
        lbl_cx   = Text("canon(x)", font_size=20, color=YELLOW)
        arr_c2   = Arrow(ORIGIN, RIGHT * 0.8, color=GREY_B,
                         stroke_width=1.4, max_tip_length_to_length_ratio=0.18)
        box_f_c  = Rectangle(width=0.60, height=0.45,
                              color=BLUE, stroke_width=1.4,
                              fill_color=BLUE, fill_opacity=0.15)
        lbl_fc   = Text("f", font_size=20, color=BLUE).move_to(box_f_c)
        arr_c3   = Arrow(ORIGIN, RIGHT * 0.8, color=GREY_B,
                         stroke_width=1.4, max_tip_length_to_length_ratio=0.18)
        lbl_out  = Text("output", font_size=20, color=GREEN)

        concept_row = VGroup(
            lbl_x_c, arr_c1, lbl_cx, arr_c2, VGroup(box_f_c, lbl_fc), arr_c3, lbl_out
        ).arrange(RIGHT, buff=0.20)
        concept_row.move_to(np.array([0.00, 1.20, 0]))

        note_cx = Text(
            "same for all orbit elements",
            font_size=13, color=GREY_B, slant=ITALIC,
        ).next_to(lbl_cx, UP, buff=0.12)

        with self.voiceover(
            text=(
                "Canonicalization is straightforward in principle. "
                "Instead of applying f directly to x, we first map x to a canonical "
                "representative of its orbit -- a standardized form. "
                "All orbit elements share the same canonical form, so f becomes invariant."
            )
        ) as tracker:
            self.play(FadeIn(concept_row),  run_time=tracker.duration * 0.45)
            self.play(FadeIn(note_cx),      run_time=tracker.duration * 0.25)
            self.play(
                Indicate(lbl_cx, color=YELLOW, scale_factor=1.10),
                run_time=tracker.duration * 0.18,
            )
            self.wait(tracker.duration * 0.12)

        # Beat 2 -- Sorting pipeline: permutation invariant
        self.play(FadeOut(concept_row), FadeOut(note_cx), run_time=0.35)
        def pipeline_row(input_vals, output_vals, result_str,
                         cell_color_in=GREY_B, cell_color_out=YELLOW,
                         result_color=GREEN, y_pos=0.0):
            in_cells, in_lbls = make_cell_row(
                input_vals, cell_color=cell_color_in,
                txt_color=WHITE, font_size=19
            )
            out_cells, out_lbls = make_cell_row(
                output_vals, cell_color=cell_color_out,
                txt_color=WHITE, font_size=19
            )
            arrow1 = Arrow(ORIGIN, RIGHT * 0.65, color=ORANGE,
                           stroke_width=1.4, max_tip_length_to_length_ratio=0.16)
            lbl_sort = Text("sort", font_size=14, color=ORANGE).next_to(
                arrow1, UP, buff=0.06
            )
            arrow2 = Arrow(ORIGIN, RIGHT * 0.65, color=BLUE,
                           stroke_width=1.4, max_tip_length_to_length_ratio=0.16)
            lbl_f = Text("f", font_size=14, color=BLUE).next_to(arrow2, UP, buff=0.06)
            result = MathTex(result_str, font_size=24, color=result_color)

            row = VGroup(
                VGroup(in_cells, in_lbls),
                VGroup(arrow1, lbl_sort),
                VGroup(out_cells, out_lbls),
                VGroup(arrow2, lbl_f),
                result,
            ).arrange(RIGHT, buff=0.22)
            row.move_to(np.array([0.00, y_pos, 0]))
            return row, VGroup(in_cells, in_lbls), VGroup(out_cells, out_lbls), \
                   VGroup(arrow1, lbl_sort), VGroup(arrow2, lbl_f), result

        row_A, in_A, out_A, arr_sort_A, arr_f_A, res_A = pipeline_row(
            ["5", "3", "4"], ["3", "4", "5"], r"x_k - x_1 = 2",
            cell_color_out=YELLOW, y_pos=0.80,
        )
        row_B, in_B, out_B, arr_sort_B, arr_f_B, res_B = pipeline_row(
            ["4", "3", "5"], ["3", "4", "5"], r"x_k - x_1 = 2",
            cell_color_out=YELLOW, y_pos=0.00,
        )

        inv_lbl = Text(
            "permutation invariant",
            font_size=15, color=GREEN, weight=BOLD,
        ).next_to(VGroup(res_A, res_B), RIGHT, buff=0.25)

        # Beat 2a -- Sorting pipeline: invariant, row A
        with self.voiceover(
            text=(
                "The classic example is sorting for permutation invariance. "
                "Take the vector five, three, four. Sorting always gives three, four, five "
                "-- the canonical form. "
                "Apply f: say it computes x-sub-k minus x-sub-one, giving two."
            )
        ) as tracker:
            self.play(FadeIn(in_A),                  run_time=tracker.duration * 0.20)
            self.play(GrowArrow(arr_sort_A[0]),
                      FadeIn(arr_sort_A[1]),          run_time=tracker.duration * 0.20)
            self.play(FadeIn(out_A),                  run_time=tracker.duration * 0.20)
            self.play(GrowArrow(arr_f_A[0]),
                      FadeIn(arr_f_A[1]),             run_time=tracker.duration * 0.20)
            self.play(FadeIn(res_A),                  run_time=tracker.duration * 0.20)

        # Beat 2b-i -- Sorting pipeline: invariant, row B (first part)
        with self.voiceover(
            text=(
                "Now consider a different permutation: four, three, five. "
                "It sorts to the same three, four, five, so output is also two."
            )
        ) as tracker:
            self.play(FadeIn(in_B),                   run_time=tracker.duration * 0.25)
            self.play(GrowArrow(arr_sort_B[0]),
                      FadeIn(arr_sort_B[1]),          run_time=tracker.duration * 0.25)
            self.play(FadeIn(out_B),                  run_time=tracker.duration * 0.25)
            self.play(GrowArrow(arr_f_B[0]),
                      FadeIn(arr_f_B[1]),             run_time=tracker.duration * 0.15)
            self.play(FadeIn(res_B),                  run_time=tracker.duration * 0.10)

        # Beat 2b-ii -- Conclude invariance
        with self.voiceover(
            text=(
                "No matter the input permutation, the result is identical -- "
                "this is permutation invariance."
            )
        ) as tracker:
            self.play(
                Indicate(out_A, color=YELLOW, scale_factor=1.06),
                Indicate(out_B, color=YELLOW, scale_factor=1.06),
                run_time=tracker.duration * 0.50,
            )
            self.play(FadeIn(inv_lbl),                run_time=tracker.duration * 0.30)
            self.wait(tracker.duration * 0.20)

        # Beat 3 -- Equivariant case
        def make_equiv_row(in_vals, canon_vals, fx_vals, out_vals, y_pos):
            in_c, in_l = make_cell_row(in_vals, cell_color=GREY_B, txt_color=WHITE, font_size=16, cell_w=0.45, cell_h=0.40)
            mid_c, mid_l = make_cell_row(canon_vals, cell_color=YELLOW, txt_color=WHITE, font_size=16, cell_w=0.45, cell_h=0.40)
            fx_c, fx_l = make_cell_row(fx_vals, cell_color=BLUE, txt_color=WHITE, font_size=16, cell_w=0.45, cell_h=0.40)
            out_c, out_l = make_cell_row(out_vals, cell_color=GREEN, txt_color=WHITE, font_size=16, cell_w=0.45, cell_h=0.40)

            arr_s = Arrow(ORIGIN, RIGHT*0.45, color=ORANGE, stroke_width=1.2, max_tip_length_to_length_ratio=0.18)
            lbl_s = Text("sort", font_size=11, color=ORANGE).next_to(arr_s, UP, buff=0.03)
            arr_f = Arrow(ORIGIN, RIGHT*0.45, color=BLUE, stroke_width=1.2, max_tip_length_to_length_ratio=0.18)
            lbl_f = Text("f", font_size=11, color=BLUE).next_to(arr_f, UP, buff=0.03)
            arr_i = Arrow(ORIGIN, RIGHT*0.45, color=ORANGE, stroke_width=1.2, max_tip_length_to_length_ratio=0.18)
            lbl_i = Text("inv.", font_size=11, color=ORANGE).next_to(arr_i, UP, buff=0.03)

            row = VGroup(
                VGroup(in_c, in_l), VGroup(arr_s, lbl_s),
                VGroup(mid_c, mid_l), VGroup(arr_f, lbl_f),
                VGroup(fx_c, fx_l), VGroup(arr_i, lbl_i),
                VGroup(out_c, out_l)
            ).arrange(RIGHT, buff=0.15)
            row.move_to(np.array([0, y_pos, 0]))
            return row, VGroup(in_c, in_l), VGroup(mid_c, mid_l), VGroup(fx_c, fx_l), VGroup(out_c, out_l), arr_s, lbl_s, arr_f, lbl_f, arr_i, lbl_i

        row_E1, in_E1, mid_E1, fx_E1, out_E1, arr_sE1, lbl_sE1, arr_fE1, lbl_fE1, arr_iE1, lbl_iE1 = make_equiv_row(
            ["5","3","4"], ["3","4","5"], ["3","7","12"], ["12","3","7"], y_pos=-1.00
        )
        row_E2, in_E2, mid_E2, fx_E2, out_E2, arr_sE2, lbl_sE2, arr_fE2, lbl_fE2, arr_iE2, lbl_iE2 = make_equiv_row(
            ["4","3","5"], ["3","4","5"], ["3","7","12"], ["7","3","12"], y_pos=-1.70
        )

        eq_lbl = Text(
            "permutation equivariant",
            font_size=14, color=GREEN, weight=BOLD,
        ).next_to(VGroup(row_E1, row_E2), DOWN, buff=0.14)

        # Beat 3a -- introduce equivariant pipeline with first row
        with self.voiceover(
            text=(
                "For permutation equivariance, we add one step: "
                "after applying f in canonical space, "
                "we invert the original sorting permutation "
                "to map the output back to the input's order."
            )
        ) as tracker:
            self.play(
                FadeIn(in_E1), FadeIn(in_E2),
                run_time=tracker.duration * 0.15,
            )
            self.play(
                GrowArrow(arr_sE1), FadeIn(lbl_sE1),
                GrowArrow(arr_sE2), FadeIn(lbl_sE2),
                FadeIn(mid_E1), FadeIn(mid_E2),
                run_time=tracker.duration * 0.25,
            )
            self.play(
                GrowArrow(arr_fE1), FadeIn(lbl_fE1),
                GrowArrow(arr_fE2), FadeIn(lbl_fE2),
                FadeIn(fx_E1), FadeIn(fx_E2),
                run_time=tracker.duration * 0.30,
            )
            self.play(
                GrowArrow(arr_iE1), FadeIn(lbl_iE1),
                GrowArrow(arr_iE2), FadeIn(lbl_iE2),
                FadeIn(out_E1), FadeIn(out_E2),
                run_time=tracker.duration * 0.30,
            )

        # Beat 3b -- conclude equivariance preserved
        with self.voiceover(
            text=(
                "Different input permutations use different sort permutations, "
                "so inverting them produces outputs correctly related by that permutation "
                "-- equivariance is preserved."
            )
        ) as tracker:
            self.play(FadeIn(eq_lbl),            run_time=tracker.duration * 0.40)
            self.play(
                Indicate(eq_lbl, color=GREEN, scale_factor=1.06),
                run_time=tracker.duration * 0.30,
            )
            self.wait(tracker.duration * 0.30)

        # Beat 4 -- Canonicalization can be learned
        self.play(
            FadeOut(row_A), FadeOut(row_B), FadeOut(inv_lbl),
            FadeOut(row_E1), FadeOut(row_E2), FadeOut(eq_lbl),
            run_time=0.40,
        )

        learned_text = Text(
            "Canonicalization can be learned!",
            font_size=26, color=TEAL, weight=BOLD,
        ).move_to(np.array([0.00, 0.30, 0]))

        # Intentionally truncated -- slide has 9+ references (Yuceer 1993, Lowe 2004,
        # Niepert 2016, Winter 2022, Vadgama 2022, Zhang 2019, Leo 2022, Kaba 2022, ...)
        cit_text = Text(
            "Kaba et al. 2022  |  Zhang et al. 2019  |  Leo et al. 2022  |  ...",
            font_size=13, color=GREY_B, slant=ITALIC,
        ).next_to(learned_text, DOWN, buff=0.30)

        with self.voiceover(
            text=(
                "And canonicalization does not need to be hand-crafted. "
                "It can be learned by a neural network, and recent work has shown "
                "this is effective in practice across multiple domains."
            )
        ) as tracker:
            self.play(FadeIn(learned_text),     run_time=tracker.duration * 0.35)
            self.play(FadeIn(cit_text),         run_time=tracker.duration * 0.30)
            self.play(
                Indicate(learned_text, color=YELLOW, scale_factor=1.05),
                run_time=tracker.duration * 0.20,
            )
            self.wait(tracker.duration * 0.15)

        # --- End Scene ---
        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.6)
        self.wait(0.3)


# Scene21_CanonChallenges1
class Scene21_CanonChallenges1(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title = Text(
            "Canonicalization: Challenges",
            font_size=36, color=WHITE,
        ).to_edge(UP, buff=0.45)
        self.play(Write(title), run_time=0.65)

        # Beat 1 -- Challenges list
        ch1_main = Text(
            "- Not always well-defined",
            font_size=19, color=RED,
        )
        ch1_sub  = Text(
            "  (e.g. rotation + permutation for point clouds)",
            font_size=14, color=GREY_B,
        )
        ch2_main = Text(
            "- Symmetries in the input",
            font_size=19, color=RED,
        )

        challenge_list = VGroup(ch1_main, ch1_sub, ch2_main).arrange(
            DOWN, aligned_edge=LEFT, buff=0.18
        ).move_to(np.array([-0.50, 1.00, 0]))

        with self.voiceover(
            text=(
                "Canonicalization is not always easy to define. "
                "Some group actions -- such as rotations combined with permutations "
                "for point clouds -- do not have a simple canonical form. "
                "And when the input itself has symmetries, "
                "the canonical form can be ambiguous."
            )
        ) as tracker:
            self.play(FadeIn(ch1_main),          run_time=tracker.duration * 0.22)
            self.play(FadeIn(ch1_sub),           run_time=tracker.duration * 0.14)
            self.play(FadeIn(ch2_main),          run_time=tracker.duration * 0.22)
            self.wait(tracker.duration * 0.42)

        # Beat 2 -- 1D continuity diagram: [5,3,3] and [5,3.01,3]
        ch3_hdr = Text(
            "- Continuity",
            font_size=19, color=RED,
        )
        ch3_sub = Text(
            "  (Zhang et al 2020, Dym et al 2024)",
            font_size=13, color=GREY_B,
        )
        ch3_row = VGroup(ch3_hdr, ch3_sub).arrange(DOWN, aligned_edge=LEFT, buff=0.10)
        ch3_row.next_to(challenge_list, DOWN, buff=0.28, aligned_edge=LEFT)

        # 1D continuity diagram
        DIAG_Y   = -1.80
        LEFT_X   = -3.20
        RIGHT_X  =  2.00

        def v_row(vals, x_center, y_center, clr=GREY_B, font_sz=17):
            cells, lbls = make_cell_row(vals, cell_color=clr,
                                        txt_color=WHITE, font_size=font_sz,
                                        cell_w=0.55, cell_h=0.44, stroke_w=1.0)
            grp = VGroup(cells, lbls)
            grp.move_to(np.array([x_center, y_center, 0]))
            return grp

        in_row1  = v_row(["5", "3",    "3"   ], LEFT_X,  DIAG_Y + 0.38, GREY_B)
        in_row2  = v_row(["5", "3.01", "3"   ], LEFT_X,  DIAG_Y - 0.38, GREY_B)
        out_row1 = v_row(["3", "3",    "5"   ], RIGHT_X, DIAG_Y + 0.38, YELLOW)
        out_row2 = v_row(["3", "3.01", "5"   ], RIGHT_X, DIAG_Y - 0.38, YELLOW)

        # "close" braces
        brace_L = Brace(VGroup(in_row1,  in_row2), direction=LEFT,  color=GREY_B, buff=0.05)  # [m2]
        lbl_close_L = Text("close", font_size=13, color=GREY_B).next_to(
            brace_L, LEFT, buff=0.10
        )
        brace_R = Brace(VGroup(out_row1, out_row2), direction=RIGHT, color=GREEN, buff=0.05)  # [m2]
        lbl_close_R = Text("close -- OK", font_size=13, color=GREEN).next_to(
            brace_R, RIGHT, buff=0.10
        )

        # Center arrow
        arr_canon = Arrow(
            start=np.array([-1.55, DIAG_Y, 0]),
            end  =np.array([ 0.80, DIAG_Y, 0]),
            color=ORANGE, stroke_width=1.6,
            max_tip_length_to_length_ratio=0.12,
        )
        lbl_canon_arr = Text(
            "canonicalization", font_size=13, color=ORANGE,
        ).next_to(arr_canon, UP, buff=0.08)

        # Beat 2a -- introduce continuity + example [5,3,3] -> [3,3,5]
        with self.voiceover(
            text=(
                "The most fundamental challenge is continuity. "
                "We want canonicalization to be continuous -- "
                "close inputs should map to close canonical forms. "
                "Consider five, three, three. Sorting gives three, three, five."
            )
        ) as tracker:
            self.play(FadeIn(ch3_row),             run_time=tracker.duration * 0.20)
            self.play(
                FadeIn(in_row1),
                run_time=tracker.duration * 0.25,
            )
            self.play(
                GrowArrow(arr_canon), FadeIn(lbl_canon_arr),
                run_time=tracker.duration * 0.25,
            )
            self.play(
                FadeIn(out_row1),
                run_time=tracker.duration * 0.30,
            )

        # Beat 2b -- perturb + conclude continuity holds in 1D
        with self.voiceover(
            text=(
                "Perturb slightly: five, three-point-oh-one, three. "
                "Sorting gives three, three-point-oh-one, five. "
                "These are close. In this one-dimensional case, continuity holds."
            )
        ) as tracker:
            self.play(
                FadeIn(in_row2),
                run_time=tracker.duration * 0.18,
            )
            self.play(
                Create(brace_L), FadeIn(lbl_close_L),
                run_time=tracker.duration * 0.15,
            )
            self.play(
                FadeIn(out_row2),
                run_time=tracker.duration * 0.18,
            )
            self.play(
                Create(brace_R), FadeIn(lbl_close_R),
                run_time=tracker.duration * 0.25,
            )
            self.play(
                Indicate(lbl_close_R, color=GREEN, scale_factor=1.08),
                run_time=tracker.duration * 0.14,
            )
            self.wait(tracker.duration * 0.10)

        # Beat 3 -- Bridge to next scene
        bridge_note = Text(
            "1D: OK.  Multi-dimensional: problem ahead.",
            font_size=17, color=GREY_B, slant=ITALIC,
        ).to_edge(DOWN, buff=0.40)

        with self.voiceover(
            text=(
                "But this is a deceptively simple case. "
                "In one dimension, sorting is well-behaved. "
                "Adding even a second row of values -- "
                "making the input a matrix of vectors -- "
                "breaks this continuity entirely."
            )
        ) as tracker:
            self.play(FadeIn(bridge_note),         run_time=tracker.duration * 0.35)
            box_ok = SurroundingRectangle(
                VGroup(out_row1, out_row2),
                color=GREEN, stroke_width=2.0, buff=0.10,
            )
            self.play(Create(box_ok), run_time=tracker.duration * 0.25)
            self.play(FadeOut(box_ok), run_time=tracker.duration * 0.20)
            self.wait(tracker.duration * 0.20)

        # --- End Scene ---
        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.6)
        self.wait(0.3)


# Scene22_CanonChallenges2
class Scene22_CanonChallenges2(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title = Text(
            "Canonicalization: Challenges (continued)",
            font_size=31, color=WHITE,
        ).to_edge(UP, buff=0.42)
        self.play(Write(title), run_time=0.65)

        # Beat 1 -- 2D continuity failure diagram
        LEFT_X_D  = -3.80
        RIGHT_X_D =  2.60
        DIAG_Y1   =  0.75   # center y of first matrix pair
        DIAG_Y2   = -0.75   # center y of second matrix pair
        ROW_GAP   =  0.45   # gap between top and bottom row of one matrix

        def v_row_d(vals, x_center, y_center, clr=GREY_B, font_sz=16):
            cells, lbls = make_cell_row(
                vals, cell_color=clr, txt_color=WHITE,
                font_size=font_sz, cell_w=0.55, cell_h=0.40, stroke_w=1.0,
            )
            grp = VGroup(cells, lbls)
            grp.move_to(np.array([x_center, y_center, 0]))
            return grp

        # LEFT inputs (close)
        in1_top = v_row_d(["5",  "3",    "3"], LEFT_X_D, DIAG_Y1 + ROW_GAP / 2)
        in1_bot = v_row_d(["1",  "7",    "8"], LEFT_X_D, DIAG_Y1 - ROW_GAP / 2)
        in2_top = v_row_d(["5",  "3.01", "3"], LEFT_X_D, DIAG_Y2 + ROW_GAP / 2)
        in2_bot = v_row_d(["1",  "7",    "8"], LEFT_X_D, DIAG_Y2 - ROW_GAP / 2)

        # RIGHT canonicalizations
        ca1_top = v_row_d(["3",  "3",    "5"], RIGHT_X_D, DIAG_Y1 + ROW_GAP / 2, YELLOW)
        ca1_bot = v_row_d(["7",  "8",    "1"], RIGHT_X_D, DIAG_Y1 - ROW_GAP / 2, YELLOW)
        ca2_top = v_row_d(["3",  "3.01", "5"], RIGHT_X_D, DIAG_Y2 + ROW_GAP / 2, RED)
        ca2_bot = v_row_d(["8",  "7",    "1"], RIGHT_X_D, DIAG_Y2 - ROW_GAP / 2, RED)

        # "close" brace LEFT
        in_grp_all   = VGroup(in1_top, in1_bot, in2_top, in2_bot)
        brace_in     = Brace(in_grp_all, direction=LEFT, color=GREY_B)
        lbl_close_in = Text("close", font_size=13, color=GREY_B).next_to(
            brace_in, LEFT, buff=0.08
        )

        # "far!" label RIGHT
        ca_grp_all  = VGroup(ca1_top, ca1_bot, ca2_top, ca2_bot)
        lbl_far     = Text("far!", font_size=26, color=RED, weight=BOLD).next_to(
            ca_grp_all, RIGHT, buff=0.28
        )

        # Center canonicalization arrow
        arr_mid = Arrow(
            start=np.array([-1.65, 0.00, 0]),
            end  =np.array([ 0.95, 0.00, 0]),
            color=ORANGE, stroke_width=1.8,
            max_tip_length_to_length_ratio=0.10,
        )
        lbl_mid = Text(
            "canonicalization", font_size=13, color=ORANGE,
        ).next_to(arr_mid, UP, buff=0.10)

        # Red highlight box around ca2_bot to show the "jump"
        jump_box = SurroundingRectangle(
            ca2_bot, color=RED, stroke_width=2.0,
            corner_radius=0.06, buff=0.06,
        )

        # Beat 1a -- Build first matrix and its canonicalization
        with self.voiceover(
            text=(
                "Now we add a second row. Each input is a matrix of two stacked vectors. "
                "Matrix one: top row five, three, three -- bottom row one, seven, eight. "
                "Sorting columns by the top row gives three, three, five -- "
                "with the bottom row reordered to seven, eight, one."
            )
        ) as tracker:
            self.play(
                FadeIn(in1_top), FadeIn(in1_bot),
                run_time=tracker.duration * 0.35,
            )
            self.play(
                GrowArrow(arr_mid), FadeIn(lbl_mid),
                run_time=tracker.duration * 0.30,
            )
            self.play(
                FadeIn(ca1_top), FadeIn(ca1_bot),
                run_time=tracker.duration * 0.35,
            )

        # Beat 1b -- Perturb and show jump
        with self.voiceover(
            text=(
                "Now perturb the top row: five, three-point-oh-one, three. "
                "Sorting now uses a different column order -- "
                "and the bottom row becomes eight, seven, one. "
                "A tiny change in input caused the bottom row to jump. "
                "Continuity has broken."
            )
        ) as tracker:
            self.play(
                FadeIn(in2_top), FadeIn(in2_bot),
                run_time=tracker.duration * 0.20,
            )
            self.play(
                Create(brace_in), FadeIn(lbl_close_in),
                run_time=tracker.duration * 0.15,
            )
            self.play(
                FadeIn(ca2_top), FadeIn(ca2_bot),
                run_time=tracker.duration * 0.20,
            )
            self.play(
                Create(jump_box),
                run_time=tracker.duration * 0.15,
            )
            self.play(
                FadeIn(lbl_far),
                run_time=tracker.duration * 0.12,
            )
            self.play(
                Indicate(lbl_far, color=RED, scale_factor=1.12),
                run_time=tracker.duration * 0.08,
            )
            self.wait(tracker.duration * 0.10)

        # Beat 2 -- Theorem box
        self.play(
            FadeOut(in_grp_all), FadeOut(brace_in), FadeOut(lbl_close_in),
            FadeOut(arr_mid), FadeOut(lbl_mid),
            FadeOut(ca_grp_all), FadeOut(jump_box), FadeOut(lbl_far),
            run_time=0.45,
        )

        thm_line1 = Text(
            "For permutations of n > 1 vectors in d > 1 dimensions,",
            font_size=18, color=WHITE,
        )
        thm_line2 = Text(
            "and for SO(2):",
            font_size=18, color=WHITE,
        )
        thm_line3 = Text(
            "there is no continuous canonicalization.",
            font_size=20, color=GREEN, weight=BOLD,
        )
        thm_block = VGroup(thm_line1, thm_line2, thm_line3).arrange(
            DOWN, buff=0.22, aligned_edge=LEFT
        ).move_to(np.array([0.00, 0.40, 0]))
        thm_box = SurroundingRectangle(
            thm_block, color=GREEN, stroke_width=1.8,
            corner_radius=0.12, buff=0.30,
        )

        with self.voiceover(
            text=(
                "This is not a special case -- it is unavoidable. "
                "No matter how we design the canonicalization, "
                "as soon as we have more than one vector and more than one dimension, "
                "some pair of close inputs will always map to far-apart canonical forms. "
                "The same impossibility holds for SO(2)."
            )
        ) as tracker:
            self.play(
                FadeIn(thm_line1), FadeIn(thm_line2),
                run_time=tracker.duration * 0.30,
            )
            self.play(Write(thm_line3),         run_time=tracker.duration * 0.22)
            self.play(Create(thm_box),          run_time=tracker.duration * 0.20)
            self.play(
                Indicate(thm_box, color=YELLOW, scale_factor=1.04),
                run_time=tracker.duration * 0.18,
            )
            self.wait(tracker.duration * 0.10)

        # Beat 3 -- Citations + bridge to next part
        cit_text = Text(
            "Dym et al. 2024  |  Zhang et al. 2020  "
            "|  Example adapted from Dym et al. 2024",
            font_size=12, color=GREY_B, slant=ITALIC,
        ).to_edge(DOWN, buff=0.32)

        next_topic = Text(
            "Next: Group / Frame Averaging",
            font_size=24, color=YELLOW, weight=BOLD,
        ).move_to(np.array([0, -1.8, 0]))

        with self.voiceover(
            text=(
                "This theoretical limit was established by Dym et al. in 2024 "
                "and Zhang et al. in 2020. "
                "It motivates alternative approaches -- "
                "particularly group averaging and frame averaging, "
                "which we turn to next."
            )
        ) as tracker:
            self.play(FadeIn(cit_text),         run_time=tracker.duration * 0.30)
            self.play(FadeIn(next_topic),       run_time=tracker.duration * 0.40)
            self.wait(tracker.duration * 0.30)

        self.wait(1.5)

        # --- End Scene ---
        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.6)
        self.wait(0.3)


In [ ]:
%manim -qh Scene18_Recap

In [ ]:
%manim -qh Scene19_TwoIdeas

In [ ]:
%manim -qh Scene20_Canonicalization

In [ ]:
%manim -qh Scene21_CanonChallenges1

In [ ]:
%manim -qh Scene22_CanonChallenges2

# SLIDE 23 TỚI SLIDE 24

In [ ]:
# Scenes: Scene23_GroupAveraging, Scene24_JanossyPooling
# Shared helper: build a row of labeled squares
# Returns (VGroup of cells, VGroup of labels)
def make_cell_row(values, cell_w=0.60, cell_h=0.52,
                  cell_color=GREY_B, txt_color=WHITE,
                  font_size=20, stroke_w=1.2):
    cells  = VGroup()
    labels = VGroup()
    for v in values:
        rect = Rectangle(
            width=cell_w, height=cell_h,
            color=cell_color, stroke_width=stroke_w,
            fill_color=BLACK, fill_opacity=0.0,
        )
        lbl = Text(str(v), font_size=font_size, color=txt_color).move_to(rect)
        cells.add(rect)
        labels.add(lbl)
    cells.arrange(RIGHT, buff=0.0)
    for i, lbl in enumerate(labels):
        lbl.move_to(cells[i])
    return cells, labels


# Scene23_GroupAveraging
class Scene23_GroupAveraging(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title = Text(
            "Group Averaging (Reynolds Operator)",
            font_size=34, color=WHITE,
        ).to_edge(UP, buff=0.42)
        self.play(Write(title), run_time=0.70)

        # Beat 1 -- Introduce Reynolds operator
        yarotsky_lbl = Text(
            "Universal approximation of invariant / equivariant functions  (Yarotsky, 2021)",
            font_size=14, color=GREY_B, slant=ITALIC,
        ).next_to(title, DOWN, buff=0.22)

        with self.voiceover(
            text=(
                "The simplest way to symmetrize any neural network is group averaging "
                "-- also called the Reynolds operator. "
                "The key insight is elegant: average the model's output "
                "over all transformations of the input."
            )
        ) as tracker:
            self.play(FadeIn(yarotsky_lbl),  run_time=tracker.duration * 0.40)
            self.wait(tracker.duration * 0.60)

        # Beat 2a -- Invariant formula + Generic NN label + diagram
        inv_formula = MathTex(
            r"\bar{f}(x) = \frac{1}{|G|} \sum_{g \in G} f(g^{-1} \cdot x)",
            font_size=30, color=WHITE,
        ).move_to(np.array([-2.50, 0.70, 0]))

        generic_nn_lbl = Text(
            "Generic NN", font_size=14, color=BLUE, weight=BOLD,
        ).next_to(inv_formula, DOWN, buff=0.14)
        generic_nn_box = SurroundingRectangle(
            generic_nn_lbl, color=BLUE, stroke_width=1.2,
            corner_radius=0.06, buff=0.06,
        )

        # Right side: abstract parallel diagram
        DIAG_CX = 2.80
        DIAG_Y_TOP = 0.90

        # Input x box
        box_x = Rectangle(
            width=0.60, height=0.48, color=WHITE, stroke_width=1.4,
            fill_color=WHITE, fill_opacity=0.08,
        ).move_to(np.array([DIAG_CX - 2.10, DIAG_Y_TOP - 0.55, 0]))
        lbl_x_box = Text("x", font_size=18, color=WHITE).move_to(box_x)

        # 4 transformed rects with distinct colors
        TRANSFORM_COLORS = [BLUE_D, GREEN_D, ORANGE, PURPLE_A]
        TRANSFORM_Y = [DIAG_Y_TOP, DIAG_Y_TOP - 0.55, DIAG_Y_TOP - 1.10, DIAG_Y_TOP - 1.65]
        transform_boxes = VGroup()
        transform_lbls  = VGroup()
        for i, (col, y) in enumerate(zip(TRANSFORM_COLORS, TRANSFORM_Y)):
            b = Rectangle(
                width=0.52, height=0.40, color=col, stroke_width=1.2,
                fill_color=col, fill_opacity=0.22,
            ).move_to(np.array([DIAG_CX - 0.70, y, 0]))
            l = Text(f"g{i+1}·x", font_size=11, color=col).move_to(b)
            transform_boxes.add(b)
            transform_lbls.add(l)

        arrows_to_trans = VGroup(*[
            Arrow(
                box_x.get_right(), transform_boxes[i].get_left(),
                color=GREY_B, stroke_width=0.9,
                max_tip_length_to_length_ratio=0.20,
            )
            for i in range(4)
        ])

        # 4 f-boxes
        f_boxes = VGroup()
        f_lbls  = VGroup()
        for i, y in enumerate(TRANSFORM_Y):
            fb = Rectangle(
                width=0.44, height=0.36, color=BLUE, stroke_width=1.2,
                fill_color=BLUE, fill_opacity=0.16,
            ).move_to(np.array([DIAG_CX + 0.52, y, 0]))
            fl = Text("f", font_size=14, color=BLUE).move_to(fb)
            f_boxes.add(fb)
            f_lbls.add(fl)

        arrows_to_f = VGroup(*[
            Arrow(
                transform_boxes[i].get_right(), f_boxes[i].get_left(),
                color=GREY_B, stroke_width=0.9,
                max_tip_length_to_length_ratio=0.20,
            )
            for i in range(4)
        ])

        # Average circle
        avg_circle = Circle(
            radius=0.28, color=YELLOW, stroke_width=1.4,
            fill_color=YELLOW, fill_opacity=0.12,
        ).move_to(np.array([DIAG_CX + 1.62, DIAG_Y_TOP - 0.83, 0]))
        avg_lbl = MathTex(r"\Sigma/|G|", font_size=13, color=YELLOW).move_to(avg_circle)  # [m2] inline

        arrows_to_avg = VGroup(*[
            Arrow(
                f_boxes[i].get_right(), avg_circle.get_left(),
                color=GREY_B, stroke_width=0.8,
                max_tip_length_to_length_ratio=0.20,
            )
            for i in range(4)
        ])

        stage_x    = VGroup(box_x, lbl_x_box)
        stage_trans = VGroup(arrows_to_trans, transform_boxes, transform_lbls)
        stage_f     = VGroup(arrows_to_f, f_boxes, f_lbls)
        stage_avg   = VGroup(arrows_to_avg, avg_circle, avg_lbl)

        with self.voiceover(
            text=(
                "To make f invariant, we compute: f-bar of x equals one over the group size, "
                "times the sum over all group elements g of f applied to g-inverse dot x."
            )
        ) as tracker:
            self.play(Write(inv_formula),          run_time=tracker.duration * 0.40)
            self.play(
                FadeIn(generic_nn_lbl),
                Create(generic_nn_box),
                run_time=tracker.duration * 0.15,
            )
            self.play(FadeIn(stage_x),             run_time=tracker.duration * 0.10)
            self.play(FadeIn(stage_trans),         run_time=tracker.duration * 0.15)
            self.play(FadeIn(stage_f),             run_time=tracker.duration * 0.10)
            self.play(FadeIn(stage_avg),           run_time=tracker.duration * 0.10)

        # Beat 2b -- Explain "Generic NN" + averaging
        with self.voiceover(
            text=(
                "Here, f can be any generic neural network "
                "-- it does not need to be equivariant itself. "
                "The averaging does all the symmetrization work."
            )
        ) as tracker:
            self.play(
                Indicate(generic_nn_lbl, color=BLUE, scale_factor=1.12),
                run_time=tracker.duration * 0.50,
            )
            self.wait(tracker.duration * 0.50)

        # Beat 3 -- Equivariant formula
        equiv_formula = MathTex(
            r"\bar{f}(x) = \frac{1}{|G|} \sum_{g \in G} g \cdot f(g^{-1} \cdot x)",
            font_size=30, color=GREEN,
        ).next_to(inv_formula, DOWN, buff=0.55)

        # Keep SurroundingRectangle + side label only — no arrow.
        g_dot_lbl = Text("+ g·  before avg", font_size=13, color=GREEN, weight=BOLD).next_to(
            equiv_formula, RIGHT, buff=0.20
        )
        equiv_rect = SurroundingRectangle(
            equiv_formula, color=GREEN, stroke_width=1.4,
            corner_radius=0.08, buff=0.10,
        )

        with self.voiceover(
            text=(
                "For equivariance, we add one more step: "
                "after applying f, we also apply g to the output before averaging. "
                "This ensures the averaged function transforms correctly "
                "when the input transforms -- equivariance is preserved by construction."
            )
        ) as tracker:
            self.play(Write(equiv_formula),        run_time=tracker.duration * 0.45)
            self.play(Create(equiv_rect),          run_time=tracker.duration * 0.20)
            self.play(FadeIn(g_dot_lbl),           run_time=tracker.duration * 0.20)
            self.wait(tracker.duration * 0.15)

        # Beat 4 -- Trade-off takeaway
        tkw_line1 = Text("Simple and expressive",             font_size=20, color=BLUE,  weight=BOLD)
        tkw_line2 = Text("Challenging if group is large / infinite", font_size=18, color=RED)
        tkw_line3 = Text("-- approximations",                 font_size=16, color=GREY_B, slant=ITALIC)
        tkw_block = VGroup(tkw_line1, tkw_line2, tkw_line3).arrange(
            DOWN, buff=0.22,
        ).move_to(np.array([0.0, -2.5, 0]))

        with self.voiceover(
            text=(
                "Group averaging is simple and expressive -- "
                "Yarotsky 2021 shows it universally approximates "
                "any invariant or equivariant function. "
                "The catch: summing over all G is expensive. "
                "For large or infinite groups, we must resort to approximations."
            )
        ) as tracker:
            self.play(
                FadeIn(tkw_block),
                run_time=tracker.duration * 0.30,
            )
            self.play(
                Indicate(inv_formula, color=YELLOW, scale_factor=1.03),
                run_time=tracker.duration * 0.17,
            )
            self.play(
                Indicate(equiv_formula, color=YELLOW, scale_factor=1.03),
                run_time=tracker.duration * 0.17,
            )
            self.wait(tracker.duration * 0.36)

        # --- End Scene ---
        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.6)
        self.wait(0.3)


# Scene24_JanossyPooling
class Scene24_JanossyPooling(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title = Text(
            "Group Averaging -- Example 1: Janossy Pooling",
            font_size=31, color=WHITE,
        ).to_edge(UP, buff=0.35)
        self.play(Write(title), run_time=0.65)

        # Recap block — centered below title, no border (fidelity: Generic NN)
        recap_formula = MathTex(
            r"\bar{f}(x) = \frac{1}{|G|} \sum_{g \in G} ", r"f", r"(g^{-1} \cdot x)",
            font_size=20, color=GREY_B,
        )
        recap_formula[1].set_color(BLUE)
        recap_generic = Text("Generic NN", font_size=15, color=BLUE, weight=BOLD)
        recap_block = VGroup(recap_formula, recap_generic).arrange(DOWN, buff=0.14)
        recap_block.next_to(title, DOWN, buff=0.28)

        # Beat 1 -- Transition + context
        g_label_math = MathTex(r"G = S_k", font_size=22, color=BLUE)
        g_label_text = Text("-- all permutations of k elements", font_size=17, color=BLUE)
        g_label = VGroup(g_label_math, g_label_text).arrange(RIGHT, buff=0.14)
        g_label.move_to(np.array([0.00, -0.30, 0]))

        with self.voiceover(
            text=(
                "Let us look at a concrete example. "
                "Consider the symmetric group S-k -- the group of all permutations of k elements. "
                "This has k-factorial elements, which can be large, "
                "but the averaging formula is well-defined."
            )
        ) as tracker:
            self.play(
                FadeIn(recap_block),
                run_time=tracker.duration * 0.30,
            )
            self.play(FadeIn(g_label),             run_time=tracker.duration * 0.40)
            self.wait(tracker.duration * 0.30)

        # Beat 2 -- Janossy formula
        janossy_formula = MathTex(
            r"\bar{f}(x_1,\ldots,x_k) = \frac{1}{|G|} \sum_{\sigma \in S_k} "
            r"f\!\left(x_{\sigma(1)}, \ldots, x_{\sigma(k)}\right)",
            font_size=26, color=WHITE,
        ).move_to(np.array([0.00, 2.00, 0]))

        self.play(
            FadeOut(recap_block), FadeOut(g_label),
            run_time=0.35,
        )

        with self.voiceover(
            text=(
                "Janossy pooling instantiates this for sequences. "
                "Given inputs x-one through x-k, we enumerate all k-factorial orderings sigma in S-k. "
                "For each ordering, we pass the reordered sequence to f, then average the results. "
                "This is the Janossy pooling formula -- "
                "the result is invariant to the input order."
            )
        ) as tracker:
            self.play(Write(janossy_formula),      run_time=tracker.duration * 0.55)
            self.play(
                Indicate(janossy_formula, color=YELLOW, scale_factor=1.04),
                run_time=tracker.duration * 0.25,
            )
            self.wait(tracker.duration * 0.20)

        # Beat 3a -- k=3 enumerate orderings
        s3_label = Text(
            "S3:  3! = 6 orderings",
            font_size=17, color=TEAL, weight=BOLD,
        ).move_to(np.array([-3.20, 2.80, 0]))  # above the grid, clear of title

        PERMS = [
            ["a", "b", "c"],
            ["a", "c", "b"],
            ["b", "a", "c"],
            ["b", "c", "a"],
            ["c", "a", "b"],
            ["c", "b", "a"],
        ]
        GRID_CX   = -3.20   # center x for each perm group
        GRID_Y0   =  1.50   # y of first perm group (top)
        ROW_BUFF  =  0.46

        perm_groups = VGroup()
        for idx, perm in enumerate(PERMS):
            cy = GRID_Y0 - idx * ROW_BUFF
            cells_g = VGroup()
            lbls_g  = VGroup()
            for letter in perm:
                rect = Rectangle(
                    width=0.40, height=0.34,
                    color=GREY_B, stroke_width=1.0,
                    fill_color=BLACK, fill_opacity=0.0,
                )
                lbl = Text(letter, font_size=15, color=WHITE).move_to(rect)
                cells_g.add(rect)
                lbls_g.add(lbl)
            cells_g.arrange(RIGHT, buff=0.0)
            for i, l in enumerate(lbls_g):
                l.move_to(cells_g[i])
            grp = VGroup(cells_g, lbls_g)
            grp.move_to(np.array([GRID_CX, cy, 0]))
            perm_groups.add(grp)

        with self.voiceover(
            text=(
                "For k equals three, S-3 has six elements -- six permutations. "
                "Suppose our input is a, b, c. "
                "We enumerate all six orderings: "
                "a-b-c, a-c-b, b-a-c, b-c-a, c-a-b, c-b-a."
            )
        ) as tracker:
            self.play(FadeIn(s3_label),            run_time=tracker.duration * 0.15)
            self.play(
                LaggedStart(
                    *[FadeIn(pg) for pg in perm_groups],
                    lag_ratio=0.19,
                ),
                run_time=tracker.duration * 0.85,
            )

        # Beat 3b -- Apply f + conclude invariance
        F_BOX_CX = GRID_CX + 1.22   # x center of f-boxes

        f_small_boxes = VGroup()
        f_small_lbls  = VGroup()
        arrows_to_fbox = VGroup()

        for idx, pg in enumerate(perm_groups):
            fy = pg.get_center()[1]
            fb = Rectangle(
                width=0.34, height=0.30, color=BLUE, stroke_width=1.0,
                fill_color=BLUE, fill_opacity=0.14,
            ).move_to(np.array([F_BOX_CX, fy, 0]))
            fl = Text("f", font_size=13, color=BLUE).move_to(fb)
            arr = Arrow(
                pg.get_right(), fb.get_left(),
                color=GREY_B, stroke_width=0.8,
                max_tip_length_to_length_ratio=0.22,
            )
            f_small_boxes.add(fb)
            f_small_lbls.add(fl)
            arrows_to_fbox.add(arr)

        # Sum circle: centered vertically in the grid, to the right of f-boxes
        grid_mid_y = (GRID_Y0 + GRID_Y0 - 5 * ROW_BUFF) / 2.0
        sum_circle = Circle(
            radius=0.36, color=YELLOW, stroke_width=1.4,
            fill_color=YELLOW, fill_opacity=0.14,
        ).move_to(np.array([F_BOX_CX + 1.30, grid_mid_y, 0]))
        sum_lbl = MathTex(r"\Sigma/6", font_size=16, color=YELLOW).move_to(sum_circle)

        # Arrows from each f-box to sum_circle, endpoints dispersed along left arc
        arrows_to_sum = VGroup()
        _cy = sum_circle.get_center()[1]
        _cr = sum_circle.radius
        for i in range(6):
            dy = (f_small_boxes[i].get_center()[1] - _cy) * 0.55
            dy = float(np.clip(dy, -_cr * 0.82, _cr * 0.82))
            dx = -float(np.sqrt(max(_cr**2 - dy**2, 0.001)))
            endpoint = sum_circle.get_center() + np.array([dx, dy, 0])
            arr = Arrow(
                f_small_boxes[i].get_right(), endpoint,
                color=GREY_B, stroke_width=0.7,
                max_tip_length_to_length_ratio=0.22,
            )
            arrows_to_sum.add(arr)

        # f-bar output label
        fbar_lbl = MathTex(r"\bar{f}", font_size=24, color=GREEN).next_to(
            sum_circle, RIGHT, buff=0.28
        )
        arr_to_fbar = Arrow(
            sum_circle.get_right(), fbar_lbl.get_left(),
            color=GREEN, stroke_width=1.2,
            max_tip_length_to_length_ratio=0.22,
        )

        with self.voiceover(
            text=(
                "We apply f to each ordering, sum the results, and divide by six. "
                "The output is the same regardless of which order "
                "we originally presented a, b, and c."
            )
        ) as tracker:
            self.play(
                FadeIn(f_small_boxes), FadeIn(f_small_lbls),
                FadeIn(arrows_to_fbox),
                run_time=tracker.duration * 0.35,
            )
            self.play(
                FadeIn(sum_circle), FadeIn(sum_lbl),
                FadeIn(arrows_to_sum),
                run_time=tracker.duration * 0.30,
            )
            self.play(
                GrowArrow(arr_to_fbar), FadeIn(fbar_lbl),
                run_time=tracker.duration * 0.20,
            )
            self.play(
                Indicate(fbar_lbl, color=GREEN, scale_factor=1.12),
                run_time=tracker.duration * 0.15,
            )

        # Beat 4 -- Relational pooling + cost note
        self.play(
            FadeOut(perm_groups), FadeOut(s3_label),
            FadeOut(f_small_boxes), FadeOut(f_small_lbls), FadeOut(arrows_to_fbox),
            FadeOut(arrows_to_sum), FadeOut(sum_circle), FadeOut(sum_lbl),
            FadeOut(arr_to_fbar), FadeOut(fbar_lbl),
            FadeOut(janossy_formula),
            run_time=0.40,
        )

        bullet1_main = Text("Janossy Pooling",                            font_size=18, color=BLUE,  weight=BOLD)
        bullet1_cite = Text("  (Murphy et al. 2019)",                     font_size=14, color=GREY_B)
        bullet1_row  = VGroup(bullet1_main, bullet1_cite).arrange(RIGHT, buff=0.05)

        bullet2_main = Text("Relational Pooling -- more expressive graph NNs", font_size=17, color=GREEN)
        bullet2_cite = Text("  (Murphy et al. 2019)",                     font_size=13, color=GREY_B)
        bullet2_row  = VGroup(bullet2_main, bullet2_cite).arrange(RIGHT, buff=0.05)

        bullets = VGroup(bullet1_row, bullet2_row).arrange(
            DOWN, aligned_edge=LEFT, buff=0.30,
        ).move_to(np.array([0.00, 0.20, 0]))

        bullet2_rect = SurroundingRectangle(
            bullet2_row, color=GREEN, stroke_width=1.4,
            corner_radius=0.07, buff=0.10,
        )

        cost_note = Text(
            "k! grows fast -- random subset sampling in practice",
            font_size=13, color=GREY_B, slant=ITALIC,
        ).to_edge(DOWN, buff=0.38)

        with self.voiceover(
            text=(
                "Beyond sequences, this applies to graph neural networks. "
                "Relational pooling applies Janossy pooling over node orderings, "
                "making message-passing networks more expressive. "
                "For large k, k-factorial grows fast "
                "-- in practice, we sample a random subset as an approximation."
            )
        ) as tracker:
            self.play(FadeIn(bullets),             run_time=tracker.duration * 0.30)
            self.play(Create(bullet2_rect),        run_time=tracker.duration * 0.25)
            self.play(
                Indicate(bullet2_main, color=GREEN, scale_factor=1.06),
                run_time=tracker.duration * 0.20,
            )
            self.play(FadeIn(cost_note),           run_time=tracker.duration * 0.15)
            self.wait(tracker.duration * 0.10)

        self.wait(1.5)

        # --- End Scene ---
        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.6)
        self.wait(0.3)



In [ ]:
%manim -qh Scene23_GroupAveraging

In [ ]:
%manim -qh Scene24_JanossyPooling

# SLIDE 25

In [ ]:
# Scene25_SignNet
class Scene25_SignNet(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title = Text(
            "Group Averaging -- Example 2: SignNet",
            font_size=32, color=WHITE,
        ).to_edge(UP, buff=0.38)
        self.play(Write(title), run_time=0.65)

        # Beat 1 -- Sign ambiguity + f(vi)=f(-vi) design goal
        diagram_origin = np.array([-4.20, 0.20, 0])
        arrow_pos = Arrow(
            start=diagram_origin,
            end=diagram_origin + np.array([1.10, 1.60, 0]),
            color=WHITE, stroke_width=2.8, buff=0,
            max_tip_length_to_length_ratio=0.18,
        )
        arrow_neg = Arrow(
            start=diagram_origin,
            end=diagram_origin + np.array([-1.10, -1.60, 0]),
            color=YELLOW, stroke_width=2.8, buff=0,
            max_tip_length_to_length_ratio=0.18,
        )
        lbl_vi     = MathTex(r"v_i",  font_size=20, color=WHITE).next_to(
            arrow_pos.get_end(), UR, buff=0.10)
        lbl_neg_vi = MathTex(r"-v_i", font_size=20, color=YELLOW).next_to(
            arrow_neg.get_end(), DL, buff=0.10)
        lbl_both   = Text(
            "both valid eigenvectors", font_size=12, color=GREY_B, slant=ITALIC,
        ).next_to(arrow_neg.get_end(), DOWN, buff=0.22)
        diagram_group = VGroup(arrow_pos, arrow_neg, lbl_vi, lbl_neg_vi, lbl_both)

        # Right side: eigenvalue equations
        eq1 = MathTex(r"Av_i = \lambda_i v_i",       font_size=22, color=WHITE)
        eq2 = MathTex(r"A(-v_i) = \lambda_i (-v_i)", font_size=22, color=YELLOW)
        eq_group = VGroup(eq1, eq2).arrange(DOWN, buff=0.32, aligned_edge=LEFT)
        eq_group.move_to(np.array([0.80, 0.40, 0]))

        # Design goal: f(vi) = f(-vi)
        design_goal = MathTex(r"f(v_i) = f(-v_i)", font_size=28, color=YELLOW)
        design_goal.next_to(eq_group, DOWN, buff=0.50)
        want_lbl = Text("want:", font_size=15, color=YELLOW).next_to(
            design_goal, LEFT, buff=0.18)
        goal_rect = SurroundingRectangle(
            design_goal, color=YELLOW, stroke_width=1.6,
            corner_radius=0.08, buff=0.12,
        )

        with self.voiceover(
            text=(
                "Our second example is SignNet. "
                "Eigenvectors have a fundamental sign ambiguity: "
                "if v-i is an eigenvector, then negative v-i is equally valid. "
                "We want any function of eigenvectors to satisfy: "
                "f of v-i equals f of negative v-i."
            )
        ) as tracker:
            self.play(FadeIn(diagram_group),          run_time=tracker.duration * 0.20)
            self.play(FadeIn(eq1),                    run_time=tracker.duration * 0.25)
            self.play(Write(eq2),                     run_time=tracker.duration * 0.30)
            self.play(
                Write(design_goal), FadeIn(want_lbl),
                Create(goal_rect),
                run_time=tracker.duration * 0.25,
            )
        # Beat 2 -- Actual SignNet formula h([phi(vi)+phi(-vi)])
        PHI_PART = r"[\varphi(v_i) + \varphi(-v_i)]_{i=1}^{k}"
        signnet_formula = MathTex(
            r"f(v) = h\!\left(",
            PHI_PART,
            r"\right)",
            font_size=27, color=WHITE,
            substrings_to_isolate=[PHI_PART],
        ).move_to(np.array([0.20, 0.60, 0]))
        # Shorthand references to sub-parts
        formula_prefix = signnet_formula[0]
        formula_phi    = signnet_formula.get_part_by_tex(PHI_PART)
        formula_suffix = signnet_formula[2]

        # SurroundingRectangle on phi-sum part via get_part_by_tex
        phi_part_rect = SurroundingRectangle(
            formula_phi,
            color=BLUE, stroke_width=1.4, corner_radius=0.06, buff=0.08,
        )

        # Annotation labels (positions resolved inside tracker after Write)
        learned_phi = Text("learned  phi", font_size=13, color=BLUE, weight=BOLD)
        learned_h_lbl = Text("learned  h",  font_size=13, color=GREEN, weight=BOLD)

        # Context group avg formula -- small, top-right corner
        context_formula = MathTex(
            r"\bar{f} = \tfrac{1}{|G|}\textstyle\sum_{s\in\{\pm1\}^n} f(sv)",
            font_size=13, color=GREY_B,
        )
        context_note = Text(
            "(group avg. form)", font_size=11, color=GREY_B, slant=ITALIC,
        )
        context_block = VGroup(context_formula, context_note).arrange(DOWN, buff=0.06)
        context_block.to_corner(UR, buff=0.28)

        with self.voiceover(
            text=(
                "SignNet achieves this elegantly: "
                "apply a shared learned network phi to v-i and to negative v-i, "
                "then add. "
                "Feed all these phi-sums into a final learned network h -- "
                "and you have a sign-invariant function."
            )
        ) as tracker:
            # FadeOut Beat 1 visual -- keep design_goal/goal_rect/want_lbl for now
            self.play(
                FadeOut(eq1), FadeOut(eq2), FadeOut(diagram_group),
                run_time=tracker.duration * 0.10,
            )
            self.play(
                Write(signnet_formula),
                run_time=tracker.duration * 0.35,
            )
            # Resolve positions after formula is drawn
            learned_phi.next_to(signnet_formula, DOWN, buff=0.55).shift(LEFT * 0.5)
            learned_h_lbl.next_to(signnet_formula, DOWN, buff=0.55).shift(RIGHT * 1.6)
            arrow_phi = Arrow(
                learned_phi.get_top(),
                formula_phi.get_bottom() + np.array([-0.10, 0, 0]),
                color=BLUE, stroke_width=1.0, buff=0.06,
                max_tip_length_to_length_ratio=0.22,
            )
            arrow_h = Arrow(
                learned_h_lbl.get_top(),
                formula_prefix.get_bottom() + np.array([0.50, 0, 0]),
                color=GREEN, stroke_width=1.0, buff=0.06,
                tip_length=0.13,
                max_tip_length_to_length_ratio=0.22,
            )
            self.play(
                FadeIn(learned_phi), GrowArrow(arrow_phi),
                FadeIn(learned_h_lbl), GrowArrow(arrow_h),
                FadeIn(context_block),
                run_time=tracker.duration * 0.30,
            )
            self.play(Create(phi_part_rect),   run_time=tracker.duration * 0.15)
            self.play(
                Flash(signnet_formula.get_center(), color=YELLOW, flash_radius=1.20,
                      line_length=0.22, num_lines=12),
                run_time=tracker.duration * 0.10,
            )
        # Beat 3 -- Pipeline diagram
        BOX_W, BOX_H = 1.70, 0.72

        def make_pipeline_box(lines, box_color):
            rect = RoundedRectangle(
                width=BOX_W, height=BOX_H,
                corner_radius=0.12,
                color=box_color, stroke_width=1.6,
                fill_color=BLACK, fill_opacity=0.0,
            )
            label_objs = [
                Text(line, font_size=12, color=box_color)
                for line in lines
            ]
            lbl_group = VGroup(*label_objs).arrange(DOWN, buff=0.06)
            lbl_group.move_to(rect)
            return VGroup(rect, lbl_group)

        box1 = make_pipeline_box(["Graph", "A / X"], GREY_B)
        box2 = make_pipeline_box(["Laplacian", "Eigenvectors"], TEAL)
        box4 = make_pipeline_box(["Prediction", "GNN / Transf."], GREEN)
        box3_rect = RoundedRectangle(
            width=BOX_W, height=BOX_H,
            corner_radius=0.12,
            color=BLUE, stroke_width=1.6,
            fill_color=BLACK, fill_opacity=0.0,
        )
        box3_formula = MathTex(r"f(v_1,\ldots,v_k)", font_size=12, color=BLUE)
        box3_sub     = Text("SignNet", font_size=12, color=BLUE)
        box3_lbls    = VGroup(box3_formula, box3_sub).arrange(DOWN, buff=0.06)
        box3_lbls.move_to(box3_rect)
        box3 = VGroup(box3_rect, box3_lbls)

        pipeline_boxes = VGroup(box1, box2, box3, box4).arrange(RIGHT, buff=0.50)
        pipeline_boxes.move_to(np.array([0.0, -0.30, 0]))

        def pipe_arrow(src_box, tgt_box):
            return Arrow(
                src_box.get_right(), tgt_box.get_left(),
                color=GREY_B, stroke_width=1.2, buff=0.06,
                max_tip_length_to_length_ratio=0.20,
            )

        arr12 = pipe_arrow(box1, box2)
        arr23 = pipe_arrow(box2, box3)
        arr34 = pipe_arrow(box3, box4)

        with self.voiceover(
            text=(
                "In practice, a graph comes with adjacency, node features, "
                "and Laplacian eigenvectors. "
                "SignNet processes the eigenvectors with the phi-sum construction, "
                "then feeds the result into any prediction model -- "
                "a GNN or Transformer -- for downstream tasks."
            )
        ) as tracker:
            self.play(
                FadeOut(signnet_formula), FadeOut(phi_part_rect),
                FadeOut(learned_phi), FadeOut(arrow_phi),
                FadeOut(learned_h_lbl), FadeOut(arrow_h),
                FadeOut(context_block),
                FadeOut(goal_rect), FadeOut(want_lbl), FadeOut(design_goal),
                run_time=tracker.duration * 0.12,
            )
            self.play(
                LaggedStart(
                    FadeIn(box1),
                    GrowArrow(arr12), FadeIn(box2),
                    GrowArrow(arr23), FadeIn(box3),
                    GrowArrow(arr34), FadeIn(box4),
                    lag_ratio=0.25,
                ),
                run_time=tracker.duration * 0.65,
            )
            self.play(
                Flash(box3.get_center(), color=BLUE, flash_radius=0.80,
                      line_length=0.18, num_lines=10),
                run_time=tracker.duration * 0.23,
            )
        # Beat 4 -- Manifold placeholder + applications
        manifold_ellipse = Ellipse(
            width=2.20, height=1.40,
            color=TEAL, stroke_width=1.6,
            fill_color=TEAL, fill_opacity=0.06,
        ).move_to(np.array([-4.00, 0.10, 0]))
        manifold_dash = DashedLine(
            manifold_ellipse.get_top(),
            manifold_ellipse.get_bottom(),
            color=GREY_B, stroke_width=1.2, dash_length=0.10,
        )
        eigvec_lbl = MathTex(
            r"\text{Eigvec }11", font_size=12, color=TEAL,
        ).next_to(manifold_ellipse, UP, buff=0.12)
        lbl_left = MathTex(
            r"\varphi(v_{11})", font_size=12, color=WHITE,
        ).next_to(manifold_ellipse, LEFT, buff=0.14)
        lbl_right = MathTex(
            r"\varphi(-v_{11})", font_size=12, color=YELLOW,
        ).next_to(manifold_ellipse, RIGHT, buff=0.14)
        sign_note = Text(
            "+-sign flip on manifold", font_size=11, color=GREY_B, slant=ITALIC,
        ).next_to(manifold_ellipse, DOWN, buff=0.14)
        manifold_group = VGroup(
            manifold_ellipse, manifold_dash,
            eigvec_lbl, lbl_left, lbl_right, sign_note,
        )

        # Application block -- right side
        app_line1 = Text("Graph positional encodings", font_size=17, color=TEAL, weight=BOLD)
        app_line2 = Text("Neural fields on manifolds",  font_size=16, color=GREEN)
        app_cite  = Text("(Lim, Robinson et al., 2023)", font_size=13, color=GREY_B, slant=ITALIC)
        app_block = VGroup(app_line1, app_line2, app_cite).arrange(
            DOWN, aligned_edge=LEFT, buff=0.20,
        ).move_to(np.array([1.40, 0.10, 0]))

        with self.voiceover(
            text=(
                "SignNet extends beyond graphs to neural fields on manifolds -- "
                "where eigenvectors of the Laplace-Beltrami operator define shape features. "
                "The sign-invariant processing makes representations consistent "
                "across arbitrary eigendecompositions."
            )
        ) as tracker:
            self.play(
                FadeOut(box1), FadeOut(box2), FadeOut(box3), FadeOut(box4),
                FadeOut(arr12), FadeOut(arr23), FadeOut(arr34),
                run_time=tracker.duration * 0.10,
            )
            self.play(
                LaggedStart(
                    FadeIn(manifold_group),
                    FadeIn(app_block),
                    lag_ratio=0.45,
                ),
                run_time=tracker.duration * 0.68,
            )
            self.wait(tracker.duration * 0.22)

        self.wait(0.6)

        # --- End Scene ---
        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.6)
        self.wait(0.3)


In [ ]:
%manim -qh Scene25_SignNet

# SLIDE 26 TỚI SLIDE 27

In [ ]:
# Slide 26: Frame Averaging — the middle ground
class Scene26_FrameAveraging(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title = Text(
            "Frame Averaging",
            font_size=32, color=WHITE,
        ).to_edge(UP, buff=0.38)

        # Beat 1 — Spectrum

        spectrum_line = Line(
            start=np.array([-3.5, 0.20, 0]),
            end=np.array([ 3.5, 0.20, 0]),
            color=GREY_B, stroke_width=1.4,
        )

        # Three anchor points on the line
        pos_left  = np.array([-3.5, 0.20, 0])
        pos_mid   = np.array([ 0.0, 0.20, 0])
        pos_right = np.array([ 3.5, 0.20, 0])

        dot_left  = Dot(pos_left,  radius=0.07, color=RED)
        dot_mid   = Dot(pos_mid,   radius=0.10, color=YELLOW)   # bigger — emphasis
        dot_right = Dot(pos_right, radius=0.07, color=TEAL)

        lbl_left  = Text("Canonicalization", font_size=13, color=RED).next_to(
            dot_left,  UP, buff=0.14)
        lbl_mid   = Text("Frame Averaging",  font_size=13, color=YELLOW).next_to(
            dot_mid,   UP, buff=0.14)
        lbl_right = Text("Group Averaging",  font_size=13, color=TEAL).next_to(
            dot_right, UP, buff=0.14)

        sub_left  = Text("single canonical", font_size=16, color=WHITE).next_to(
            dot_left,  DOWN, buff=0.14)
        sub_mid   = MathTex(r"F(x) \subseteq G", font_size=16, color=WHITE).next_to(
            dot_mid,   DOWN, buff=0.14)
        sub_right = MathTex(r"|G|",              font_size=16, color=WHITE).next_to(
            dot_right, DOWN, buff=0.14)

        spectrum_group = VGroup(
            spectrum_line,
            dot_left, dot_mid, dot_right,
            lbl_left, lbl_mid, lbl_right,
            sub_left, sub_mid, sub_right,
        )

        with self.voiceover(
            text=(
                "We've seen two extremes: canonicalization picks a single representative, "
                "while group averaging sums over all of G. "
                "Frame averaging lives in between — "
                "we average only over a carefully chosen subset F of x inside G."
            )
        ) as tracker:
            self.play(Write(title), run_time=tracker.duration * 0.15)
            self.play(
                LaggedStart(
                    Create(spectrum_line),
                    FadeIn(dot_left),  FadeIn(lbl_left),  FadeIn(sub_left),
                    FadeIn(dot_mid),   FadeIn(lbl_mid),   FadeIn(sub_mid),
                    FadeIn(dot_right), FadeIn(lbl_right), FadeIn(sub_right),
                    lag_ratio=0.18,
                ),
                run_time=tracker.duration * 0.55,
            )
            self.play(
                Flash(dot_mid.get_center(), color=YELLOW,
                      flash_radius=0.40, line_length=0.14, num_lines=8),
                run_time=tracker.duration * 0.20,
            )
            self.wait(tracker.duration * 0.15)

        # Beat 2 — Invariant + Equivariant formulas
        inv_label = Text("Invariant:", font_size=15, color=GREY_B)
        inv_formula = MathTex(
            r"\bar{f}(x) = \frac{1}{|F(x)|} \sum_{\tau \in F(x)} f(\tau^{-1} \cdot x)",
            font_size=22, color=WHITE,
        )
        inv_pair = VGroup(inv_label, inv_formula).arrange(RIGHT, buff=0.20)

        eqv_label = Text("Equivariant:", font_size=15, color=GREY_B)
        eqv_formula = MathTex(
            r"\bar{f}(x) = \frac{1}{|F(x)|} \sum_{\tau \in F(x)} \tau \cdot f(\tau^{-1} \cdot x)",
            font_size=22, color=WHITE,
        )
        eqv_pair = VGroup(eqv_label, eqv_formula).arrange(RIGHT, buff=0.20)

        formulas_group = VGroup(inv_pair, eqv_pair).arrange(
            DOWN, buff=0.50, aligned_edge=LEFT,
        ).move_to(np.array([0.30, 0.10, 0]))

        # isolate via substrings_to_isolate won't work post-hoc; use bounding approach
        eqv_diff_rect = SurroundingRectangle(
            eqv_formula, color=GREEN, stroke_width=1.4,
            corner_radius=0.06, buff=0.08,
        )
        diff_label = Text(
            "+ outer tau action", font_size=14, color=GREEN, slant=ITALIC,
        ).next_to(eqv_diff_rect, DOWN, buff=0.10)

        with self.voiceover(
            text=(
                "Concretely: for invariance, we average f of tau-inverse times x "
                "over all tau in the frame F of x. "
                "For equivariance, we apply tau to the output before averaging. "
                "Both mirror group averaging, but sum only over F of x -- far smaller than G."
            )
        ) as tracker:
            self.play(
                FadeOut(spectrum_group),
                run_time=tracker.duration * 0.10,
            )
            self.play(Write(inv_formula), FadeIn(inv_label), run_time=tracker.duration * 0.30)
            self.play(Write(eqv_formula), FadeIn(eqv_label), run_time=tracker.duration * 0.30)
            self.play(
                Create(eqv_diff_rect), FadeIn(diff_label),
                run_time=tracker.duration * 0.20,
            )
            self.wait(tracker.duration * 0.10)

        # Beat 3 — Frame equivariance condition F(g·x) = g·F(x)
        condition = MathTex(
            r"F(g \cdot x) = g \cdot F(x)",
            font_size=32, color=YELLOW,
        ).move_to(np.array([0.0, 0.55, 0]))
        condition_label = Text(
            "frame equivariance condition",
            font_size=14, color=GREY_B, slant=ITALIC,
        ).next_to(condition, DOWN, buff=0.22)
        condition_rect = SurroundingRectangle(
            condition, color=YELLOW, stroke_width=1.8,
            corner_radius=0.10, buff=0.14,
        )

        # Commutative diagram:
        cd_x    = MathTex(r"x",           font_size=14, color=GREY_B)
        cd_gx   = MathTex(r"g \cdot x",   font_size=14, color=GREY_B)
        cd_Fx   = MathTex(r"F(x)",        font_size=14, color=GREY_B)
        cd_gFx  = MathTex(r"g \cdot F(x)", font_size=14, color=GREY_B)

        cd_x.move_to(  np.array([-1.20, -1.00, 0]))
        cd_gx.move_to( np.array([ 1.20, -1.00, 0]))
        cd_Fx.move_to( np.array([-1.20, -1.85, 0]))
        cd_gFx.move_to(np.array([ 1.20, -1.85, 0]))

        arr_top    = Arrow(cd_x.get_right(),   cd_gx.get_left(),
                           color=GREY_B, stroke_width=1.0, buff=0.06,
                           max_tip_length_to_length_ratio=0.20)
        arr_left   = Arrow(cd_x.get_bottom(),  cd_Fx.get_top(),
                           color=GREY_B, stroke_width=1.0, buff=0.06,
                           max_tip_length_to_length_ratio=0.20)
        arr_right  = Arrow(cd_gx.get_bottom(), cd_gFx.get_top(),
                           color=GREY_B, stroke_width=1.0, buff=0.06,
                           max_tip_length_to_length_ratio=0.20)
        arr_bottom = Arrow(cd_Fx.get_right(),  cd_gFx.get_left(),
                           color=GREY_B, stroke_width=1.0, buff=0.06,
                           max_tip_length_to_length_ratio=0.20)
        lbl_top    = MathTex(r"g\cdot", font_size=14, color=GREY_B).next_to(
            arr_top, UP, buff=0.05)
        lbl_bot    = MathTex(r"g\cdot", font_size=14, color=GREY_B).next_to(
            arr_bottom, DOWN, buff=0.05)
        commute_diagram = VGroup(
            cd_x, cd_gx, cd_Fx, cd_gFx,
            arr_top, arr_left, arr_right, arr_bottom,
            lbl_top, lbl_bot,
        )

        citation_b3 = Text(
            "(Puny et al. 2022; Atzmon et al. 2022)",
            font_size=12, color=GREY_B, slant=ITALIC,
        ).to_corner(DR, buff=0.30)

        with self.voiceover(
            text=(
                "The key requirement is that the frame itself must be equivariant: "
                "if you transform the input x by g, the frame transforms by the same g. "
                "Without this, the average would not be invariant or equivariant — "
                "it would just be an arbitrary subset. "
                "This diagram shows: transforming first then computing the frame "
                "gives the same result as computing the frame first then transforming."
            )
        ) as tracker:
            self.play(
                FadeOut(inv_pair), FadeOut(eqv_pair),
                FadeOut(eqv_diff_rect), FadeOut(diff_label),
                run_time=tracker.duration * 0.10,
            )
            self.play(
                Write(condition), FadeIn(condition_label),
                run_time=tracker.duration * 0.30,
            )
            self.play(Create(condition_rect), run_time=tracker.duration * 0.18)
            self.play(FadeIn(commute_diagram),  run_time=tracker.duration * 0.25)
            self.play(FadeIn(citation_b3),       run_time=tracker.duration * 0.17)

        # Beat 4 — Trade-off table: Cost / Continuity
        COL_X = [-2.50, -0.40, 1.05]

        def make_row(cells, colors, bold=False):
           row = VGroup()
           for text, col, cx in zip(cells, colors, COL_X):
                w = BOLD if bold else NORMAL
                t = Text(text, font_size=13, color=col, weight=w)
                t.move_to(np.array([cx, 0, 0]), aligned_edge=LEFT)
                row.add(t)
           return row

        header = make_row(
            ["Method", "Cost", "Continuity"],
            [GREY_B, GREY_B, GREY_B], bold=True,
        )
        row1 = make_row(
            ["Canonicalization", "Cheap", "Not continuous"],
            [RED, GREEN, RED],
        )
        row2 = make_row(
            ["Frame Averaging",  "Medium", "Continuity: partial"],
            [YELLOW, YELLOW, YELLOW], bold=True,
        )
        row3 = make_row(
            ["Group Averaging",  "Expensive", "Continuous"],
            [TEAL, RED, GREEN],
        )

        table_group = VGroup(header, row1, row2, row3).arrange(
            DOWN, buff=0.22, aligned_edge=LEFT,
        ).move_to(np.array([0.20, -0.10, 0]))

        fa_highlight_rect = SurroundingRectangle(
            row2, color=YELLOW, stroke_width=1.2, corner_radius=0.06, buff=0.08,
        )

        with self.voiceover(
            text=(
                "Frame averaging hits a sweet spot: cheaper than full group averaging "
                "when the frame is small. "
                "Unlike canonicalization, it can yield a continuous function "
                "when the frame is equivariant — though continuity is not guaranteed in all cases. "
                "Crucially, it preserves the expressiveness of the base model."
            )
        ) as tracker:
            self.play(
                FadeOut(condition), FadeOut(condition_label),
                FadeOut(condition_rect), FadeOut(commute_diagram),
                FadeOut(citation_b3),
                run_time=tracker.duration * 0.10,
            )
            self.play(
                LaggedStart(
                    FadeIn(header),
                    FadeIn(row1),
                    FadeIn(row2),
                    FadeIn(row3),
                    lag_ratio=0.30,
                ),
                run_time=tracker.duration * 0.60,
            )
            self.play(Create(fa_highlight_rect), run_time=tracker.duration * 0.20)
            self.wait(tracker.duration * 0.10)

        self.wait(0.5)
        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.6)
        self.wait(0.3)


# Scene27_PointCloudsProteins
class Scene27_PointCloudsProteins(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title = Text(
            "Example: Point Clouds & Proteins",
            font_size=32, color=WHITE,
        ).to_edge(UP, buff=0.38)

        # Beat 1 — E(3) setting: point cloud placeholder + symmetry group
        dots_positions = [
            np.array([-4.20,  0.30, 0]), np.array([-3.80,  0.60, 0]),
            np.array([-3.50,  0.10, 0]), np.array([-4.00, -0.20, 0]),
            np.array([-3.60, -0.50, 0]), np.array([-4.30, -0.40, 0]),
            np.array([-3.90,  0.50, 0]), np.array([-4.10, -0.10, 0]),
            np.array([-3.70,  0.20, 0]),
        ]
        cloud_dots = VGroup(*[Dot(p, radius=0.06, color=TEAL) for p in dots_positions])
        cloud_label = Text(
            "Point Cloud X", font_size=13, color=GREY_B,
        ).next_to(cloud_dots, DOWN, buff=0.18)

        e3_title = MathTex(r"\mathbf{E(3)}", font_size=26, color=WHITE)
        # MathTex for SO(3) and R^3
        e3_rot  = MathTex(r"\text{Rotations}\;SO(3)",           font_size=14, color=TEAL)
        e3_refl = Text("Reflections",                            font_size=14, color=YELLOW)
        e3_tran = MathTex(r"\text{Translations}\;\mathbb{R}^3",  font_size=14, color=GREEN)
        e3_items = VGroup(e3_rot, e3_refl, e3_tran).arrange(
            DOWN, aligned_edge=LEFT, buff=0.18,
        )
        e3_block = VGroup(e3_title, e3_items).arrange(DOWN, buff=0.28)
        e3_block.move_to(np.array([2.0, 0.10, 0]))

        with self.voiceover(
            text=(
                "Our concrete example is point clouds — like protein structures. "
                "The symmetry group is E-3: rotations, reflections, and translations. "
                "A frame-averaged model must treat all orientations and positions "
                "of the same point cloud identically."
            )
        ) as tracker:
            self.play(Write(title), run_time=tracker.duration * 0.12)
            self.play(FadeIn(cloud_dots), FadeIn(cloud_label), run_time=tracker.duration * 0.20)
            self.play(Write(e3_title), run_time=tracker.duration * 0.15)
            self.play(
                LaggedStart(*[FadeIn(i) for i in e3_items], lag_ratio=0.35),
                run_time=tracker.duration * 0.28,
            )
            self.wait(tracker.duration * 0.25)

        self.wait(0.1)  # M-3: clean gap before Beat 2a FadeOut begins

        # Beat 2a — Covariance to PCA eigenvectors tp frame definition

        cov_formula = MathTex(
            r"\text{Cov}(X) = \tfrac{1}{n} X^{\top} X "
            r"\;\xrightarrow{\text{eigen}}\; v_1, \ldots, v_d",
            font_size=18, color=WHITE,
        ).move_to(np.array([0.60, 0.90, 0]))

        pca_frame = MathTex(
            r"F(X) = \bigl\{\,\bigl([\alpha_1 v_1,\ldots,\alpha_d v_d],\,t\bigr)"
            r"\mid \alpha_i \in \{-1,+1\}\,\bigr\}",
            font_size=18, color=WHITE,
        ).move_to(np.array([0.60, 0.05, 0]))

        # Annotations: 3 Arrow + label
        annot_eigv  = Text("eigenvectors", font_size=12, color=BLUE)
        annot_cent  = Text("centroid",     font_size=12, color=GREEN)
        annot_sign  = Text("sign flips",   font_size=12, color=YELLOW)

        # Positions resolved after pca_frame is placed
        annot_eigv.next_to(pca_frame, DOWN, buff=0.50).shift(LEFT * 1.50)
        annot_cent.next_to(pca_frame, DOWN, buff=0.50)
        annot_sign.next_to(pca_frame, DOWN, buff=0.50).shift(RIGHT * 1.50)

        arr_eigv = Arrow(
            annot_eigv.get_top(),
            pca_frame.get_bottom() + LEFT * 1.0,
            color=BLUE, stroke_width=1.0, buff=0.05,
            max_tip_length_to_length_ratio=0.22,
        )
        arr_cent = Arrow(
            annot_cent.get_top(),
            pca_frame.get_bottom(),
            color=GREEN, stroke_width=1.0, buff=0.05,
            max_tip_length_to_length_ratio=0.22,
        )
        arr_sign = Arrow(
            annot_sign.get_top(),
            pca_frame.get_bottom() + RIGHT * 1.0,
            color=YELLOW, stroke_width=1.0, buff=0.05,
            max_tip_length_to_length_ratio=0.22,
        )
        annot_group = VGroup(
            annot_eigv, arr_eigv,
            annot_cent, arr_cent,
            annot_sign, arr_sign,
        )

        with self.voiceover(
            text=(
                "To build a frame for point clouds, we use PCA: "
                "compute the covariance of X, extract eigenvectors v-one through v-d, "
                "and center at the centroid t."
            )
        ) as tracker:
            self.play(
                FadeOut(e3_block),
                FadeOut(cloud_dots), FadeOut(cloud_label),
                run_time=tracker.duration * 0.15,
            )
            self.play(Write(cov_formula), run_time=tracker.duration * 0.30)
            self.play(Write(pca_frame),   run_time=tracker.duration * 0.35)
            self.play(FadeIn(annot_group), run_time=tracker.duration * 0.20)

        # Beat 2b — 2^d
        frame_size = MathTex(r"|F(X)| = 2^d", font_size=20, color=YELLOW)
        frame_note = Text(
            "(e.g. d=3  →  8 frames, vs infinite E(3))",
            font_size=12, color=GREY_B,
        )
        frame_line = VGroup(frame_size, frame_note).arrange(RIGHT, buff=0.20)
        frame_line.next_to(pca_frame, DOWN, buff=0.80)

        # Concrete point-cloud equivariant formula — boxed
        pc_formula = MathTex(
            r"\frac{1}{|F(X)|} \sum_{(R,\,t)\in F(X)}"
            r"\varphi\!\left(XR - \mathbf{1}t\right) R^{\top} + \mathbf{1}t",
            font_size=16, color=WHITE,
        ).next_to(frame_line, DOWN, buff=0.28)
        pc_rect = SurroundingRectangle(
            pc_formula, color=BLUE, stroke_width=1.6,
            corner_radius=0.08, buff=0.10,
        )
        pc_cite = Text(
            "(Puny et al. 2022)", font_size=11, color=GREY_B, slant=ITALIC,
        ).next_to(pc_rect, RIGHT, buff=0.12)

        with self.voiceover(
            text=(
                "Each alpha-i is plus-one or minus-one, giving 2-to-the-d frame elements. "
                "Frame averaging then applies: "
                "average phi of X rotated and translated, over all frame elements."
            )
        ) as tracker:
            # FadeOut cov_formula before this beat's visuals appear
            self.play(
                FadeOut(cov_formula), FadeOut(annot_group),
                run_time=tracker.duration * 0.05,
            )
            self.play(
                Write(frame_size), FadeIn(frame_note),
                run_time=tracker.duration * 0.20,
            )
            self.play(Write(pc_formula),  run_time=tracker.duration * 0.45)
            self.play(
                Create(pc_rect), FadeIn(pc_cite),
                run_time=tracker.duration * 0.25,
            )
            self.wait(tracker.duration * 0.05)

        # Beat 3 — Sign-flip structure of the frame + cross-reference
        conn_line1 = MathTex(
            r"\alpha_i \in \{-1,+1\} \;\Rightarrow\; 2^d \text{ sign-flip frames}",
            font_size=18, color=YELLOW,
        )
        conn_line2 = Text(
            "(same structure as SignNet sign averaging -- Lim et al. 2023)",
            font_size=12, color=GREY_B, slant=ITALIC,
        )
        conn_group = VGroup(conn_line1, conn_line2).arrange(
            DOWN, buff=0.16,
        ).move_to(np.array([0.60, -0.80, 0]))
        conn_rect = SurroundingRectangle(
            conn_line1, color=YELLOW, stroke_width=1.4,
            corner_radius=0.08, buff=0.12,
        )

        with self.voiceover(
            text=(
                "Each alpha-i being plus-one or minus-one gives exactly 2-to-the-d sign-flip frames. "
                "This sign-flip structure directly mirrors the eigenvector sign ambiguity "
                "handled by SignNet."
            )
        ) as tracker:
            self.play(
                FadeOut(frame_line), FadeOut(pc_formula),
                FadeOut(pc_rect),    FadeOut(pc_cite),
                run_time=tracker.duration * 0.10,
            )
            self.play(FadeIn(conn_line1),  run_time=tracker.duration * 0.30)
            self.play(Create(conn_rect),   run_time=tracker.duration * 0.20)
            self.play(FadeIn(conn_line2),  run_time=tracker.duration * 0.20)
            self.play(
                Flash(
                    conn_line1.get_center(), color=YELLOW,
                    flash_radius=0.55, line_length=0.18, num_lines=10,
                ),
                run_time=tracker.duration * 0.15,
            )
            self.wait(tracker.duration * 0.05)

        # Beat 4 — SE(3) variant + protein application
        def make_variant_box(title_text, title_color, lines, box_color,
                             stroke_w=1.2):
            rect = RoundedRectangle(
                width=4.0, height=1.10,
                corner_radius=0.10,
                color=box_color, stroke_width=stroke_w,
                fill_color=BLACK, fill_opacity=0.08,
            )
            t0 = Text(title_text, font_size=15, color=title_color, weight=BOLD)
            t1 = Text(lines[0],   font_size=12, color=GREY_B)
            t2 = Text(lines[1],   font_size=12, color=title_color, slant=ITALIC)
            lbl = VGroup(t0, t1, t2).arrange(DOWN, aligned_edge=LEFT, buff=0.10)
            lbl.move_to(rect)
            return VGroup(rect, lbl)

        e3_box  = make_variant_box(
            "E(3)", TEAL,
            ["Rotation + Reflection + Translation",
             "→ General point clouds"],
            TEAL, stroke_w=1.2,
        )
        se3_box = make_variant_box(
            "SE(3)", GREEN,
            ["Rotation + Translation only",
             "→ Proteins, antibodies"],
            GREEN, stroke_w=1.8,
        )
        comparison = VGroup(e3_box, se3_box).arrange(DOWN, buff=0.30)
        comparison.move_to(np.array([1.80, 0.00, 0]))

        # Correct citations: Martinkus 2023 + Jin 2023
        citation_b4 = Text(
            "Protein binding · Antibody design  "
            "(Martinkus et al. 2023; Jin et al. 2023)",
            font_size=12, color=GREY_B, slant=ITALIC,
        ).to_edge(DOWN, buff=0.40)

        with self.voiceover(
            text=(
                "For proteins and antibodies, we use S-E-3 — "
                "rotations and translations only, no reflections — "
                "since molecular chirality matters. "
                "Frame averaging makes any backbone GNN or Transformer "
                "equivariant to rigid-body motions."
            )
        ) as tracker:
            self.play(
                FadeOut(pca_frame), FadeOut(conn_line1),
                FadeOut(conn_rect), FadeOut(conn_line2),
                run_time=tracker.duration * 0.12,
            )
            self.play(
                LaggedStart(FadeIn(e3_box), FadeIn(se3_box), lag_ratio=0.50),
                run_time=tracker.duration * 0.50,
            )
            self.play(FadeIn(citation_b4), run_time=tracker.duration * 0.22)
            self.wait(tracker.duration * 0.16)

        self.wait(0.5)
        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.6)
        self.wait(0.3)


In [ ]:
%manim -qh Scene26_FrameAveraging

In [ ]:
%manim -qh Scene27_PointCloudsProteins

# SLIDE 28, 29 VÀ SLIDE 30

In [ ]:
# Scene28_FALimits
class Scene28_FALimits(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title = Text(
            "Theoretical Limits of Frame Averaging",
            font_size=30, color=WHITE,
        ).to_edge(UP, buff=0.38)

        # Beat 1 -- Opening question
        subtitle = Text(
            "(Dym et al. 2024)",
            font_size=14, color=GREY_B, slant=ITALIC,
        ).next_to(title, DOWN, buff=0.12)

        question = Text(
            "Do frames solve the continuity issue?",
            font_size=24, color=WHITE,
        ).move_to(np.array([0.0, 0.80, 0]))

        with self.voiceover(
            text=(
                "Frame averaging sounds like the best of both worlds. "
                "But there are fundamental limits -- theoretical results tell us "
                "when a finite, continuous, equivariant frame simply cannot exist."
            )
        ) as tracker:
            self.play(Write(title), run_time=tracker.duration * 0.20)
            self.play(FadeIn(subtitle), run_time=tracker.duration * 0.12)
            self.play(FadeIn(question), run_time=tracker.duration * 0.40)
            self.wait(tracker.duration * 0.28)

        # Beat 2 -- Theorem 1: Sn on R^(d x n)
        thm1_header = Text("Theorem 1", font_size=18, color=YELLOW, weight=BOLD)
        thm1_body = MathTex(
            r"G = S_n \text{ on } \mathbb{R}^{d \times n},\quad n, d > 1",
            font_size=22, color=WHITE,
        )
        thm1_conclusion = MathTex(
            r"\Rightarrow \text{ only continuous frame} = S_n",
            font_size=22, color=RED,
        )
        thm1_group = VGroup(thm1_header, thm1_body, thm1_conclusion).arrange(
            DOWN, buff=0.28, aligned_edge=LEFT,
        ).move_to(np.array([0.0, 0.10, 0]))

        thm1_box = SurroundingRectangle(
            thm1_group, color=YELLOW, stroke_width=1.8,
            corner_radius=0.10, buff=0.22,
        )

        with self.voiceover(
            text=(
                "For the permutation group acting on n-by-d matrices -- "
                "with n and d both greater than one -- "
                "the only continuous equivariant frame is the full symmetric group S-n itself. "
                "There is no smaller finite frame that stays continuous."
            )
        ) as tracker:
            self.play(
                FadeOut(question),
                run_time=tracker.duration * 0.08,
            )
            self.play(
                LaggedStart(
                    Write(thm1_header),
                    Write(thm1_body),
                    Write(thm1_conclusion),
                    lag_ratio=0.35,
                ),
                run_time=tracker.duration * 0.50,
            )
            self.play(Create(thm1_box), run_time=tracker.duration * 0.20)
            self.play(
                Flash(
                    thm1_conclusion.get_center(),
                    color=RED, flash_radius=0.35,
                    line_length=0.12, num_lines=8,
                ),
                run_time=tracker.duration * 0.15,
            )
            self.wait(tracker.duration * 0.07)

        # Beat 3 -- Theorem 2: SO(2) on R^2

        thm2_header = Text("Theorem 2", font_size=18, color=ORANGE, weight=BOLD)
        thm2_body = MathTex(
            r"G = SO(2) \text{ on } \mathbb{R}^2",
            font_size=22, color=WHITE,
        )
        thm2_conclusion = MathTex(
            r"\Rightarrow \text{ continuous equivariant frame is generically not finite}",
            font_size=19, color=RED,
        )
        thm2_group = VGroup(thm2_header, thm2_body, thm2_conclusion).arrange(
            DOWN, buff=0.28, aligned_edge=LEFT,
        ).move_to(np.array([0.0, 0.10, 0]))

        thm2_box = SurroundingRectangle(
            thm2_group, color=ORANGE, stroke_width=1.8,
            corner_radius=0.10, buff=0.22,
        )

        thm2_conclusion_rect = SurroundingRectangle(
            thm2_conclusion, color=ORANGE, stroke_width=1.4,
            corner_radius=0.06, buff=0.08,
        )

        with self.voiceover(
            text=(
                "Similarly, for S-O-2 -- the rotation group on the plane -- "
                "a continuous equivariant frame is generally not finite. "
                "You would need infinitely many rotations to stay continuous."
            )
        ) as tracker:
            self.play(
                FadeOut(thm1_group), FadeOut(thm1_box),
                run_time=tracker.duration * 0.10,
            )
            self.play(
                LaggedStart(
                    Write(thm2_header),
                    Write(thm2_body),
                    Write(thm2_conclusion),
                    lag_ratio=0.35,
                ),
                run_time=tracker.duration * 0.50,
            )
            self.play(Create(thm2_box), run_time=tracker.duration * 0.18)
            self.play(
                GrowFromCenter(thm2_conclusion_rect),
                run_time=tracker.duration * 0.15,
            )
            self.wait(tracker.duration * 0.07)

        # Beat 4 -- Weighted frames as partial fix
        # LEFT column: the problem
        prob_header = Text("The problem", font_size=16, color=RED, weight=BOLD)
        prob_body = Text(
            "Finite frame ==> discontinuity risk",
            font_size=14, color=GREY_B,
        )
        prob_col = VGroup(prob_header, prob_body).arrange(
            DOWN, aligned_edge=LEFT, buff=0.18,
        ).move_to(np.array([-3.00, 0.10, 0]))

        # RIGHT column: weighted frames
        fix_header = Text("Weighted frames", font_size=16, color=GREEN, weight=BOLD)
        fix_formula = MathTex(
            r"\bar{f}(x) = \sum_{\tau \in F(x)} w(\tau,\, x)\, \phi\!\left(\tau^{-1} \cdot x\right)",
            font_size=18, color=WHITE,
        )
        fix_col = VGroup(fix_header, fix_formula).arrange(
            DOWN, buff=0.22, aligned_edge=LEFT,
        ).move_to(np.array([1.80, 0.10, 0]))

        fix_rect = SurroundingRectangle(
            fix_formula, color=GREEN, stroke_width=1.4,
            corner_radius=0.08, buff=0.10,
        )

        # Dividing line
        div_line = DashedLine(
            start=np.array([-0.60, 1.20, 0]),
            end=np.array([-0.60, -1.10, 0]),
            color=GREY_B, stroke_width=1.0, dash_length=0.12,
        )

        citation_b4 = Text(
            "(Dym et al. 2024)",
            font_size=12, color=GREY_B, slant=ITALIC,
        ).to_corner(DR, buff=0.30)

        with self.voiceover(
            text=(
                "The good news: weighted frames can help. "
                "Instead of equal weights, each frame element carries a learned weight w of tau and x. "
                "Smaller frames become possible -- with partial recovery of continuity."
            )
        ) as tracker:
            self.play(
                FadeOut(thm2_group), FadeOut(thm2_box), FadeOut(thm2_conclusion_rect),
                run_time=tracker.duration * 0.10,
            )
            self.play(
                FadeIn(div_line),
                FadeIn(prob_col),
                run_time=tracker.duration * 0.25,
            )
            self.play(
                FadeIn(fix_header),
                Write(fix_formula),
                run_time=tracker.duration * 0.40,
            )
            self.play(
                Create(fix_rect),
                FadeIn(citation_b4),
                run_time=tracker.duration * 0.18,
            )
            self.wait(tracker.duration * 0.07)

        self.wait(0.5)
        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.6)
        self.wait(0.3)


# Scene29_Symmetrization
class Scene29_Symmetrization(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title = Text(
            "Symmetrization",
            font_size=34, color=WHITE,
        ).to_edge(UP, buff=0.38)

        # Beat 1 -- Title + spectrum reveal
        # (~22 words, ~7 sec)

        spectrum_line = Line(
            start=np.array([-3.60, 0.60, 0]),
            end=np.array([3.60, 0.60, 0]),
            color=GREY_B, stroke_width=1.4,
        )

        pos_left  = np.array([-3.60, 0.60, 0])
        pos_mid   = np.array([0.00,  0.60, 0])
        pos_right = np.array([3.60,  0.60, 0])

        dot_left  = Dot(pos_left,  radius=0.08, color=RED)
        dot_mid   = Dot(pos_mid,   radius=0.11, color=YELLOW)
        dot_right = Dot(pos_right, radius=0.08, color=TEAL)

        lbl_left  = Text("Canonization",    font_size=14, color=RED).next_to(dot_left,  UP, buff=0.14)
        lbl_mid   = Text("Frame Averaging", font_size=14, color=YELLOW).next_to(dot_mid,   UP, buff=0.14)
        lbl_right = Text("Group Averaging", font_size=14, color=TEAL).next_to(dot_right, UP, buff=0.14)

        spectrum_group = VGroup(
            spectrum_line,
            dot_left, lbl_left,
            dot_mid,  lbl_mid,
            dot_right, lbl_right,
        )

        with self.voiceover(
            text=(
                "Now let's put all three approaches on a single spectrum "
                "and compare them on cost and continuity."
            )
        ) as tracker:
            self.play(Write(title), run_time=tracker.duration * 0.20)
            self.play(
                Create(spectrum_line),
                run_time=tracker.duration * 0.20,
            )
            self.play(
                LaggedStart(
                    FadeIn(dot_left),  FadeIn(lbl_left),
                    FadeIn(dot_mid),   FadeIn(lbl_mid),
                    FadeIn(dot_right), FadeIn(lbl_right),
                    lag_ratio=0.20,
                ),
                run_time=tracker.duration * 0.45,
            )
            self.play(
                Flash(dot_mid.get_center(), color=YELLOW,
                      flash_radius=0.35, line_length=0.12, num_lines=8),
                run_time=tracker.duration * 0.10,
            )
            self.wait(tracker.duration * 0.05)
        # Beat 2a -- Cost row
        # (~30 words, ~10 sec)
        cost_left = VGroup(
            Text("+ one transformation:", font_size=13, color=GREEN),
            Text("  cheap",               font_size=13, color=GREEN),
        ).arrange(DOWN, aligned_edge=LEFT, buff=0.05).next_to(dot_left, DOWN, buff=0.50)

        cost_mid = Text(
            "+ can be cheaper",
            font_size=13, color=GREEN,
        ).next_to(dot_mid, DOWN, buff=0.50)

        cost_right = VGroup(
            Text("- full group: can be expensive,", font_size=12, color=RED),
            Text("  approximations needed",          font_size=12, color=RED),
        ).arrange(DOWN, aligned_edge=LEFT, buff=0.05).next_to(dot_right, DOWN, buff=0.50)

        cost_row = VGroup(cost_left, cost_mid, cost_right)

        # Nhan Cost: ben trai cost_left (xuat hien cung beat 2a)
        label_cost = Text("Cost:", font_size=13, color=WHITE).next_to(cost_left, LEFT, buff=0.5)

        with self.voiceover(
            text=(
                "Starting with cost: canonization is cheap -- it picks a single transformation. "
                "Frame averaging can be cheaper than full group averaging when the frame is small. "
                "Group averaging over the full group can be expensive or require approximations."
            )
        ) as tracker:
            self.play(
                ShowPassingFlash(
                    spectrum_line.copy().set_color(YELLOW),
                    time_width=0.60,
                ),
                run_time=tracker.duration * 0.18,
            )
            self.play(
                LaggedStart(
                    FadeIn(label_cost),
                    FadeIn(cost_left),
                    FadeIn(cost_mid),
                    FadeIn(cost_right),
                    lag_ratio=0.35,
                ),
                run_time=tracker.duration * 0.57,
            )
            self.wait(tracker.duration * 0.25)
        # Beat 2b -- Continuity row
        cont_left = Text(
            "- not continuous",
            font_size=13, color=RED,
        ).next_to(cost_left, DOWN, buff=0.22)

        # Thêm nhãn Continuity: bên trái cont_left và căn lề trái với nhãn Cost:
        label_continuity = Text("Continuity:", font_size=13, color=WHITE).next_to(cont_left, LEFT, buff=0.5)
        label_continuity.align_to(label_cost, LEFT)

        cont_mid = VGroup(
            Text("- continuity", font_size=13, color=RED),
            Text("  still an issue", font_size=13, color=RED),
        ).arrange(DOWN, aligned_edge=LEFT, buff=0.05).next_to(cost_mid, DOWN, buff=0.22)

        cont_right = Text(
            "+ continuous",
            font_size=13, color=GREEN,
        ).next_to(cost_right, DOWN, buff=0.22)

        cont_right_rect = SurroundingRectangle(
            cont_right, color=GREEN, stroke_width=1.4,
            corner_radius=0.06, buff=0.08,
        )

        with self.voiceover(
            text=(
                "On continuity: canonization is not continuous -- a small change in input "
                "can jump to a different canonical form. "
                "Frame averaging still has continuity issues. "
                "Only group averaging is continuous by construction."
            )
        ) as tracker:
            self.play(
                LaggedStart(
                    FadeIn(label_continuity), # Hiển thị chữ Continuity:
                    FadeIn(cont_left),
                    FadeIn(cont_mid),
                    FadeIn(cont_right),
                    lag_ratio=0.35,
                ),
                run_time=tracker.duration * 0.80,
            )
            self.wait(tracker.duration * 0.20)
        # Beat 3 -- General advantages
        adv_header = Text(
            "General advantages:",
            font_size=20, color=WHITE,
        ).move_to(np.array([0.0, 1.80, 0]))

        bullet1 = VGroup(
            Text("Preserves expressiveness of the base model:", font_size=15, color=GREEN),
            Text("  reuse successful existing models & training methods", font_size=14, color=GREY_B),
        ).arrange(DOWN, aligned_edge=LEFT, buff=0.06)

        bullet2 = Text(
            "Can easily combine symmetries",
            font_size=15, color=GREEN,
        )

        bullet3 = Text(
            "Easy to apply",
            font_size=15, color=GREEN,
        )

        bullets = VGroup(bullet1, bullet2, bullet3).arrange(
            DOWN, aligned_edge=LEFT, buff=0.28,
        ).move_to(np.array([0.20, 0.10, 0]))

        bullets_rect = SurroundingRectangle(
            bullets, color=GREEN, stroke_width=1.2,
            corner_radius=0.10, buff=0.22,
        )

        with self.voiceover(
            text=(
                "All three approaches share important general advantages. "
                "First: they preserve the expressiveness of the base model, "
                "letting you reuse successful existing models and training methods. "
                "Second: they can easily combine symmetries. "
                "Third: they are easy to apply."
            )
        ) as tracker:
            self.play(
                FadeOut(spectrum_group),
                FadeOut(label_cost), FadeOut(label_continuity),
                FadeOut(cost_row),
                FadeOut(cont_left), FadeOut(cont_mid),
                FadeOut(cont_right),
                run_time=tracker.duration * 0.12,
            )
            self.play(FadeIn(adv_header), run_time=tracker.duration * 0.12)
            self.play(
                LaggedStart(
                    FadeIn(bullet1, shift=RIGHT * 0.20),
                    FadeIn(bullet2, shift=RIGHT * 0.20),
                    FadeIn(bullet3, shift=RIGHT * 0.20),
                    lag_ratio=0.35,
                ),
                run_time=tracker.duration * 0.56,
            )
            self.play(Create(bullets_rect), run_time=tracker.duration * 0.14)
            self.wait(tracker.duration * 0.06)
        # Beat 4 -- Takeaway box
        tkw_title = Text("Key Takeaway", font_size=18, color=YELLOW, weight=BOLD)
        tkw_body  = Text(
            "Symmetrize any existing model -- off the shelf,\nno architectural changes needed.",
            font_size=15, color=WHITE,
        )
        tkw_group = VGroup(tkw_title, tkw_body).arrange(DOWN, buff=0.22)
        tkw_group.move_to(ORIGIN)

        tkw_box = SurroundingRectangle(
            tkw_group, color=YELLOW, stroke_width=1.8,
            corner_radius=0.12, buff=0.30,
        )

        with self.voiceover(
            text=(
                "The bottom line: symmetrization lets you take any existing non-equivariant model "
                "and make it equivariant, off the shelf. "
                "The right choice depends on your group, your budget, and your continuity needs."
            )
        ) as tracker:
            self.play(
                FadeOut(adv_header), FadeOut(bullets), FadeOut(bullets_rect),
                run_time=tracker.duration * 0.10,
            )
            self.play(
                FadeIn(tkw_group),
                Create(tkw_box),
                run_time=tracker.duration * 0.45,
            )
            self.play(
                Flash(
                    tkw_box.get_center(),
                    color=YELLOW, flash_radius=0.50,
                    line_length=0.18, num_lines=10,
                ),
                run_time=tracker.duration * 0.30,
            )
            self.wait(tracker.duration * 0.15)

        self.wait(0.5)
        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.6)
        self.wait(0.3)


# Scene30_Bridge
class Scene30_Bridge(VoiceoverScene):
    def construct(self):
        self.set_speech_service(EdgeTTSService(voice=VOICE))

        title = Text(
            "How Can We Get Equivariant Models?",
            font_size=30, color=WHITE,
        ).to_edge(UP, buff=0.38)

        # Beat 1 -- Off-the-shelf recap (LEFT column)
        left_header = Text(
            "Using an off-the-shelf model:",
            font_size=16, color=GREY_B, slant=ITALIC,
        )
        left_b1 = Text("Data augmentation",      font_size=15, color=WHITE)
        left_b2 = Text("Canonization",            font_size=15, color=WHITE)
        left_b3 = Text("Group / Frame averaging", font_size=15, color=WHITE)

        left_col = VGroup(left_header, left_b1, left_b2, left_b3).arrange(
            DOWN, aligned_edge=LEFT, buff=0.22,
        ).move_to(np.array([-3.00, 0.00, 0]))

        with self.voiceover(
            text=(
                "We've now covered three ways to symmetrize any existing model: "
                "data augmentation, canonization, and group or frame averaging. "
                "All of these work off the shelf, with no changes to the underlying architecture."
            )
        ) as tracker:
            self.play(Write(title), run_time=tracker.duration * 0.18)
            self.play(FadeIn(left_header), run_time=tracker.duration * 0.12)
            self.play(
                LaggedStart(
                    FadeIn(left_b1, shift=RIGHT * 0.18),
                    FadeIn(left_b2, shift=RIGHT * 0.18),
                    FadeIn(left_b3, shift=RIGHT * 0.18),
                    lag_ratio=0.35,
                ),
                run_time=tracker.duration * 0.52,
            )
            self.wait(tracker.duration * 0.18)

        # Beat 2 -- Constructing equivariant models (RIGHT column)
        bridge_arrow = Arrow(
            start=np.array([-1.20, 0.00, 0]),
            end=np.array([0.70, 0.00, 0]),
            color=YELLOW, stroke_width=5,
            max_tip_length_to_length_ratio=0.22,
        )


        right_header = Text(
            "Constructing equivariant models:",
            font_size=16, color=YELLOW, weight=BOLD,
        )
        right_b1 = Text(
            "Parameter sharing, group convolution",
            font_size=14, color=YELLOW,
        )
        right_b2 = Text("Invariant theory",     font_size=14, color=YELLOW)
        right_b3 = Text("Representation theory", font_size=14, color=YELLOW)

        right_col = VGroup(right_header, right_b1, right_b2, right_b3).arrange(
            DOWN, aligned_edge=LEFT, buff=0.22,
        ).move_to(np.array([3.00, 0.00, 0]))

        with self.voiceover(
            text=(
                "But there's a second family. "
                "Rather than symmetrizing an existing model, "
                "we can construct equivariant layers from scratch -- "
                "using parameter sharing, group convolution, invariant theory, "
                "and representation theory. "
                "This is what the next part of this tutorial explores."
            )
        ) as tracker:
            self.play(
                GrowArrow(bridge_arrow),
                run_time=tracker.duration * 0.20,
            )
            self.play(FadeIn(right_header), run_time=tracker.duration * 0.12)
            self.play(
                LaggedStart(
                    FadeIn(right_b1, shift=LEFT * 0.18),
                    FadeIn(right_b2, shift=LEFT * 0.18),
                    FadeIn(right_b3, shift=LEFT * 0.18),
                    lag_ratio=0.35,
                ),
                run_time=tracker.duration * 0.50,
            )
            self.wait(tracker.duration * 0.18)

        # Beat 3 -- Hand-off sentence
        handoff_text = Text(
            "Next: Constructing Equivariant Models",
            font_size=22, color=WHITE, slant=ITALIC,
        ).move_to(np.array([0.0, 0.30, 0]))

        handoff_sub = Text(
            "-- parameter sharing, group convolution, invariant & representation theory",
            font_size=14, color=GREY_B,
        ).next_to(handoff_text, DOWN, buff=0.22)

        with self.voiceover(
            text=(
                "We've covered how to symmetrize existing models -- "
                "next, we'll see how to build equivariant layers from scratch."
            )
        ) as tracker:
            self.play(
                FadeOut(left_col), FadeOut(bridge_arrow), FadeOut(right_col),
                run_time=tracker.duration * 0.15,
            )
            self.play(FadeIn(handoff_text), run_time=tracker.duration * 0.35)
            self.play(FadeIn(handoff_sub),  run_time=tracker.duration * 0.25)
            self.play(
                Flash(
                    handoff_text.get_center(),
                    color=YELLOW, flash_radius=0.60,
                    line_length=0.20, num_lines=10,
                ),
                run_time=tracker.duration * 0.18,
            )
            self.wait(tracker.duration * 0.07)

        self.wait(0.8)
        self.play(*[FadeOut(m) for m in self.mobjects], run_time=0.6)
        self.wait(0.3)


In [ ]:
%manim -qh Scene28_FALimits

In [ ]:
%manim -qh Scene29_Symmetrization

In [ ]:
%manim -qh Scene30_Bridge